# Pix2Struct TextCaps-base — DIMER E2E text-aware captioning fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/pix2struct-textcaps-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/pix2struct-textcaps-pipeline/blob/main/tutorials/pix2struct_textcaps_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-google%2Fpix2struct--textcaps--base-ffcc4d?style=flat)](https://huggingface.co/google/pix2struct-textcaps-base) [![Upstream](https://img.shields.io/badge/Upstream-google--research%2Fpix2struct-181717?style=flat&logo=github&logoColor=white)](https://github.com/google-research/pix2struct) [![arXiv](https://img.shields.io/badge/arXiv-2210.03347-b31b1b.svg)](https://arxiv.org/abs/2210.03347)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.5 — **standalone** (§4)  
**Capability:** text-aware image captioning and bounded supervised fine-tuning of the caption decoder's last blocks on a photograph/reference-captions dataset, using the pinned `google/pix2struct-textcaps-base` weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/pix2struct_textcaps_pipeline/`, at revision `728a24c4227c`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `61bee0d7e2378e601b68f853ceee4f7cf99f1b88` (~1133 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2.5); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime builds an isolated hash-locked environment from the pinned dependencies (nothing is installed into the notebook kernel, so no restart is needed), stages and digest-verifies the pinned `google/pix2struct-textcaps-base` snapshot (a 1.13 GB `model.safetensors`), reads the four text columns of one digest-pinned VizWiz-Captions shard from the Hugging Face Hub (about 0.5 MB over HTTP range requests, no credential) and the image column of its first row group (336 photographs, about 84 MB, each refused on any size or SHA-256 mismatch), cuts the 318 captioned ones by image into 208 / 40 / 70 training, validation and exposed-development-test photographs, reads the image column of row group 2 the same way (336 more photographs, about 84 MB) as the **fresh test** — its 324 captioned photographs, minus any near-copy of a training photograph confirmed by a pre-registered, recorded review (none was confirmed), captions three drawn scenes with known text through the inference contract with an input manifest and a rejection probe, scores the frozen model on the test photographs with BLEU-4, ROUGE-L, CIDEr-D and unigram F1 beside the constant-caption and colour-nearest-neighbour baselines, runs a bounded fine-tuning of the caption decoder's last blocks on the training photographs at three learning rates and keeps one by pre-registered validation-CIDEr-D rules, scores the fresh test only under the pre-registered settings, with a paired bootstrap interval (and the exposed development test for continuity), re-captions the drawn scenes with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify caption parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.5 §5). A CUDA runtime is used automatically when present; the CPU path works but is slow (Pix2Struct encodes every image at up to 2,048 patches). In the pre-registered evaluation of record (REL13: the Colab CLI T4 run of 2026-10-17 of blob `07064ffc`, commit `e86f270`), the default path took 2,036.4 s wall on a T4, 348.8 / 349.1 / 349.8 s of it the three grid fine-tunings; the CPU path is estimated at several hours (an estimate, not a measurement), so choose a GPU runtime.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` and `BYOD_PATH` (a zip already in the runtime; on Colab an empty path opens an upload dialog) in Section 4, then re-run Section 4 and every code cell of Sections 5–9 in order (Sections 6 and 7 reload the frozen pipeline from the verified snapshot when `pipe` was adapted by the VizWiz pass, so the frozen numbers are the base model's on your photographs) to supply one zip holding a `records.jsonl` (or `records.json`) of `{id, image, captions}` objects — `image` a file name inside the zip, `captions` one or more reference captions, optional `category` — beside the image files (folders inside the zip are kept, so `image` may be `images/x.jpg`; the records file may sit at the root or inside one folder, and image paths are relative to it). They pass through the same validation, seeded image-disjoint split, baselines, fine-tuning, held-out evaluation, artifact export and reload-parity cells as the VizWiz sample. The expected schema and the ceilings (at least 50 records at one record per image so every split keeps 8, at most 5,000 records, 5,002 zip members and 2 GiB extracted) are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path; a BYOD run is exploratory, so it has no fresh test and its reports go to `*_exploratory_<n>.json`. A refused zip prints one message naming the zip and the rule, keeps earlier output files but marks them as not results for your data (`outputs/pix2struct_textcaps_run_invalidated.json`), and stops the later sections until Section 4 succeeds.

`google/pix2struct-textcaps-base` is the Pix2Struct model [1] — a ViT-style image encoder over variable-resolution 16×16 patches (up to 2,048 per image) and a 12-layer text decoder that cross-attends to them; 282,285,696 parameters, pretrained by parsing masked web screenshots into simplified HTML and fine-tuned on TextCaps [2], whose captions must mention the text visible in the image — published under the **Apache-2.0** licence. At inference it reads any text in the image from pixels and generates a sentence token by token with greedy decoding under a caller-owned `max_new_tokens` budget. **No score exists**: the caption is generated text with no probability and no correctness signal, and a fluent caption is **not evidence that it describes the image** or that the text it quotes is really there.

What this notebook adds to inference is **adaptation with reference captions**. The dataset is real and out of the model's distribution: VizWiz-Captions [3] (**CC BY 4.0**) — photographs taken by blind people, each with up to five crowd-written captions in a style TextCaps does not have (what is being held, what a label says, that the picture is blurry or badly framed). About half the photographs contain text, which is exactly what this checkpoint was trained to quote, so the per-category breakdown (`text` / `no-text`) is part of the reading. The notebook reads only the four text columns of one pinned Hub shard (about 0.5 MB over HTTPS range requests) and the image column of its **first row group** (336 photographs in one 84 MB range read, each pinned by size and SHA-256 in the carried module) — the same pinned sample the BLIP captioning pipeline uses, so the two captioners can be read side by side. **Data overlap (DAT9):** these are photographs from VizWiz, which is not in the checkpoint's TextCaps fine-tuning data (TextCaps images are Open Images photographs); the web-screenshot pretraining corpus is not public, so overlap with it cannot be fully ruled out, though rendered web pages make it unlikely. **What touched the test:** the default recipe was carried over from the BLIP captioning row, which chose it by comparing configurations on these same 70 test photographs, and this row's own pre-flight (blob `2b72c3f7`, 2026-09-24) also scored them at three learning rates. They are therefore reported as the **exposed development test**. The learning rate is re-selected here on validation only, by rules pre-registered in `docs/learning-rate-grid-preregistration.md` before any run, and the headline comparison is made on a **fresh test** — row group 2 of the same pinned shard, never fetched or scored by any earlier run — scored only under those settings. The honest question is a narrow one: does a bounded adaptation of the caption decoder's last blocks on 208 photographs move the consensus metric on a fresh, image-disjoint test, on which category, and which of the four metrics move with it? Three captioning metrics are implemented in pure Python in the carried modules (**BLEU-4** [4], **ROUGE-L** [5] and **CIDEr-D** [6]: BLEU-4 and CIDEr-D follow the `coco-caption` formulas and ROUGE-L uses its aggregation — the best precision and the best recall over the references taken separately — with CIDEr-D's document frequencies taken from the evaluated set; the tokenisation is simplified (lower-case, non-word characters replaced by spaces), **not the PTB tokenizer** `coco-caption` runs, so absolute values differ from `pycocoevalcap` on the same captions) beside the plumbing check `unigram_f1`, and two **non-neural baselines** — the corpus-medoid constant caption and a colour nearest neighbour — show where a system with no model sits. Nothing here is a quality claim about your photographs: it is one seeded split of one corpus.

**Weight-format note:** the pinned revision ships the model as SafeTensors (`model.safetensors`, digest-pinned in the manifest); the processor is the non-VQA variant, so no header text is rendered into the image and no font is downloaded. Section 3 stages and digest-verifies the snapshot before the processor or the model is constructed.

**Learning objectives:** install the pinned runtime; read what the carried pipeline, metrics and dataset modules guarantee; stage and digest-verify the immutable upstream snapshot; read the captions of a digest-pinned corpus without downloading its shard and its photographs from one pinned row group with per-file digests, validate them and split by image without leakage; caption through the public API over drawn scenes with known text and read `caption`, `new_tokens`, `truncated` and `text_recall` correctly (generated text, no score); score the frozen model against several reference captions per photograph with BLEU-4, ROUGE-L and CIDEr-D beside two non-neural baselines and read the `text` / `no-text` breakdown; run a bounded fine-tuning with explicit hyperparameters, choose its learning rate and epoch on validation under pre-registered rules, and say what a test split that was used for selection can and cannot show; score a fresh test only under the pre-registered settings and read a paired bootstrap interval rather than a point difference; re-caption drawings from a different image family with the adapted model; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** OCR as a product (the model quotes text inside a caption; it returns no transcription, boxes or confidences), visual question answering (a separate checkpoint), dense or region captioning (one sentence per image, no localisation), captions in languages other than English, batch throughput, sampling, beam search or repetition penalties (the notebook decodes greedily for reproducibility), SPICE (needs a scene-graph parser), evaluation on the TextCaps or VizWiz benchmarks proper (one seeded sample from row group 0 and a 324-photograph fresh test from row group 2 are scored here), fine-tuning of the image encoder, the embeddings or the output projection, training on images that are not the pinned sample or your own uploads, and any claim that a VizWiz split stands in for your photographs. The repository exposes none of these.

> **⚠️ Notice.** For research, training and evaluation only. Not validated for production,
> clinical, diagnostic or safety-critical use. Running this notebook downloads third-party
> model weights and datasets under their own licenses (see the model card) and uses compute
> on your own Colab or Kaggle account. Data you bring stays in your runtime, and you are
> responsible for having the right to use it. Results here are tutorial evidence, not
> benchmarks. Everything is provided "as is", without warranty of any kind. To the maximum
> extent permitted by law, the maintainers and the DIMER platform accept no liability for
> its use.

**Who this notebook is for.** A learner who knows basic Python, has used Colab or Jupyter and has met image captioning, and wants to see how an OCR-free captioner reads the text in a photograph, how captions are scored honestly (BLEU-4, ROUGE-L, CIDEr-D) against baselines that barely look at the image, and how a bounded fine-tuning is measured and exported. The audience is students and practitioners preparing their own captioning data; no prior experience with Pix2Struct or fine-tuning is assumed — each term is explained where it first matters and again in the **Glossary**. A T4 GPU runtime is recommended (CPU works but the fine-tuning is slow).

**Input → Model → Output.**

| | |
|---|---|
| Input | photographs with human reference captions: a digest-pinned VizWiz-Captions sample split by photograph (208 training, 40 validation, 70 exposed development-test photographs) plus a fresh test of 324 captioned photographs from another row group, or your own zip |
| Model | Pix2Struct TextCaps-base: variable-resolution patches into an image encoder and a 12-layer decoder that generates the caption with greedy decoding; only the last two decoder blocks are trained |
| Output | a caption per photograph, fresh-test BLEU-4 / ROUGE-L / CIDEr-D overall and per `text` / `no-text` category beside a constant-caption and a colour-neighbour baseline (scored only under the pre-registered settings), the same on the exposed development test, a validation-selected learning rate, and a safetensors adapter that reloads with identical captions |

**How to use this notebook.** Choose **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Run all completes in one pass: Section 1 installs nothing into the notebook's own Python, so no restart is needed (the pre-registered evaluation of record (REL13: the Colab CLI T4 run of 2026-10-17 of blob `07064ffc`, commit `e86f270`) completed in one pass). Sections 1–3 are **infrastructure** — the isolated environment, the carried package and the verified snapshot — and their cells are collapsed; you may run them without studying them. The learning path starts in Section 4. Form fields (`# @param`) are the only values meant to be edited, and the defaults are the **pre-registered settings**: changing any of them makes the run exploratory, and an exploratory run never scores the fresh test (it says so when it happens). Before each principal result the notebook asks you to **Predict**; after it comes a collapsible **Check your reasoning** whose worked answer quotes the pre-registered evaluation of record (REL13: the Colab CLI T4 run of 2026-10-17 of blob `07064ffc`, commit `e86f270`) — the validation grid, the 70 exposed development-test photographs and the fresh test; `docs/release-verification.md` records it and any reproductions. **Troubleshooting**, a **Glossary** and a **Conclusion** template are at the end. Writing your predictions down is optional.

**Roadmap:** 1–3 infrastructure → 4 the VizWiz sample and a photograph-level split *(evaluation practice: leakage)* → 5 the inference contract on a drawn scene with text → 6 two baselines and the frozen model *(core concept: CIDEr-D and the text / no-text split)* → 7 a validation-only learning-rate grid and bounded fine-tuning of two decoder blocks *(core concepts: what is trained; selecting without the test)* → 8 the fresh test, scored only under the pre-registered settings, and the exposed development test → 9 captions again, export and reload *(engineering)* → conclude.

## Prerequisites

- **Learner:** basic Python and Colab or Jupyter familiarity; no prior experience with Pix2Struct or fine-tuning. The metrics, baselines, image-level splits, validation selection and adapters are explained where they are first used and again in the Glossary.
- **Runtime:** a fresh supported **Linux x86_64** runtime (Google Colab, Kaggle or Linux Jupyter). Section 1 builds its own Python 3.12.12 environment from a hash-locked list of manylinux wheels, so the Python version of the kernel itself does not matter and nothing is installed into it; a Windows or macOS kernel is not supported. The default path runs on CPU (float32) and uses CUDA automatically when available; a GPU runtime is recommended for the fine-tuning section. Every image is encoded at up to 2,048 patches, so captioning and validation scoring cost seconds per photograph on CPU. The pinned `torch==2.14.0` install and the 1.13 GB checkpoint are the large downloads of the run; the row group of photographs adds about 84 MB.
- **Duration:** in the pre-registered evaluation of record (REL13: the Colab CLI T4 run of 2026-10-17 of blob `07064ffc`, commit `e86f270`), the default path took **2,036.4 s** wall on a T4 (Colab CLI wall, including the 67 s isolated-environment build), 348.8 / 349.1 / 349.8 s of it the three grid fine-tunings at 1e-5 / 2e-5 / 5e-5. **CPU has not been timed** for the whole path: the recorded CPU inference runs took 2.8–3.4 s per image for the 2,048-patch encoder pass, and the default path now captions about 1,300 images and trains three times four epochs of about 900 decoder steps, so expect **several hours** — an estimate, not a measurement. Choose **Runtime → Change runtime type → T4 GPU** before Section 1.
- **Knowledge:** basic Python and PIL; what an encoder–decoder model's generated tokens are; what BLEU-4, ROUGE-L and CIDEr-D measure (n-gram precision with a brevity penalty, longest-common-subsequence F-measure, TF-IDF-weighted n-gram consensus) and why none is a human judgement; why a confident caption is not a correct one, and why a quoted string is not proof the text is in the image.
- **Data contract:** records are `{id, image, captions}` — an image file (a path relative to the records file; folders inside a BYOD zip are kept) decodable by Pillow with sides between `MIN_IMAGE_SIDE` (16) and `MAX_IMAGE_SIDE` (4096) px and one or more non-empty reference captions of at most `MAX_CAPTION_CHARS` (500) characters (`MIN_CAPTIONS` = 1; VizWiz supplies up to five); optional `image_id` (defaults to the id) groups records on the same image and optional `category` labels the breakdown (`text` / `no-text` in the sample, from the corpus's text-detected flag). Ids match `[A-Za-z0-9_.:-]{1,64}` and are unique; a dataset needs 8..5,000 records; every record on the same image lands in the same split so a test image is never trained on; every (image, reference caption) pair is one training sample. BYOD accepts one zip of images plus a `records.jsonl` / `records.json` in that shape. Because each split is validated with the same 8-record floor, the effective minimum is **50 records at one record per image** (the 0.15 / 0.20 validation / test fractions leave 8 and 10); at most 5,002 zip members and 2 GiB extracted; a zip is refused, with the file and the rule named, when it has no records file, more than one, a member path outside the zip's folder, a split under 8 records, or exceeds those ceilings.
- **Validation is structural, not semantic:** every image is opened and decoded and every caption checked, but nothing checks that a reference caption is right — a mislabelled corpus is fine-tuned on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — photographs of people, documents or homes are exactly that. The default path uploads nothing.
- **External access (data):** besides the model snapshot, the default path reads two parts of one object in the Hub dataset repository `mm-eval/VizWiz-Captions` at the immutable revision `c4a6d897…` (`data/val-00004-of-00005.parquet`, 392,245,504 bytes, SHA-256 `4492465a…`): the declared size and SHA-256 are checked against the pins before any byte is read; the four text columns of all 1,550 rows are fetched over HTTPS range requests through `pyarrow` (only the parquet footer and those column chunks) and refused unless their decoded SHA-256 matches; then the image columns of row groups 0 and 2 only (336 JPEG files each, about 84 MB each) are read the same way, each photograph pinned by size and SHA-256 in the carried module and refused on any mismatch. The corpus is CC BY 4.0 [3].
- **External access:** the Hugging Face Hub only, to fetch the pinned `google/pix2struct-textcaps-base` snapshot (~1133 MB in total) at revision `61bee0d7e237…`. No GitHub access and no credentials are required; nothing is installed from this repository. The isolated environment also needs PyPI (`pypi.org`, `files.pythonhosted.org`) for the pinned `uv` wheel and the 48 hash-locked packages, and the managed CPython 3.12.12 build (python-build-standalone) that `uv` downloads.

## 1. Install the pinned runtime (isolated environment)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

Hosted runtimes such as Colab and Kaggle import some packages, NumPy and often PyTorch among them, before the first cell runs, and Python cannot swap a module that is already loaded. Installing the pins into the notebook's own Python would therefore leave mixed versions or require a manual restart. Instead, the next cell builds a separate environment (`dimer_isolated_env_<lock digest>/`) and leaves the kernel's packages untouched: it downloads the pinned `uv` 0.12.15 wheel and refuses it unless its size and SHA-256 match, has `uv` install the managed CPython **3.12.12** (whatever Python the kernel itself runs), and installs the 48 packages of the carried hash-locked requirements (`tutorials/requirements-colab.lock.txt`, compiled from the pins in the cell) with `--require-hashes --only-binary :all:`, so every package, direct or transitive, is the exact file that was locked. No repository code is installed. The lock holds manylinux x86_64 wheels, so the notebook supports **Linux x86_64 runtimes only** (Google Colab, Kaggle or Linux Jupyter); on any other platform the cell stops with that message.

The same cell then starts one Python process in that environment and routes **every later code cell** to it. Printed output comes back to the notebook as usual, variables persist from cell to cell, and an error stops **Run all** as it would in the kernel. It is the only cell that runs in the kernel (it is marked `# dimer: kernel cell`), and it is safe to re-run: a complete environment built from this exact lock is reused rather than rebuilt, and a live worker is kept with every variable created so far, so the cells after it keep working. To start over, restart the session and choose **Run all**. `DIMER_NOTEBOOK_CI_PREINSTALLED=1` lets an executor that has already installed exactly these pins run every cell in its own kernel instead. `DIMER_ISOLATED_ENV` overrides the folder the environment is built in (default `dimer_isolated_env_<first 12 hex of the lock digest>` in the working directory); it changes only where the environment lives, not what is installed.

In [ ]:
# @title Infrastructure: build (or reuse) the isolated environment and route later cells to it
# dimer: kernel cell (runs in the notebook kernel, not in the isolated environment)
import hashlib
import io
import os
import platform
import signal
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile
from multiprocessing.connection import Connection
from pathlib import Path

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
    'pyarrow==25.0.1',
]
MANAGED_PYTHON = '3.12.12'
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_BYTES = 20081404
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
LOCK_NAME = 'requirements-colab.lock.txt'
LOCK_SHA256 = '9c9d7dd59af7dad632ec635ca22ba30097a3ceb07a5806ce8316fb77319d0258'
LOCKED_PACKAGES = 48
# The hash-locked requirements, compiled from the PINS above with `uv pip compile --generate-hashes` for manylinux x86_64.
LOCK_TEXT = r'''# This file was autogenerated by uv via the following command:
#    uv pip compile pyproject.toml --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --only-binary :all: -o tutorials/requirements-colab.lock.txt
certifi==2026.7.22 \
    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \
    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55
    # via requests
charset-normalizer==3.5.2 \
    --hash=sha256:01077390b03f7988f11d700a2194e69b119741a86b1a638b1db88891e3eced8e \
    --hash=sha256:01b0c0d2262a9e28e8484a278c7e1b5d650e3ac8cf2683d2967e25899f208bdf \
    --hash=sha256:04851f73ae72b8413dddadb16a49dfee95263553741fd42d546f7d66907e6be5 \
    --hash=sha256:0521c5665880b33d603717defa76c094048900010897909952397feb3039da56 \
    --hash=sha256:0774bf9bf620249fee3e0b8b9fd3065de213be30f3aa94ce2494b3b638949e26 \
    --hash=sha256:0891b9d3903c5571c03771ca669a4b0ec5618ca722a5c957d3d29cd4e5062848 \
    --hash=sha256:0c951d5e6dd9c2ff60609476752bee49da4206adde960ebc247766937f72e718 \
    --hash=sha256:0fed1d06615f022ee3b13caf5e8b180cfea32bb2c5aded8a9d44277afc040f93 \
    --hash=sha256:114e4d0c92d618409ed82a99e22b5c5e768fe995f2973f78265f4524f49d4640 \
    --hash=sha256:11912e4bb14baae7c5d8791aa55ba0a3a03ec6729073307b0f57270abaa713d3 \
    --hash=sha256:11a4d68a6ecda3292cb1e50239e111543ba5d709bb62a6b4ea1afcfa729d8875 \
    --hash=sha256:124fbf1a8ff966d87ae05bb8bd45a71f966055ed8bba320d0c7cf450bc5f4d0e \
    --hash=sha256:1461ac396c4fdb983a675f20aa555624f0ee18ac83d832b9244ffff3d8055275 \
    --hash=sha256:1503bccbeb36d5527790c3930327704c39af22de3112f1b1666a9f3ce15ee204 \
    --hash=sha256:15bb4005af6320d259dc7593ca84a38d7fe06a421dbcf7b910ae23979101e787 \
    --hash=sha256:15c44f7edfd477b06f517a5cc317fc1707edb9de2c865f43d4b6513907473234 \
    --hash=sha256:16fa0eccf81304b79c5cd87f9271c3b85dd9dd99245e4422ae9c0dd45e0f99d3 \
    --hash=sha256:183b88127acdb4fabe59d951ab424faf1af7b63cdbb5f776186c1ea2ffcaed98 \
    --hash=sha256:195c26fb65950f8fce54e26349852b7bdd7c5f120aeefbcc440b8a20faaed4a3 \
    --hash=sha256:1afb975bd5d68d5ce9f6b6d44fdf2f7e34b895a35e95708a7a91b20a3b51d187 \
    --hash=sha256:1b4cbc7c3491ccb4aa17fcd8165649d01cf39f76de1696da8631b5f71b85401d \
    --hash=sha256:1bc0baf5ef96b6ede57d47f4b8fe4d9d84019c3bfcbeb20a41edc6a6ee341f1f \
    --hash=sha256:1c50fe28bbc2ced33386f298650d91218076c05420e6cbd790b913adc41659e7 \
    --hash=sha256:1db38f4c5496827c1a501846d64d14c3b80c7e6714e406cd7dc36a9899fa1011 \
    --hash=sha256:211d5a3eb6af8f513b8d4ca19a8c1b7accab1b5f0d3175f9826b03c1a920dc1f \
    --hash=sha256:23851fb4e1b85ed3f6c2a27b777cdfe2e19fb5b38429a8faf38c7542b7665869 \
    --hash=sha256:254eb48b9fa5ee9898a3c445825a1f340fe53712a098904b39b0bddba8ea3cb1 \
    --hash=sha256:2625388c6c754520c37abaf3b41eb34d1cc4a373f457898f08606c8e362b891d \
    --hash=sha256:281cb91036248400f4cc957495cccd44c275c2e0c5854f7e45ac5cf7dc193847 \
    --hash=sha256:28a15fdad492a99b6eccfaaed66ef3f74050680545ea61ec8b2f4c538f1f1320 \
    --hash=sha256:28b4f0d66fb834ff90f28209ac7bce77868c45d8c93e26f906709d9b7c2e1af9 \
    --hash=sha256:2a925889534b3748302dae5dead07cc13480de1dac3aea80a941b729b471ef93 \
    --hash=sha256:2b7b3bbfb4fe8ef40600792d762fbaa9057559f9d3fad209525b7a22b99e91fd \
    --hash=sha256:2c9ad19a6cfcd5ea5c0d41161d22f9df1dcc277e9bef2751391334546a314c00 \
    --hash=sha256:2cc961b171b3f3440f410489ab3573e86aea8736134ebbb40ea1338b7f0831bc \
    --hash=sha256:2ce45c6627b22c47e390bc91a41c3d13032192e699fa0bea96e9671b373d69b0 \
    --hash=sha256:2e06a3a98f916dd41d27f3105e02e7a40181c98c94b9158733d03a6f80506c09 \
    --hash=sha256:304d5463e65a35d7bb0850550e0780395395f6fcf452f04db7d5ca7cecc425ac \
    --hash=sha256:304d8e4d493af723536393eee0c689eb7813f4a474c8b479dee63f1fdd98f621 \
    --hash=sha256:30fcd120b732aa79317f08dee04d7de0847822e4cf7ee0e9f445bb958832252c \
    --hash=sha256:31f3930700408d211f13378ccbe1c40845d8da54bd0681fac3a9b5aae81c7aa8 \
    --hash=sha256:34276fd796040bf0993ab33a369aa572e6979c7aab225a88893667ad8eac8f7a \
    --hash=sha256:355ad8011081dec5412240c087a9a0c9d4d5039f3ed11a3f13e18c2b29b56c51 \
    --hash=sha256:38a873987f3be698494da8b2e3085e29da02da7b633dce73e79c699a113d7bf0 \
    --hash=sha256:39de2a259fc954455c57274dc94c79d5842774e1247a016aff30bc0efed0f4ef \
    --hash=sha256:3d14b50de6bf4d0edf857a9386836846f982b8f524e188e2e68b96d702bcf4aa \
    --hash=sha256:3d21b8b13c7592db2ac5e544a6d83187b995257472b0c9e8351b6d507ae37ed6 \
    --hash=sha256:3d31298449090ab8d47b7b1b2a555ff73cac7ed438a08b7ac160980c7ebed649 \
    --hash=sha256:3ddacd27458c45bdacd6bd6db644bfb730efbf9e830310186e3045c9c5be8fb2 \
    --hash=sha256:3df041de8887954562c9b261cba85ca0e9ded74048daf125f45edcfaa4832229 \
    --hash=sha256:40ab6bffa02ae10a0581e6c198be7d2d8ca5c2a0c64e4ed3465d766df457573e \
    --hash=sha256:4275811936e2f06feff5e598fb42a1b7ae852da8e39605211892b56b81a34efd \
    --hash=sha256:443eae2bf318abeaf6f15d785138f71fd6de770e99a92158b8b814265e079115 \
    --hash=sha256:447441e76ec720b15e64418d32e092297340387053047c7c694f579efb0ee1d9 \
    --hash=sha256:4495c5002a7b28557e7e222e77e0b661183e432b7d6d2e788101e3f240e05b8c \
    --hash=sha256:44bd4fbb29dfbeba60e7d2bd000c59e4b21ddb3cc53912b14048d37092706d7c \
    --hash=sha256:4685902cf26edf013ed7a3da0f426ebba7a00ebb9541386d835afbf002c11cab \
    --hash=sha256:498dc3188ca05a68231ac3fdbfc7f57eb67e1343c30e0fea17f8218c1599b253 \
    --hash=sha256:4c2b5031f63e331e3839b40aed2dd6f191e9c07edbde303e7876846ea1946995 \
    --hash=sha256:4d48f2d08b9de5864e2c8744d4461b862fb149a18274abc8b698c45975573438 \
    --hash=sha256:4f87960d57feabfb618e4e0af6e7371645fa26a277860739d6e5d6e0012c92f0 \
    --hash=sha256:50e3adfb96fc189eb27b1cf62d3b598b89b4bb0420d93a3d3e42e137409011be \
    --hash=sha256:51cf45226a9b588d0d2b4880c62d686934b63ab0bd79ca23ab0e9762eb27441b \
    --hash=sha256:52aa6992700996af31f375de0c6bacd402b0097fe40b53c426b9f51a90ebabc7 \
    --hash=sha256:55ea99acb17b9325618de155a0cd6a2e8f5d10be008113e1d433bbb58db543b2 \
    --hash=sha256:56bc200a365efb37383b7852e4cc5898d3b2da5987289b543956cf8cad71018a \
    --hash=sha256:588461c2e8384d309bd63e5826019b6977bc66d629b99ac8737bb795d7b2cb5a \
    --hash=sha256:58ca3755ee7ff7f59b57789ec9833c9de9ea275405cdd240eda1f193112e398a \
    --hash=sha256:58f361dcbab699cf8f42db3f47c8e7fd1036f138c23a5d08de9fde5f425a730c \
    --hash=sha256:598a11a2c7ebaa5334bf698bf29568c9c390abac6a154d8170fedecd1cea38c5 \
    --hash=sha256:59f63901b0031c3136cf64704dcb21de0bbae62ce2c9529bc39d27665463de37 \
    --hash=sha256:5cde776b7cc66e4f6c99612cea4aa7269aa65863f7a15841b2c264f103822f4e \
    --hash=sha256:5e2b6b57e9733d39f0c9fd3185efa6b8e29652c4cd8fe94180272cf6ed9a78c4 \
    --hash=sha256:5fb29fb8cd1a46c27a1bf9613ad5ec2599310d46b4025d9556404a6b6a292800 \
    --hash=sha256:6045373d5a89a5ec71afde535db987ca28e76dfa276c2d4c818265b375d4b055 \
    --hash=sha256:619799369eeef6366ed3e8755a5670f4f2f0fb6b30a0fd7264dc0fdc2357058e \
    --hash=sha256:62588a277bfb59def052abd940703fa35107152bf479781a878617d60faf8fb5 \
    --hash=sha256:62603db9a7caa0802eaa28c1c46fecd7b3a263a774069c24c3c28c302448721c \
    --hash=sha256:65cd72beeeca9d3aaea1201e5923859f308f952f9c71de93f06063c79f0f7a3b \
    --hash=sha256:68eb192d85ab8e5f6ec69c2bc6ac0179fbf04a5ac1569d12fbef74883fe102d0 \
    --hash=sha256:6bd128f206a7752ae1f2ab6c61bf8a24ba28913a10df8b14c2637b973ff97a80 \
    --hash=sha256:6be488a102b8cf28d0391d8c4ba7748938ae28b78ad901f8585520fca33ead1a \
    --hash=sha256:7218e8f32b0956cfcd048fd42d9d5779809745ca1d86113ca56f66e7ae1549c4 \
    --hash=sha256:7441d755b7ab94f8d4eb3e43ec05482d760842fd263d003a99102d742cd835e2 \
    --hash=sha256:749e97e1b32313717a565abbe321bc2190bc8b35f1a67e4cdbc7c56c8d8ffe58 \
    --hash=sha256:75a3ceed0724d625d64b86ca20aba182e4df462e04c2414fc941c0f523f06aac \
    --hash=sha256:780fbe7cab297b81dad9fb8dc5eb003c0468ffb0d9e5f65068c53a34661a96bc \
    --hash=sha256:78456a747de8dc58360ffa581f30a002baf5aa28cb262536545e91f113ed7639 \
    --hash=sha256:7967d08cf06dee78443b874f98c98036f624f3a4e73e11f9f64f5be4d25393cf \
    --hash=sha256:7a881931aa470808df94a8c380eed2bbbc76cd9dc622310f99665658c821eb6d \
    --hash=sha256:7dcd882da75ef9adf94903b1e3b9419e8aa8fb4c7396822b834b9ef7fb96954f \
    --hash=sha256:7e841fb9010836c992c9f12fcbd43a831de93a5f726fc1ccd8ca1d0268c5014c \
    --hash=sha256:7fdde2c9fd9e3eca40631e024664cf2584272cc8f96308cbe5fdfc930f51d8bc \
    --hash=sha256:8024d00c3faf3fc0c16e07a69f4405e8eac7cc0ab15f65fe6cf43827c4cf72b4 \
    --hash=sha256:80d02b6f04e92601a081dd97b23d3128033098bff5d35d392ddcc0476ea11253 \
    --hash=sha256:838dcc90063569a0448120554591a1d6c4a4ffe11babf048908793154ab86ade \
    --hash=sha256:849df64e889b2e17230d58410a03dba311a65b163508fd33679b2b737d4b7858 \
    --hash=sha256:87475fabc8d9996fd9c27debb395e642e8c838d78a00b6e932227a0e06b81e26 \
    --hash=sha256:87e50a3e7cb90af586b6c5faf23e302a970415ac73bd7bd90a515a04b427ef96 \
    --hash=sha256:89b53f3cda69831909888e0494f4fa0bcd3537e3e138dabeb620bd6ad946bae8 \
    --hash=sha256:8a893cc101149f80a653f82062ebc95b34525a2614382e1da5458fe7c6997249 \
    --hash=sha256:8b2bfab86aa71ae13aa41a6a26aab338e0db2b8bc75434b05aea89e011ff35a4 \
    --hash=sha256:8d86d6fc60743dc916eb79e2eb1ec4818e21e427731543af40a3021851174a13 \
    --hash=sha256:915563965d418f986e7e145accc592eae9e1a1be3566ff98a05d7a9ec42a76e1 \
    --hash=sha256:92888bb3187c5ba50500b00b3b310c9f2c651709d28036077680cb5255450a03 \
    --hash=sha256:93223adc95033dd47133a46ccfc316a0139176fd79085762e27202ec56018f03 \
    --hash=sha256:9373ad13ef0d2c0fb761e04e55bfdee5a08b52cef2c882c8fbe9935b1517152e \
    --hash=sha256:9409a8bf35cf78353942504b24a57de3d75b708997a1e4bd8db71ac8633ce364 \
    --hash=sha256:9b7f416ff0978e2f2249330527f0ad6fa02f4932e6199692d3b52da2048c19e4 \
    --hash=sha256:9bde855991b7e362c146535e3136a50bfaffc0487d38b33ca7e5edefc6e23849 \
    --hash=sha256:9cae88599c7219005d879f98e5ed53341e9a122af585e1091200358a3003d2a0 \
    --hash=sha256:9cf9b1a857e25c4baceeb3624e92a56df3668f398c4acba74e174d81fb4d1d3a \
    --hash=sha256:9f56f72050826f63dcee7a7f55b0a77168cb3bfc553fd405e7f8f9ece75a4036 \
    --hash=sha256:a090bb2c68df85450502e3e20d665e3a5af9c65a84d6508ed477badd49166fd3 \
    --hash=sha256:a192e2c40070d92c3ccf777e3a5c4ff515573cd2bb7ed0c537fdadbbec5bbf21 \
    --hash=sha256:a19a731138fc27d5682277d3b9df22855cea1239bce7fcec5f78f42ef2d1f3c3 \
    --hash=sha256:a66c3bc5ab1f0ff2164fc9965ddd611ff0802173f4b9d24554c563f6ab7e1d6e \
    --hash=sha256:a815775b6c38d4e0ff7bcffbeba67feded90202bb6a226b8dd35f1c855217413 \
    --hash=sha256:a89012d6d5476ee112d20d998570ed58df2260a852afb1758809cd6900411d21 \
    --hash=sha256:ae4f5fea5b8b8ccff88238cc8569303e5ee95efae67fa62922a311397a71f346 \
    --hash=sha256:b6856554c4f44d79fc2307d5768854310a8f0096e501c75637542c82292b0429 \
    --hash=sha256:b6b751274acb69d77b3323d6b7dbaa3c7fdfc1eb829b7eb61d262f32e1af9685 \
    --hash=sha256:b736353c0a625bbd5fcec108576e2385db3496f4f771f785ff32e108d3c3bc45 \
    --hash=sha256:b7fd005a73d9e657273b7a10dc71a9e03c8fb9ee6999798d6918ce095b81ac7f \
    --hash=sha256:b91363207bd9dc966a691e959bb47f64b30f7ac4b072be9968b366982f7db77c \
    --hash=sha256:ba0b1d2620edf869789c3879223f52bf2afc5d31b3cb47cc57b3a12c05e2aa9d \
    --hash=sha256:bbbfc8e28816f19d7c0f1816664980c0a9875d01b27cdf8eedddb639d9e108ad \
    --hash=sha256:bd16aabe4a02a297c23417aa17ac6299dbd8c49f673bcd645b4929b11f5a4400 \
    --hash=sha256:c0afc6800ba57ccc350374c5bd6150419915d95ce93cdbab2d783d75eaf30ecb \
    --hash=sha256:c6708715abcf3c73b99508253e961a9967f02fe536532834149574eda6de0d1c \
    --hash=sha256:c7c9ab723cde841fefb34efbad91e87f00a674b1fe1cd0784fde742bf2c154dc \
    --hash=sha256:c8f3d67aeaf55f017982b73683f0e7342ba2f6635a78f69ce89ebb26aa411e5c \
    --hash=sha256:c9790464842f85f437dbbb54417eda1e0e6bfc52dd8d22d6fd1c994b73b2dc74 \
    --hash=sha256:ca403d7e4798f525fdfc78e258820419cbbd0f0ecbab9de7840e3c017cf6b8cf \
    --hash=sha256:d008d90a7f2471519aef0c90dfbe73b3e6e4d5e66ac48e19154c17e89e98b604 \
    --hash=sha256:d19fbd981a488e22cd04883659ca6b08f50b5974f9fd7c95655ef6a043e5893f \
    --hash=sha256:d1befeed746d247c81127bb14de9dc3d30edb6e5976d34f83f86ed262b1d9105 \
    --hash=sha256:d2374b62878abb00cd8309b32af6c0b715cd02dec0ca74ef12e5069bdc64144a \
    --hash=sha256:d376bbd28b3a8999db1a103b3b388aee6f1ddeb3e51bc2172993efdcd86e064d \
    --hash=sha256:d4a7319f304a774bed22115bc891618e45f85065ab44ea6acd07d274e750519a \
    --hash=sha256:d6734d2ef8a50fbf8445c139477da401f50d62a0606bf00e20ec6d87773fefb1 \
    --hash=sha256:d760fe2a4d7c3b226cb9026d6a842868d52a7901bd98420e1baf14e80da85cf5 \
    --hash=sha256:d913de495d90407cd859d263bee2e5d1a4ed3eb6573c04e70d9ec619a7cbed7f \
    --hash=sha256:db19d07e2e0129e974a0e65d0064fc222a446cd5122c2fd4184d2af9fc734a9e \
    --hash=sha256:dca9ab98072a5a54ebacebdc45f53e645336b320c667410b061be1ca588ae709 \
    --hash=sha256:ddc7dacc8ece3a182e7f15cb862d1fd616b46d076cb1ae9dd232b2c38b655874 \
    --hash=sha256:ddf19c062bea7a0cc80f519243d2c01dd091be0cf952a0750d4ad576709559f5 \
    --hash=sha256:def79fa35ef0cef8d2accec024f4fdc7ead3012ff02f5215c783f39f03ef8cfc \
    --hash=sha256:df29a0a7107f7011e77f4eebdddec4c7331e24d787a0b21a46d63bdf7445da95 \
    --hash=sha256:e09a3942ecbdee5cce73ea9d42da82b81b72ac1bf031ce069b93b5adf4eac8cd \
    --hash=sha256:e242bb1c5e76e97dfa9e7f209a71e93a01d7f19ffdd5cfbb2e2d55b4f08f8ab0 \
    --hash=sha256:e243bd13217235fc7290c621941c3f5cc8b66e4872495be821d7436ba2fb838d \
    --hash=sha256:e2af3aad578aa6bd1384bcf4750fc285e5a9de53f40b7d41e5a0bf748edeb2b3 \
    --hash=sha256:e4e81e09c1578b8df602e3db08b0b3ea0a6947ad612f52bf8dc5ea8d47691f0c \
    --hash=sha256:e54da4baf05720032d527874d40b65fa4d7e5c6c6a43d0c3adbeffcaf275a2b3 \
    --hash=sha256:e80e6c2f55656b4824d72065abb4ddd6a525c74bd78a0aab5d9fc2cf4fb5af50 \
    --hash=sha256:ed2a239c0ea213acc1908150a3037257083c7c083128f1a4cec2ec4b97dca491 \
    --hash=sha256:ed905975ab14056a2e5eb1c376cb2e1ebc5396baf84163939c518556fccde9f5 \
    --hash=sha256:ee21e28f0430bd6dc9086c6e525d5e818a44a5ad19720c8a0ef766792f3eb5e5 \
    --hash=sha256:ee43c17b173d46a3212baa6ead3ae258eeabdae48c263a01ccf0218c366dd655 \
    --hash=sha256:ef4fcbf3327382cd4c9f540babd61248208af7b93eec4de397b4d5f58a09e288 \
    --hash=sha256:eff0ac9dbe711a4aee69bf04a83896aa9b85f19641264053a9f6d48573abb7dd \
    --hash=sha256:f0aa869112ef88429ae17820d99c3dd9504c9e9c671d3c246f3d7442cb051084 \
    --hash=sha256:f3c96f633825733f735c5a9cf21d21a257d8e1edf0b1cee0a064b9c424ca0f7d \
    --hash=sha256:f5833ad231be5eb6553de524a70f48d71b2c8563101750531e0b80184e175cd4 \
    --hash=sha256:f5ec61164adcec446f8969a3358ec3f9b26bbda3b9213e5586d219afa8df2915 \
    --hash=sha256:f7d486c83842422badd511868fd8a9a20e9407ace71564b6af47ce7e60a336c1 \
    --hash=sha256:fb9e68df06293761f9fe66ade60a9bc6d0f5e42b8acf2939a9158af86ab0e5bd \
    --hash=sha256:fc14a032f813bf5fe624d991960ea83e9715adc27e4c1830a2361eb1d02ac341 \
    --hash=sha256:fcff63213e8e6e47770541a4607175404f47cbb3ebea7b6058cc82d524a0e424 \
    --hash=sha256:fd1fbe0f116b6e55da77aca2c6ddcddcfac2186cbf78bdebf40fc156efca389d \
    --hash=sha256:fe9753dfee015c570d73df76f899f18444d41388bffcde097deba51c4fadbb9f
    # via requests
cuda-bindings==13.4.3 \
    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \
    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \
    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \
    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \
    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \
    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \
    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \
    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \
    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \
    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \
    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \
    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \
    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \
    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \
    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \
    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \
    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \
    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \
    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \
    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \
    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \
    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \
    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368
    # via torch
cuda-pathfinder==1.8.3 \
    --hash=sha256:e29e59829c297a7a5233bd9cc71094fc5bddbd076951482670178f9eade39b1f
    # via cuda-bindings
cuda-toolkit==13.0.3.0 \
    --hash=sha256:d693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f
    # via torch
filelock==4.0.12 \
    --hash=sha256:5f17ee83ecee8a6f3e389c75822fb70a1c2f0506b99438a6dffa1d56793588c8 \
    --hash=sha256:cf42711a7ac791818b299fab0332a088c65aeeefa36290de98db92c434303b0c
    # via
    #   huggingface-hub
    #   torch
    #   transformers
fsspec==2026.9.0 \
    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \
    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f
    # via
    #   huggingface-hub
    #   torch
hf-xet==1.6.0 \
    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \
    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \
    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \
    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \
    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \
    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \
    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \
    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \
    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \
    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \
    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \
    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \
    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \
    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \
    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \
    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \
    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b
    # via huggingface-hub
huggingface-hub==0.36.2 \
    --hash=sha256:1934304d2fb224f8afa3b87007d58501acfda9215b334eed53072dd5e815ff7a \
    --hash=sha256:48f0c8eac16145dfce371e9d2d7772854a4f591bcb56c9cf548accf531d54270
    # via
    #   pix2struct-textcaps-pipeline (pyproject.toml)
    #   tokenizers
    #   transformers
idna==3.20 \
    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \
    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c
    # via requests
jinja2==3.1.6 \
    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \
    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67
    # via torch
markupsafe==3.0.4 \
    --hash=sha256:007e1ffd9bf65bb6ee96df7b258fc632a4868dd5566037986c64781f35a36e98 \
    --hash=sha256:02fa4acbc6a3fc5c693c34d4dd8c1130b7fe99cc915181b0ddd6f72aeb296002 \
    --hash=sha256:03470d1a8268e692ecf79ecd565593e59d44219377a7ead61f1f1b94c1f7ff6b \
    --hash=sha256:04e7902ba80ee4bac1d50a549606527a1dcf0476cd81403db41099d3b60ec653 \
    --hash=sha256:051417f74bcaaefa316276e0ff723f541616ca51043d070da00249d9bddd3e3c \
    --hash=sha256:05295589e619b9bed252a86b532b8e27350abc372d18ba89b59375325e91ec1e \
    --hash=sha256:06de8ef6331f6e822c28d577dc8bf43fe398800477c49498f38fc38b67ff33fc \
    --hash=sha256:0764a13d34cae40db7bbf3a09b7e9b491bf4603e20b263a7a9d6b8e324975d0a \
    --hash=sha256:077293e425f28ec737dbcad442a71752e28f8ae27cde3d68acd1fb212091cd92 \
    --hash=sha256:0930db9bdc62d22944e10b066448bb65dc9abe9112880c7cab8da54db4284d5f \
    --hash=sha256:0cee7cb0f9a1b6892ea482237d9403b3d1b4603aee057d0ff01f0fac2d019a97 \
    --hash=sha256:0d9c47709875fdb321452056622e930c52afbc07a7d780762fbb8b4d91ce6fa4 \
    --hash=sha256:11935df9bf455ed0c04eb87bcd720f02b1fe5e02128a9430f23aed6f93336fc7 \
    --hash=sha256:12a606a492de952afcb43b59a14aaaaad120e708d3663dd0fdf2d738d427a691 \
    --hash=sha256:14bd2d845d62ab678eaf81da89d7b621b51756c72346745c1a594c09d49207a2 \
    --hash=sha256:15ba9e28640feef770374b116a6f019c21f52404aeabe516aa7f800587b98cfc \
    --hash=sha256:18a801868a884f216e784d7d14db2a4077143ce7610440aee2ce8f734e7cfcde \
    --hash=sha256:1c0df495a977d10460a94941799c72d5b5ab03d3858d949b55b5a66c8f371c99 \
    --hash=sha256:1caa2fa5a6184fb233153b35f654e6687bd555476f6170f29d8ee9be1a8b0af9 \
    --hash=sha256:1e1451fab512d1bcc3dc26988ec1edb0b82c2db909132872cd9356070a6b63df \
    --hash=sha256:1f1f9477e174582b0a1b583d60b66e1f2cf5d3fe12cee985e4aedf44766600e5 \
    --hash=sha256:2628d3a8cb648ecebb3c5d6b0a1052d400e4d8b7ac0fb786be8d285b50040d17 \
    --hash=sha256:26e9867520db70d37f7fb421a7f0d8adb40171011fb84ce869afa1a83370dfa8 \
    --hash=sha256:2a6ef68ae94aed8721934072b27a3b654ea2100b97e4ab864cf1489c90926fbc \
    --hash=sha256:2b2b1e18af909b448bb3cf9e3433366f7a8726271fc214e8b10e0f62a78c724b \
    --hash=sha256:2cb3dd71fc6be918ad4264346a8ed69485f9b7ed7bf35495d8e22807cd6b8bea \
    --hash=sha256:2d1b7d9308288661f56672b1b157d75fc536714d3638487bbea17b6318a78248 \
    --hash=sha256:2dad610540cb2e6272855c178f08ae9a1c7ac258a7fb71660553a5f104b42741 \
    --hash=sha256:2e5a7cd7fdd14fcb1ae5d7d8bf23d24fbd1daefd1fbca2580132e1ea75f098b5 \
    --hash=sha256:2e9ad7dd851bf45fab9f75cbff4cb493fee9979e8d8c7c9c3ee119022518edd6 \
    --hash=sha256:340cbb1957ba99929cbf19a75626d36ba1ae21d1730b287d1cf7f824a20c4fc7 \
    --hash=sha256:34bdde374c5932765d7dc685c4a1d191a3207852d67e8e0a9eb6ea85156181f1 \
    --hash=sha256:353bd63081912ab8cfa6a0c7d185934cdf8426f04c618bba6bc4b394f2069b67 \
    --hash=sha256:387d8cd30e69b3f0a72877b9ae717033396404e19095b17fe89753a981fda44f \
    --hash=sha256:3882fb412298575bae3b9c46868251f15cc69307359f87bb1b382e53d6e5a2c9 \
    --hash=sha256:38fc55594dab834470b6733dead2ee9e3f657fb0608c769dcafa0ba5ab52f45c \
    --hash=sha256:396ec4e65cc889f69786b3b89478b471cee5a3bcf468b9d9bb03e1a30fb291fc \
    --hash=sha256:39dbacefc411633db5b4378b066a9aca70a3d7e2922c9e578d825f844026eeba \
    --hash=sha256:3a93d9616ddecfb393727a0041a562cf0b15a244e20f2bd25efc7949be4c4f17 \
    --hash=sha256:3d23795802fc8bd72534836d64489bbf0f67c088959091bdb22e10735a5107bf \
    --hash=sha256:434139499bb20b502ed3baa1f169e618f924a97e7a777fea1a49446d80106cf6 \
    --hash=sha256:436e3ffc6310d3c41878c601db29098102fe5d8a467c49da4a4125254e0980f2 \
    --hash=sha256:489505b03f692c3f376394e49194fa7a7f9e8558d6e293a7056a0032b0c38163 \
    --hash=sha256:4a540e2d3192792fc84eced57bef37851ccb2b41f73291bb17408eea77bcd278 \
    --hash=sha256:4a7cdc2a420ca01058182da4253329764d4bfa055564d1eced90e6ba1e8b1d3d \
    --hash=sha256:4bced6e2a6dba6a28f7dd3c6ce14df1b2dd495923f16ea484cad03decd463b2b \
    --hash=sha256:4cf3468d5ec187ffffcaca8e61929a37448f215dafc1386a12c750a72fe53634 \
    --hash=sha256:4e2c4809c14559aa7ef426f27fb35afbb38104c349a903bf8f3600456764bb38 \
    --hash=sha256:4ed644d75aa94a2baf7ec3a96eaa160ea58c742eb9d27c6506053c5c40fc84ed \
    --hash=sha256:4f6e0852a0283b1b1fd776eeb7b766a5f440b3e2bd31ab51af3b400585f3965c \
    --hash=sha256:5066b244f576f91afc8ee3ba029a89f99d39c79b1853fe9d39bea9f0afbec148 \
    --hash=sha256:5086f9975abb1ab531ee6afca1761e4b59a19b446f3f6522ed776963228cfe5a \
    --hash=sha256:50b5bedc9ed8a94fc8857a42ef4f84a81ea88f8d4f05dc8705fb23ee6d8dcca7 \
    --hash=sha256:52704c5d36eb6dda8866493decd61111fff86244c9b1ad225ca01b9e91e5970f \
    --hash=sha256:55ffd6ce583d97dc71dc92e930324c8c0d25aea7e3ade6ae54ef77cedb096811 \
    --hash=sha256:569d65055d367e3dcdf30c3f41119467b73d9ee9faf332bdf40402644f5ac08e \
    --hash=sha256:57f9947a7e57a081c1e3e0a2dd0d2dcf290a4531450e6f611e30084c222a7295 \
    --hash=sha256:5989cb26b2e1efc6a42216a9f6b5ee495ce5ace2e5b352a9af489976b32d1ee2 \
    --hash=sha256:5c22873ad1f0532ba40fa1727f3c0fc1bbbaab6d373d4cbe3f0dc74b2e2521c7 \
    --hash=sha256:5e8b3d0b18fd623afa12ecb2ce8d8becef69f9b5440c6330c7972200e0bb84b0 \
    --hash=sha256:61631e08084be9e21a8967ec3139c7616ed7c5e9368e05c86d1b39562c8a57b6 \
    --hash=sha256:64511c54db4e4987aef4c41923235927428729e8174c5dba488429be70a998ed \
    --hash=sha256:6669c1bf34080161ce49c589cc512ef24d4c704ac9d2b2d3667f519c60418378 \
    --hash=sha256:672d207103e6b16ca098611b0f9efad6bc00afd47c03d6ef62186495ca677dc0 \
    --hash=sha256:6768d67d1bce64270e0fdc2e69309d68b9b18ae56ddf6c711d168e9d051c2cac \
    --hash=sha256:6a45c3d514f2436064db00d7fc8778d888f0236ebfed649b53d13a59e69ad51b \
    --hash=sha256:6bd9e1788e15bfcf6a9082de42e30387e7b85d211ab21e57a939bb8cfaaf8d96 \
    --hash=sha256:6d2a9efe686f9de00d0d1ea32a4a5a86d558a2277501bd78d964214eab625e59 \
    --hash=sha256:6da83a088f8ef93b2d483a8232a4dbf4d69d3d8496b568a03c56becac43e1808 \
    --hash=sha256:7018d4af1cd272e847aa5917983ab5e83e4f6579f9dbfecd4a79c0ca80b144c2 \
    --hash=sha256:71f88e749ea29f67f21f3b36433c1dc54c7729ed2a6d9e2da2e0d9e0d7b224eb \
    --hash=sha256:737c9c3981998eba27f11786f84fddcbabc74068b72a4a1f454ea02094b57b65 \
    --hash=sha256:73e77980c7207854f00fc4e71fb1626868d5740ab4012623d55c7a99ad122a72 \
    --hash=sha256:799c39bdf5e2f1292fedd3009f7b3c9e760f10b2420cb9638d56920840ff6db8 \
    --hash=sha256:7a83aa6e4805df46fed18e989d3d16f86ef60cb50bbc8d9ce3a6be89165fbf6e \
    --hash=sha256:7d3391b2188d18737cb2fa147028b1096236eaa7e156446c650a489fa2cadc91 \
    --hash=sha256:7e1636da3d8dfc220b6dd10264db5f2b165e4888c4518594898fbe381049af8a \
    --hash=sha256:805c8b84534fa10891890f0e4be39f3a99e94615d93e8836bf9fa1fdca2feeb2 \
    --hash=sha256:811d02d5122171c1941357efd8f9bf4ffe907b7f0a1a4e729a880e4be3f46e3e \
    --hash=sha256:8138eb83940ec7299024d92d4dee45f601b9e6c5ffde9d25f4e35e326203c707 \
    --hash=sha256:83b3944fea42a8400edf92fd1770fb8d0d4f7de651353bd2d8525a92dba69a21 \
    --hash=sha256:849dd2bb0e5e4ab2b71c7191726a4a8d5aa8a610daa584728cbee0b710ddc4ef \
    --hash=sha256:8698d70a8081ee8c090dbb394768b5789a1da8b131b5499f89d071dd3cfaf6be \
    --hash=sha256:8781a792a070cf2bd1b86d3aa943894115faaba6e88122a7bf32d62072742453 \
    --hash=sha256:88d59b473bfb03259722600839af9bbd7fa13a2eb514beefeedb95997882f69a \
    --hash=sha256:8909c2f1c6dd65e054ac4b573a91c8384d1492281e55d82d159d653f7a13adf6 \
    --hash=sha256:8965520ac587c94a4ac48b729be3d8b8de00af39699b17585dfb599babe77977 \
    --hash=sha256:8b5d563170ff8ba3181caa967c99a3c804d1dedb702c7cb93a6a7c32247da978 \
    --hash=sha256:8e124f974786f831d6043728e38296969d3579db8896fe004682f5758e613581 \
    --hash=sha256:8f0fac8b13d14bb06c68195f849371924ae53dd7b1c00fed24650f704383b692 \
    --hash=sha256:9240187afb63d2f9ddc3e032c670356fe941f6e20662ea168a5dc3f1f317e1b3 \
    --hash=sha256:925f929d6b59a8b3f8b8c6ac363cd0af7eecc81efb3071770b3c6717c450a369 \
    --hash=sha256:9348cbb300d224fe3b89793262cb093504d4ae927004468463f745188a193e4a \
    --hash=sha256:9388003072b95f2f1e3fd908604194d653ba21330d811961a78b7da1a77e9e36 \
    --hash=sha256:9438a2648b2195980cb2dd8e53ed7b8df91319e2d0b70ae61a9e1d1bc8d3bec9 \
    --hash=sha256:94e4c421742086aeee4c32a506eec8859d7634aad943f7e6aacf70f813478768 \
    --hash=sha256:94f5407f7bc64fa6463906b896f9904beeeb7dd8dc116ee8e9056c8714ff9916 \
    --hash=sha256:971a3bbb75d97ae4e2e8f7d4834236f86f85f0c85e04ab2e191db1123b04f80b \
    --hash=sha256:9e227f3dbe6bde7491cf0a9965d00b88c6b1a4a95d11480ddf88bb96d397c19f \
    --hash=sha256:9e25feb9e330b63edb0278a0acdf85e50d0cb0fbf49c3084abbe4e24ae195346 \
    --hash=sha256:9f098115c247e11d138ab83a28fa0323c77015007ea2df73ba5fd714dfefd67c \
    --hash=sha256:a18f38cafc329bac5e3c2b96c765b4c96d3d103421ed22ab7988c1e3fce27464 \
    --hash=sha256:a4bbd2d87dd233b9fc5812160c3d0ffbe42edc22a26ce0469f58479ede633fe9 \
    --hash=sha256:a5fcffb37e602b0b3c1638a97746b9b96125caa9bcf6fa41d337a9261de231ee \
    --hash=sha256:a8e9f292fcda89b324f2f5c91d13f1424a153e40fc2756f38ee23b15835ff300 \
    --hash=sha256:a9f54054101545a9a9cccefddf54316aa6e4491611fcbef9e91b3b6bebec04f6 \
    --hash=sha256:aa2c838cc024642cc04c6854232f32b43e5e22833dd11119c1766c7873b8370d \
    --hash=sha256:ac0c7c9f1609b0c4c114feb1d7a3409564c7fb77e360bed9e97e5d25dfeaf868 \
    --hash=sha256:add96447a86d205ab616665d53b2950ee81083757f56e6ea833c8b2917646b46 \
    --hash=sha256:ae9dcb8fbe244cb82f8a6458b455b927a03685e383d9bacf1ea5ce180b96dc97 \
    --hash=sha256:b4a635a0487774f841cb1fb62e907e7195cc95bc761e053184b8acc3ceb20733 \
    --hash=sha256:b4d12837e0203bbace818ff4a7461afdcd78bcd782351cea148139180d7bcffe \
    --hash=sha256:b61687d0828e72bf5cda24a2690188f37170bd31c9359ac97e4e66569f120a16 \
    --hash=sha256:b807e598953730f82e4eae3bd30f6a122cf6b31c398c6b504c0e04c13c170429 \
    --hash=sha256:b8cd1f918b26fd7b1832ece557cc18f2d8747309ff8b3f0ef9d4250c5ad67a39 \
    --hash=sha256:b91cc9d336957239ff200f30097e6fea2dc6d6fb3c81e853eaa09eac904fd894 \
    --hash=sha256:bd3ce56ae2cbae3ba82b683bc425cd7e48d2ed8b10f3e818186b6f5646d9271c \
    --hash=sha256:be6cb0c799abb0e2ba3e618e6d28ddddf7e485f6c2ce938dfa237daf3905072c \
    --hash=sha256:befb4158af32106b9a93db8d6d1d1cbbd418c0d5aca0cabb7b1780abf0c89169 \
    --hash=sha256:bf053da3c97a4bc5ecfbb218cdd2983febd91c617be8367d139882aa11e490aa \
    --hash=sha256:c02e8f18bdedba082cef725942ac823b9b60656db07f7e265cb31618dfd00d77 \
    --hash=sha256:c1bc67752d5f21013cfe430df4062441714eab79f65a6a05e01505957e9c35fe \
    --hash=sha256:c61750fadcd119d0825bcb7d7d675dd264dcc89cc05292aab5be68ebdbb374ad \
    --hash=sha256:c90d5b3d4e944e065a301d741b3c1d784f6bd1f503aa68b4967e32b2ba313d85 \
    --hash=sha256:c9a7f43c0b202b334cc9184af09bb8f21d3a209e038efaf106936fb69e6b026e \
    --hash=sha256:cb96e6e088d6cf71c1ea977510948320234824cf226e32f6f6e044f7a9c82b34 \
    --hash=sha256:cf63c214fe879a65e69a386f915e36104fc84254ab141240f8854602d8e0be2a \
    --hash=sha256:d1aca03ede943eb80ab3d63bb082c84b7aab85ea83bd0fd0c200260945fb49d9 \
    --hash=sha256:d2e56fd3b00222722abfb3f5f0759ddbae4b90811b5ad4343c64030ad1bde70c \
    --hash=sha256:d5f93ebbeb8032d47e349328ec8662d973d9b05a70b3c35df1f91fe419b84749 \
    --hash=sha256:d882a373d8093c2941e01291b7ced96e9cbe4781da9a7751ca7e6c70385e5214 \
    --hash=sha256:d920abdfa61279ba1a2ef9484aab07bf03331f8c08a10120fa332353d06e6932 \
    --hash=sha256:da2af0d7aebfc2074080d72efa6ab8317c62481ef1f896f65d9999c1c01f4494 \
    --hash=sha256:dd8ea6ebee7aedbf7c749fa80521d9ccf1ba473e0d1e14805caafbaad281c889 \
    --hash=sha256:de8b364c423ef0a4bad9069657d617f9a5d2b2062457a89b1fa16ee199c399c1 \
    --hash=sha256:df1ae86ff54725a01fa1a0510b914ca53a161b7050be74f6204e24aded5971d0 \
    --hash=sha256:dff05cb7016dff1e9fd68f4122c127b65dfc59de5306cfb7ad92f956f230bee2 \
    --hash=sha256:e1a622f13970d81f95d0c72f9dc090dce9085fccfa4c9f2174377ee32bd15786 \
    --hash=sha256:e49fb0d1ce92cfa0cb198cc5b1b11cdf9d0638658e2a2db2687e39db7c87fc78 \
    --hash=sha256:e5c802729725bd07e2bc3ab7b76dc7e0bbfc53129d8f1eb1c002c24cf774717e \
    --hash=sha256:e841068dc0be4cb6dfb5c890eb88cbdcff2f4a332393c7ec94e8e618bd32c1a8 \
    --hash=sha256:e916035e3e9930cbdfdd10abf48861340221857f45509565898e012263f7b289 \
    --hash=sha256:eba154571c16e032112afac0dc2dfe9e63c2ceb7aedd07bb7eecf2ce26d4dd4c \
    --hash=sha256:f03460ff076f70ab595bb45a0205ccea1971443575b6920c52e755dec2b3fbfe \
    --hash=sha256:f0ec3b750b59375eab5b0fb2b9254810c00a3375be6d789899f1055a1d556237 \
    --hash=sha256:f291bcf42ae98eb5107edb162c3c998b4a89648fd8e99ed4cbd12705292788cd \
    --hash=sha256:f61efe1d2fe0de16158a5fe1d1cf3c14bdb6aecd54d8938fd26512c525c1f624 \
    --hash=sha256:f68edfc67aabac33708941f26f22a7b8e9f81429bc0cf249fcf7d66b23af8d19 \
    --hash=sha256:fa95848c929b6a75f6848d3c9793e59db365ee436776e57db835cdbfa79ba977 \
    --hash=sha256:fd9f8797427910198f95bced71ddfed61130d7e349213bfb8466c9c99e2c46a8 \
    --hash=sha256:fdb4ca07ab75ffadab4a8b135ad59cdbb3156b99310f3d565370da74a15d6bd3
    # via jinja2
mpmath==1.3.0 \
    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \
    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c
    # via sympy
networkx==3.7 \
    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \
    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a
    # via torch
numpy==2.5.3 \
    --hash=sha256:012e66aca395d795496446e52aeeb5866312a5d4d3f27da270e5a0b43f70dc5c \
    --hash=sha256:09d5a423c71ad5feb5625844ad58050e35df43871004b52ac9c0ad44a56775be \
    --hash=sha256:09ffa5d903faeaa5c4dd05009cf81c8bab9f2cb37c548b8d39b65b4cfa7c97f7 \
    --hash=sha256:0a59a421a32580a009e8a1751345bf829631b990dc1794b80514ab722b435def \
    --hash=sha256:116f96cadd935c6122e9228d676fe7ede19e741f5c8bb1c3cddbe0c51ccebea2 \
    --hash=sha256:1302b90c0e52281681b2975adfe8a860cb7b12216a27b4b0b4207c44bf7bccf0 \
    --hash=sha256:15aa985ac73a8db02db7663381aa109510449d3819d37206caed27b33a65a8a6 \
    --hash=sha256:1aad64d99730d013cfc6debafed22783b4fc5a7f4b8bc744d2d8cf7dcc880551 \
    --hash=sha256:1c80eabb4035ecf4ca9cd49cde8a9fdd69a729e63e6474887d1523ade7aa277f \
    --hash=sha256:1f3ed25271581281f2fccb1adcedfcde4c07362eec69189b50baf6f90e3ae159 \
    --hash=sha256:1fb6f8fb9ff0b3a69f52c66ce397b0246583e9f28616231b0e32ca49259a5fa6 \
    --hash=sha256:214045a5bf00113a146ab9ee9730c44501af6723cdf1f6830932f7b5ef2e7af0 \
    --hash=sha256:26e15e4aecd8617dfbaecb37d223e365d7b39411fba20454be2670a96aa74cb5 \
    --hash=sha256:2c25dfa72943e4336ddb6b0ee4277b47a0c85bede0807530ec68103bf58e2c10 \
    --hash=sha256:2d8240cb4c16fd831074aa2b2cf9fc54664d826341d61c372245b96a74a49a9a \
    --hash=sha256:350ba9783ce969cf9f7ce6e6a9a58e1a6e2a19ca025b7ee448c4db727706212a \
    --hash=sha256:4c8a6d2ebce6305fd82fbefca827775437147052a976ee7c94b36a0c1b52ac6c \
    --hash=sha256:4f8929ee6c96bfbd7b4ed2032e0c03af86fe1826740ab61ddabf9072d06e57ff \
    --hash=sha256:536f963710a4e63934d80ac0dc4f478804a83e9a84b6828018f25d09953ada33 \
    --hash=sha256:54a115e5a73b8fc44f0cebef486365a1894b5c9760685d4558b72b7c3eb846e0 \
    --hash=sha256:595d020938c84e320bcf40ad71089e108eac0d377cd018e14a8c094f39e98d85 \
    --hash=sha256:66a78fe4556c60aceda5916f9eacd638b18e9e681016ec302dcb4682d6d4d034 \
    --hash=sha256:6b05c171afb3aa07adbd20abc00aea86fe375beb0fdb9ef780ec5b7f63bab1c0 \
    --hash=sha256:6cef4bb1706dfec49243c05d921eefb4e190d41e2528b30d8035ea1f36b4c24a \
    --hash=sha256:6f24021b9f22bc6301c37b196974a92c1c18dccedb6fef3dd252e95f2d6adbe4 \
    --hash=sha256:71b39d9f935b6ec0f8753e3e2afb51e3efba6f2e05b68b32a40754d24bcd4a3c \
    --hash=sha256:71cad2b2a7451ab79d8f5e71b453485b6775963d5cf794179144a7463fe6e8ec \
    --hash=sha256:76c2c1e6bfa5c84adc6434dfbf013aa92096a7985221762c8f11fedfd20fff58 \
    --hash=sha256:8617bbfae4486cf99c9f899966699428d19da931d06ca94ad3da986c76e15997 \
    --hash=sha256:86bff898a431c0fb71f7610b75726e75a54d47b37edc9d537f48de63bb3c0b90 \
    --hash=sha256:8e4dd766076855b5ff7ea52fa5f07ce26286726e0f8bff446b7739d02e6ea204 \
    --hash=sha256:92f30e89b8ee0ecf363033576c422b2f58fed6a80bed0aa48dff6d14c654663e \
    --hash=sha256:93e1f5447e2b1e479d7bd74701e84746b86450cff1fc368b132d195e2b8f8211 \
    --hash=sha256:9a37475425b431b4d060f23b4f52cd2f3aef6bc7c654bd760adf0040eec9d435 \
    --hash=sha256:9deb49575e5b0b94ed72c8a64ec4d033381adc27e9060ae842971f697ba96104 \
    --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321 \
    --hash=sha256:a6391fafaba97500887132cd582abc6e19452b1ac775a47caa7b24490e152058 \
    --hash=sha256:a72f874bc9e10e4b8f80426fb49716d5141f64442a0c8418065093ec8017fbb0 \
    --hash=sha256:ac7bb1c52d445bd4f8f7f97fefe6abc3a084dc4d63df50d79b17fa2b78e89297 \
    --hash=sha256:adc1ada2662f8a5f960b8a10d9986897e7499ef07e06d4cfe7197f8cce923c07 \
    --hash=sha256:b00eefbcf0f292945c4b4dec2ae845389ef5bcdcd596e6e4328051db5b5ba694 \
    --hash=sha256:b0521d0f4aebb6e06189451025fa17a913287b13c03d5fe05c017333b654ea5b \
    --hash=sha256:b5d93cf48f687479941d12b69c873ad2cc76bbd487f0091c2200636497f34034 \
    --hash=sha256:b7e18c623bb5c95acb3b3328861272816ba199fb531921c5d6d0b675f1fde9e3 \
    --hash=sha256:bd4cb9ad3c7889b9b3fe0a9a9fb5d2ed26f9879bff2608d9f01aed147a20d231 \
    --hash=sha256:be5a8381859b6da607c84f4f7d6847725f1cf1853ef8a2c9e115b7d58bef47dc \
    --hash=sha256:befa1ae5bd6030b3f512b43ff3fa5290bbed6b84411a44244b14adf835f5b89d \
    --hash=sha256:bf63afbe037eb5d2fe87fbcc7778e61da53ebaf21d938a4515aa73b62532a5d4 \
    --hash=sha256:c00abe94c1a69d75d827dcf1c025b25c8a45d230b3bcd77a9020883a1b047653 \
    --hash=sha256:c2381f82999704f818e2c987a865050e285ec3621262c66d40f5a96c8f899f8e \
    --hash=sha256:c76d5dde9f445058f83d0c02af00557a4db91de9a9a57c0df87d1535001d654b \
    --hash=sha256:cb189f09db39283b26bfd061ec16189e14f71c6755207f72a0f7540867afe5b9 \
    --hash=sha256:ccb32e0525d29e8b0572eb84c9a57af0e7a4e615726927506f55063c62414034 \
    --hash=sha256:ccbc4665079665c3cf3bab4db9f6b095370cd6437d66be549b6c2a1fd19e1958 \
    --hash=sha256:d1c89973648c85069c5046ad460f7b8a00218b29a2e42359ac8cc63e9ab94832 \
    --hash=sha256:df2d5874ff183595a4ba404edd04f6bd9b5505c1d7708573f6a6c17489a67563 \
    --hash=sha256:e01c918ac3d48e18a927cf7b14a26a3e29ff2bdf2eacb976da0aecd6a43ed034 \
    --hash=sha256:e6ab667ba76450084eb64013762c438ea76d9d29cc676dcd6c2e9892ba37f841 \
    --hash=sha256:e931e4f499e0dc7ef29d269a8e5b35dd722e5d14be07df6240166ea7c6532fae \
    --hash=sha256:f54660b0eb6b0b9f36e7fe1cdfdff472028dd0d14acd9b9b65098efbad059469 \
    --hash=sha256:f59a878c33d6b88122d80d239bb3b845d58708750b0cb06a09aebb9b18ec696c \
    --hash=sha256:f7fabeb6cea87d65f3b926de33d03fb016cfdc29314c90974383b5582ae72891 \
    --hash=sha256:f9579f383d1bf9df80081e72760e84960a7fd4f88cf0c9e535a8597c9bb646f5 \
    --hash=sha256:f9a2353b37a1a9e78fd82b27ad7e2a32a2d036604d18f02b05e3136c62ca3b09 \
    --hash=sha256:fc36dc566135b5eceec4cf89758fcb719266a019ef07dae1754ae7c9f617ef3e \
    --hash=sha256:ffdc76bfcae6b255dff75202c5e7feaf95b40246bc0a17944facc1fecf9f79ab
    # via
    #   pix2struct-textcaps-pipeline (pyproject.toml)
    #   torchvision
    #   transformers
nvidia-cublas==13.1.1.3 \
    --hash=sha256:37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436 \
    --hash=sha256:b6cdce694e47ff6aadf0a69df1cab6628d696f5ff56e8d16af50309d855fa20f \
    --hash=sha256:b7a210458267ac818974c53038fbec2e969d5c99f305ab15c72522fa9f001dd5
    # via
    #   cuda-toolkit
    #   nvidia-cudnn-cu13
    #   nvidia-cusolver
nvidia-cuda-cupti==13.0.85 \
    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \
    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \
    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151
    # via cuda-toolkit
nvidia-cuda-nvrtc==13.0.88 \
    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \
    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \
    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b
    # via
    #   cuda-toolkit
    #   nvidia-cublas
nvidia-cuda-runtime==13.0.96 \
    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \
    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \
    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492
    # via cuda-toolkit
nvidia-cudnn-cu13==9.24.0.43 \
    --hash=sha256:67a7273b5cf062f9446fd76cf464351a1c0f66501e6cd78f6675c0d604d8ac87 \
    --hash=sha256:71f181cd810e90f9b6023b01186fe82d13d65f0ec098581ee201d39fad769e4b \
    --hash=sha256:a6812a554a1ff0413e9c52b84c26c050380649ab9615f9c16bded368ce9f421f
    # via torch
nvidia-cufft==12.0.0.61 \
    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \
    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \
    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3
    # via cuda-toolkit
nvidia-cufile==1.15.1.6 \
    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \
    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1
    # via cuda-toolkit
nvidia-curand==10.4.0.35 \
    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \
    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \
    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f
    # via cuda-toolkit
nvidia-cusolver==12.0.4.66 \
    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \
    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \
    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65
    # via cuda-toolkit
nvidia-cusparse==12.6.3.3 \
    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \
    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \
    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79
    # via
    #   cuda-toolkit
    #   nvidia-cusolver
nvidia-cusparselt-cu13==0.8.1 \
    --hash=sha256:4dca476c50bf4780d46cd0bfbd82e2bc10a08e4fef7950917ce8d7578d22a23f \
    --hash=sha256:786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0 \
    --hash=sha256:dccbd362f91a7b9024d1f55ee9f548ac065027ff15d8c8b0db889ab3a8f31215
    # via torch
nvidia-nccl-cu13==2.30.7 \
    --hash=sha256:ca786ffa5a647c75d4d1f5cc72a6c4f537947e2ba8823d7c8aaf768e7a7b9f77 \
    --hash=sha256:cefa7fdb9710efd0f39c5f1be1d61ff6fc9a996c451265bd7fbdcf9455ed4b50
    # via torch
nvidia-nvjitlink==13.4.92 \
    --hash=sha256:25f74fad0d654271c921ac4dca614bd6258bc21791242fc7b2289dad7ae9c099 \
    --hash=sha256:9e4a7ff4f0cafa8c624917055b863dc11f5c2912ead23c462889e166f3b0e57d \
    --hash=sha256:b286f3a4f227a9363efdec263c7b91788cef1478d2b8a5fa8bab7f3e82ff82fd \
    --hash=sha256:e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323
    # via
    #   cuda-toolkit
    #   nvidia-cufft
    #   nvidia-cusolver
    #   nvidia-cusparse
nvidia-nvshmem-cu13==3.4.5 \
    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \
    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9
    # via torch
nvidia-nvtx==13.0.85 \
    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \
    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \
    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519
    # via cuda-toolkit
packaging==26.3 \
    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \
    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c
    # via
    #   huggingface-hub
    #   transformers
pillow==11.3.0 \
    --hash=sha256:023f6d2d11784a465f09fd09a34b150ea4672e85fb3d05931d89f373ab14abb2 \
    --hash=sha256:02a723e6bf909e7cea0dac1b0e0310be9d7650cd66222a5f1c571455c0a45214 \
    --hash=sha256:040a5b691b0713e1f6cbe222e0f4f74cd233421e105850ae3b3c0ceda520f42e \
    --hash=sha256:05f6ecbeff5005399bb48d198f098a9b4b6bdf27b8487c7f38ca16eeb070cd59 \
    --hash=sha256:068d9c39a2d1b358eb9f245ce7ab1b5c3246c7c8c7d9ba58cfa5b43146c06e50 \
    --hash=sha256:0743841cabd3dba6a83f38a92672cccbd69af56e3e91777b0ee7f4dba4385632 \
    --hash=sha256:092c80c76635f5ecb10f3f83d76716165c96f5229addbd1ec2bdbbda7d496e06 \
    --hash=sha256:0b275ff9b04df7b640c59ec5a3cb113eefd3795a8df80bac69646ef699c6981a \
    --hash=sha256:0bce5c4fd0921f99d2e858dc4d4d64193407e1b99478bc5cacecba2311abde51 \
    --hash=sha256:1019b04af07fc0163e2810167918cb5add8d74674b6267616021ab558dc98ced \
    --hash=sha256:106064daa23a745510dabce1d84f29137a37224831d88eb4ce94bb187b1d7e5f \
    --hash=sha256:118ca10c0d60b06d006be10a501fd6bbdfef559251ed31b794668ed569c87e12 \
    --hash=sha256:13f87d581e71d9189ab21fe0efb5a23e9f28552d5be6979e84001d3b8505abe8 \
    --hash=sha256:155658efb5e044669c08896c0c44231c5e9abcaadbc5cd3648df2f7c0b96b9a6 \
    --hash=sha256:1904e1264881f682f02b7f8167935cce37bc97db457f8e7849dc3a6a52b99580 \
    --hash=sha256:19d2ff547c75b8e3ff46f4d9ef969a06c30ab2d4263a9e287733aa8b2429ce8f \
    --hash=sha256:1a992e86b0dd7aeb1f053cd506508c0999d710a8f07b4c791c63843fc6a807ac \
    --hash=sha256:1b9c17fd4ace828b3003dfd1e30bff24863e0eb59b535e8f80194d9cc7ecf860 \
    --hash=sha256:1c627742b539bba4309df89171356fcb3cc5a9178355b2727d1b74a6cf155fbd \
    --hash=sha256:1cd110edf822773368b396281a2293aeb91c90a2db00d78ea43e7e861631b722 \
    --hash=sha256:1f85acb69adf2aaee8b7da124efebbdb959a104db34d3a2cb0f3793dbae422a8 \
    --hash=sha256:23cff760a9049c502721bdb743a7cb3e03365fafcdfc2ef9784610714166e5a4 \
    --hash=sha256:2465a69cf967b8b49ee1b96d76718cd98c4e925414ead59fdf75cf0fd07df673 \
    --hash=sha256:2a3117c06b8fb646639dce83694f2f9eac405472713fcb1ae887469c0d4f6788 \
    --hash=sha256:2aceea54f957dd4448264f9bf40875da0415c83eb85f55069d89c0ed436e3542 \
    --hash=sha256:2d6fcc902a24ac74495df63faad1884282239265c6839a0a6416d33faedfae7e \
    --hash=sha256:30807c931ff7c095620fe04448e2c2fc673fcbb1ffe2a7da3fb39613489b1ddd \
    --hash=sha256:30b7c02f3899d10f13d7a48163c8969e4e653f8b43416d23d13d1bbfdc93b9f8 \
    --hash=sha256:3828ee7586cd0b2091b6209e5ad53e20d0649bbe87164a459d0676e035e8f523 \
    --hash=sha256:3cee80663f29e3843b68199b9d6f4f54bd1d4a6b59bdd91bceefc51238bcb967 \
    --hash=sha256:3e184b2f26ff146363dd07bde8b711833d7b0202e27d13540bfe2e35a323a809 \
    --hash=sha256:41342b64afeba938edb034d122b2dda5db2139b9a4af999729ba8818e0056477 \
    --hash=sha256:41742638139424703b4d01665b807c6468e23e699e8e90cffefe291c5832b027 \
    --hash=sha256:4445fa62e15936a028672fd48c4c11a66d641d2c05726c7ec1f8ba6a572036ae \
    --hash=sha256:45dfc51ac5975b938e9809451c51734124e73b04d0f0ac621649821a63852e7b \
    --hash=sha256:465b9e8844e3c3519a983d58b80be3f668e2a7a5db97f2784e7079fbc9f9822c \
    --hash=sha256:48d254f8a4c776de343051023eb61ffe818299eeac478da55227d96e241de53f \
    --hash=sha256:4c834a3921375c48ee6b9624061076bc0a32a60b5532b322cc0ea64e639dd50e \
    --hash=sha256:4c96f993ab8c98460cd0c001447bff6194403e8b1d7e149ade5f00594918128b \
    --hash=sha256:504b6f59505f08ae014f724b6207ff6222662aab5cc9542577fb084ed0676ac7 \
    --hash=sha256:527b37216b6ac3a12d7838dc3bd75208ec57c1c6d11ef01902266a5a0c14fc27 \
    --hash=sha256:5418b53c0d59b3824d05e029669efa023bbef0f3e92e75ec8428f3799487f361 \
    --hash=sha256:59a03cdf019efbfeeed910bf79c7c93255c3d54bc45898ac2a4140071b02b4ae \
    --hash=sha256:5e05688ccef30ea69b9317a9ead994b93975104a677a36a8ed8106be9260aa6d \
    --hash=sha256:6359a3bc43f57d5b375d1ad54a0074318a0844d11b76abccf478c37c986d3cfc \
    --hash=sha256:643f189248837533073c405ec2f0bb250ba54598cf80e8c1e043381a60632f58 \
    --hash=sha256:65dc69160114cdd0ca0f35cb434633c75e8e7fad4cf855177a05bf38678f73ad \
    --hash=sha256:67172f2944ebba3d4a7b54f2e95c786a3a50c21b88456329314caaa28cda70f6 \
    --hash=sha256:676b2815362456b5b3216b4fd5bd89d362100dc6f4945154ff172e206a22c024 \
    --hash=sha256:6a418691000f2a418c9135a7cf0d797c1bb7d9a485e61fe8e7722845b95ef978 \
    --hash=sha256:6abdbfd3aea42be05702a8dd98832329c167ee84400a1d1f61ab11437f1717eb \
    --hash=sha256:6be31e3fc9a621e071bc17bb7de63b85cbe0bfae91bb0363c893cbe67247780d \
    --hash=sha256:7107195ddc914f656c7fc8e4a5e1c25f32e9236ea3ea860f257b0436011fddd0 \
    --hash=sha256:71f511f6b3b91dd543282477be45a033e4845a40278fa8dcdbfdb07109bf18f9 \
    --hash=sha256:7859a4cc7c9295f5838015d8cc0a9c215b77e43d07a25e460f35cf516df8626f \
    --hash=sha256:7966e38dcd0fa11ca390aed7c6f20454443581d758242023cf36fcb319b1a874 \
    --hash=sha256:79ea0d14d3ebad43ec77ad5272e6ff9bba5b679ef73375ea760261207fa8e0aa \
    --hash=sha256:7aee118e30a4cf54fdd873bd3a29de51e29105ab11f9aad8c32123f58c8f8081 \
    --hash=sha256:7b161756381f0918e05e7cb8a371fff367e807770f8fe92ecb20d905d0e1c149 \
    --hash=sha256:7c8ec7a017ad1bd562f93dbd8505763e688d388cde6e4a010ae1486916e713e6 \
    --hash=sha256:7d1aa4de119a0ecac0a34a9c8bde33f34022e2e8f99104e47a3ca392fd60e37d \
    --hash=sha256:7db51d222548ccfd274e4572fdbf3e810a5e66b00608862f947b163e613b67dd \
    --hash=sha256:819931d25e57b513242859ce1876c58c59dc31587847bf74cfe06b2e0cb22d2f \
    --hash=sha256:83e1b0161c9d148125083a35c1c5a89db5b7054834fd4387499e06552035236c \
    --hash=sha256:857844335c95bea93fb39e0fa2726b4d9d758850b34075a7e3ff4f4fa3aa3b31 \
    --hash=sha256:8797edc41f3e8536ae4b10897ee2f637235c94f27404cac7297f7b607dd0716e \
    --hash=sha256:8924748b688aa210d79883357d102cd64690e56b923a186f35a82cbc10f997db \
    --hash=sha256:89bd777bc6624fe4115e9fac3352c79ed60f3bb18651420635f26e643e3dd1f6 \
    --hash=sha256:8dc70ca24c110503e16918a658b869019126ecfe03109b754c402daff12b3d9f \
    --hash=sha256:91da1d88226663594e3f6b4b8c3c8d85bd504117d043740a8e0ec449087cc494 \
    --hash=sha256:921bd305b10e82b4d1f5e802b6850677f965d8394203d182f078873851dada69 \
    --hash=sha256:932c754c2d51ad2b2271fd01c3d121daaa35e27efae2a616f77bf164bc0b3e94 \
    --hash=sha256:93efb0b4de7e340d99057415c749175e24c8864302369e05914682ba642e5d77 \
    --hash=sha256:97afb3a00b65cc0804d1c7abddbf090a81eaac02768af58cbdcaaa0a931e0b6d \
    --hash=sha256:97f07ed9f56a3b9b5f49d3661dc9607484e85c67e27f3e8be2c7d28ca032fec7 \
    --hash=sha256:98a9afa7b9007c67ed84c57c9e0ad86a6000da96eaa638e4f8abe5b65ff83f0a \
    --hash=sha256:9ab6ae226de48019caa8074894544af5b53a117ccb9d3b3dcb2871464c829438 \
    --hash=sha256:9c412fddd1b77a75aa904615ebaa6001f169b26fd467b4be93aded278266b288 \
    --hash=sha256:a1bc6ba083b145187f648b667e05a2534ecc4b9f2784c2cbe3089e44868f2b9b \
    --hash=sha256:a418486160228f64dd9e9efcd132679b7a02a5f22c982c78b6fc7dab3fefb635 \
    --hash=sha256:a4d336baed65d50d37b88ca5b60c0fa9d81e3a87d4a7930d3880d1624d5b31f3 \
    --hash=sha256:a6444696fce635783440b7f7a9fc24b3ad10a9ea3f0ab66c5905be1c19ccf17d \
    --hash=sha256:a7bc6e6fd0395bc052f16b1a8670859964dbd7003bd0af2ff08342eb6e442cfe \
    --hash=sha256:b4b8f3efc8d530a1544e5962bd6b403d5f7fe8b9e08227c6b255f98ad82b4ba0 \
    --hash=sha256:b5f56c3f344f2ccaf0dd875d3e180f631dc60a51b314295a3e681fe8cf851fbe \
    --hash=sha256:be5463ac478b623b9dd3937afd7fb7ab3d79dd290a28e2b6df292dc75063eb8a \
    --hash=sha256:c37d8ba9411d6003bba9e518db0db0c58a680ab9fe5179f040b0463644bc9805 \
    --hash=sha256:c84d689db21a1c397d001aa08241044aa2069e7587b398c8cc63020390b1c1b8 \
    --hash=sha256:c96d333dcf42d01f47b37e0979b6bd73ec91eae18614864622d9b87bbd5bbf36 \
    --hash=sha256:cadc9e0ea0a2431124cde7e1697106471fc4c1da01530e679b2391c37d3fbb3a \
    --hash=sha256:cc3e831b563b3114baac7ec2ee86819eb03caa1a2cef0b481a5675b59c4fe23b \
    --hash=sha256:cd8ff254faf15591e724dc7c4ddb6bf4793efcbe13802a4ae3e863cd300b493e \
    --hash=sha256:d000f46e2917c705e9fb93a3606ee4a819d1e3aa7a9b442f6444f07e77cf5e25 \
    --hash=sha256:d9da3df5f9ea2a89b81bb6087177fb1f4d1c7146d583a3fe5c672c0d94e55e12 \
    --hash=sha256:e5c5858ad8ec655450a7c7df532e9842cf8df7cc349df7225c60d5d348c8aada \
    --hash=sha256:e67d793d180c9df62f1f40aee3accca4829d3794c95098887edc18af4b8b780c \
    --hash=sha256:ea944117a7974ae78059fcc1800e5d3295172bb97035c0c1d9345fca1419da71 \
    --hash=sha256:eb76541cba2f958032d79d143b98a3a6b3ea87f0959bbe256c0b5e416599fd5d \
    --hash=sha256:ec1ee50470b0d050984394423d96325b744d55c701a439d2bd66089bff963d3c \
    --hash=sha256:ee92f2fd10f4adc4b43d07ec5e779932b4eb3dbfbc34790ada5a6669bc095aa6 \
    --hash=sha256:f0f5d8f4a08090c6d6d578351a2b91acf519a54986c055af27e7a93feae6d3f1 \
    --hash=sha256:f1f182ebd2303acf8c380a54f615ec883322593320a9b00438eb842c1f37ae50 \
    --hash=sha256:f8a5827f84d973d8636e9dc5764af4f0cf2318d26744b3d902931701b0d46653 \
    --hash=sha256:f944255db153ebb2b19c51fe85dd99ef0ce494123f21b9db4877ffdfc5590c7c \
    --hash=sha256:fdae223722da47b024b867c1ea0be64e0df702c5e0a60e27daad39bf960dd1e4 \
    --hash=sha256:fe27fb049cdcca11f11a7bfda64043c37b30e6b91f10cb5bab275806c32f6ab3
    # via
    #   pix2struct-textcaps-pipeline (pyproject.toml)
    #   torchvision
pyarrow==25.0.1 \
    --hash=sha256:0b1edbb2f385a6a65e9711b62ba86ac54a7816a3f8d17bb3e8a5929d65fb2485 \
    --hash=sha256:0b726ad7e7b669be982b0c71c07fe4b037d654354130da79a7902a669e93a66b \
    --hash=sha256:0befcf816e45a1af33ac775a9970b749e4868a230c7372f0ae5e932bee27039f \
    --hash=sha256:0fe7c8b6c03969b49c8c66182e4a18e3819ab92d07cfab5d8370c531b9369ef0 \
    --hash=sha256:119297a6dc197e45d9c6d4415f7814a67ffa36c180d26f68c154c58067ae782d \
    --hash=sha256:169d3429d5be7c752125890620f75a60776d38b0035eddae939651640822332e \
    --hash=sha256:25f8720bf6387d5dc2ebd2622112de630760419e4b66134405dd24110d15f37e \
    --hash=sha256:31e49a7888fcdf3a835da33ae777f6bb9a866334e5a789282fc26dcf426f7f15 \
    --hash=sha256:35935cd5de130aa5cf4dea052a63e6bf2e17006c35c3a468194242b9b2bf5956 \
    --hash=sha256:38a9a4b4b9613380e200641891495a56c3d5a98a092db4a870af9975e220471d \
    --hash=sha256:3f89685964f46e4216103c75483aac0c0692a5f72212d7ca835adba5ede56ce3 \
    --hash=sha256:4288f27577352d608ca08553b0865e4a9b3aa14820c5d95b53337218d609835b \
    --hash=sha256:4340f0ba6c1d2e13f21658de1d7c662ca2545018568d0030a1e9afca159d87e3 \
    --hash=sha256:44a9120ce5bd81936b8ab9a88076e3fd47c2c6838e0e43630fed83626aca81d9 \
    --hash=sha256:4facd65742a024a4a366328a1d2292062d72d6e023c1b7dda8d4c37544933a25 \
    --hash=sha256:51093dd9e10325fbdb3c10a2ae7c4806e5c822d94e74ae4938b26524a3323fee \
    --hash=sha256:514ddb60285631af068875550c90eddc181db3e8e63a032b1559be189e82f056 \
    --hash=sha256:5389cdf79447ed1515c9e31620e6e1e2302249564d603f2ad727d4f6d313e4c3 \
    --hash=sha256:59a2de54c0cbd954da861eee4d1d330f8e909c45b53455baef696380f2c55033 \
    --hash=sha256:60e89d8f13861a1f7f8d950fa54aebb8023b30734d0ac51ffa80beabe2df4bba \
    --hash=sha256:6109c94d8b9f3b17a041daca16cacb2f651ad8f1ef70a4232c2c0f37a23da2a8 \
    --hash=sha256:62cd0d785b8aa6675ee355f9fc02252a340f4441257c42674937826fd7594325 \
    --hash=sha256:6943e2fe7954d29d84de45d29d34c8dc36ce96570e67d89aa9976e650a4a9138 \
    --hash=sha256:6a1fdfc6659b6b19022f2e50627fb5cf7156a66c46bf4299379955cbe742382a \
    --hash=sha256:880523be3d29efcf83d3998835d206118ccf35e3871dbd2fb60408cf6b007a80 \
    --hash=sha256:8858d7bfc22e3f51529aeaa4077225029724623e4595dc9eff8c793935c34140 \
    --hash=sha256:9150a83248bfed9813ea3c3af74c3856c1984d444aa28e58bf7733b9750ddf6a \
    --hash=sha256:9171748cdf796972d85a4b60157c279913e242992e350c90c7450182a9838b2a \
    --hash=sha256:a4d6d5e9a3d1879a97c08ded0c797579b7965eafd0f0c26c30b45ccc06db939b \
    --hash=sha256:a4dd8bf99a8fac133efc0ed6a92f5fddbe2adba0d0f6dd720e39ba9855cea85c \
    --hash=sha256:aa0559502e1cd6254d6814614085dd9c5a3dd0419362978a936a3f68a9e5c3df \
    --hash=sha256:b7a296aac7a71fa0886c08e155ddb6c636a50013f801f6178daafa0f9e726188 \
    --hash=sha256:bddd0c4f7630c2a3ddf6347c1bdaa79d97bcf6bd445f9e60c816b7d77c85a5ae \
    --hash=sha256:bf0b672390cdcb640d7288f96b826d71ff4e9abb254a86c89890baf51a29cee6 \
    --hash=sha256:c7c534ec03c358a76ea3e505e74c1b6aef290af90c444dfd092dbfe23e755b85 \
    --hash=sha256:cab40b1edfef0262e0e5251aa2c58d75630f24d06dd7794480243acc001a1d7d \
    --hash=sha256:cc4aa407fde9fc660be3939e49ea31f50f3e9fec17c0ec63159f7711edd3efc9 \
    --hash=sha256:d51592cb7561e87877c506113e7adbf1342ab579e6c21f0ef44b8ba41cb74c80 \
    --hash=sha256:dda9470024204d7bbf2042b47c6e8a0e47a3eeb8e34405882dfaea6577e0c153 \
    --hash=sha256:df961f2e7ae9cf496459259d798652c70625f6c080650d6952f8c04053c58ee9 \
    --hash=sha256:eb6203482ff3746a5632303a7279ae0b5a304c46985b49ed1378cb350ea6728d \
    --hash=sha256:f3831aaa25c67a99f99dc8b05873cb9d64560390372e2aa197ce9dd4a3f06a44 \
    --hash=sha256:f729cfdbd36fd99d543b67a914d2de044c84ebe45be8b34902b299b608c15c8f
    # via pix2struct-textcaps-pipeline (pyproject.toml)
pyyaml==6.0.3 \
    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \
    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \
    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \
    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \
    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \
    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \
    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \
    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \
    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \
    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \
    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \
    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \
    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \
    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \
    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \
    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \
    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \
    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \
    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \
    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \
    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \
    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \
    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \
    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \
    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \
    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \
    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \
    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \
    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \
    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \
    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \
    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \
    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \
    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \
    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \
    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \
    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \
    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \
    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \
    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \
    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \
    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \
    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \
    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \
    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \
    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \
    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \
    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \
    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \
    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \
    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \
    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \
    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \
    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \
    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \
    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \
    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \
    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \
    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \
    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \
    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \
    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \
    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \
    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \
    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \
    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \
    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \
    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \
    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \
    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \
    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \
    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \
    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0
    # via
    #   huggingface-hub
    #   transformers
regex==2026.9.29 \
    --hash=sha256:01000ddf0e3ffef97f2413ceb514f6313040106b6d18a03ee00a4fe35c1eb1db \
    --hash=sha256:0166844493626c5015c6088ee15c9ca2fd060ca15b7641d1657da6a58432ae33 \
    --hash=sha256:044265d77d94f5e3cb2fd72c76723807c429cb8c533e9d4672d0334a6f14f588 \
    --hash=sha256:0476e5bcbe6e1ba3d1c4cc7bbb1c3ba78e3b979b5c8a88d0a6a8cdd4992b8c84 \
    --hash=sha256:066d0e3dbfdd739bce2bf8c2a41dd16f73e3d8adc2eb06dd803a36a307f56075 \
    --hash=sha256:0b65c72739f981377c9c22e0c5c3cd7f42da7bd8a3c9209330fac772c7d893ed \
    --hash=sha256:0c992c19cd45058a4b92f68f139c93db168b48fb1f322c9a7cd620806afb6b51 \
    --hash=sha256:0cc63b5e47c12a48d90c7e9d7de6a035dd14f62868aaedbb4e0ff8ba2b8bfe7b \
    --hash=sha256:0dd8af32e9f7b56b7f95cc1fd79b23054c3bdc172392ae560acc24d57b7ffe71 \
    --hash=sha256:0def9fb6abac55492d6d51cddb7225d07d6f279e774e0adc08569a54a5fc8d46 \
    --hash=sha256:0fd2c901cc307a745ad4bc87f20060d7a0825a3371d1e93488af22e7a387f78f \
    --hash=sha256:1043aedf5917caa861bcb25a9c11460049656bdf0017a90a309fa8f255467725 \
    --hash=sha256:121a76a0985db80ceae9e171c337f8c927868e37d01b54e3ce87bc87f9c6a208 \
    --hash=sha256:143533cc4b6fbc5b95aca0a5b8d541088d374831593def000ec89322c220221d \
    --hash=sha256:14e953ff3607c92d7675bf79c4d4509ef6782aa8c08509f179f9b3d6d0679e86 \
    --hash=sha256:18ae8eed4526e35bdb754d61562b90bf5c00a67fdcf3cc1380dd59597486631b \
    --hash=sha256:19959129885356df0e97556856f77eb2888380dac18bed075a7c05c5128c618d \
    --hash=sha256:1ba8c6a416569ce0d37e83e28a254a61dc99a419084dfb6476cea02d997f74fa \
    --hash=sha256:1c2a0026062abcc321a53db4a185ceba0b59a66b5d37b0808917a88b55a5257f \
    --hash=sha256:1d9fe8091b2e89d470df68a9331111ed008ae8aae6bf1e8e1fba4086a495c84e \
    --hash=sha256:2089fe39c406784d90101c726755ffa1497bb74638fd434300d2b88006186de8 \
    --hash=sha256:23ae6fdad9e63e54038f5ef78aba2933faca61e24d432786589e737bc5522ebb \
    --hash=sha256:26ec4ccce55aa533fbd603d08911b01101a8fcfec987845ac3ae2c7087b2bde3 \
    --hash=sha256:2f7f7aa47b229f2b39a2ae2596d2ad5625d77b5eb9856fac2dab3eb506cdd0a0 \
    --hash=sha256:31b003f9a070335e2a8233ee9b14a3ca8e6d792012ae011f741bf0aaf11744c5 \
    --hash=sha256:32ab11df9677ca80bcbb5fe4eb1da9109a5019239a054836efc6fa1c64e683cf \
    --hash=sha256:33026515aebc0e70d1c89978e53e8d695d35d9e472f8d5b34465ba3c74028650 \
    --hash=sha256:34b6925af9853bf461950e6508910f179fd6e9b1a7ec8548e069606b7e51a26b \
    --hash=sha256:352cf115a810b357caa35193ab656ecf5ef41056855e82f292c99e8514f8d954 \
    --hash=sha256:39ab5894d971f9ac68baa6eca5c50387db579cfcacf36ae8df3feceb1815e6d0 \
    --hash=sha256:3a21a9509d0ee88e7a70e1ad228cd2f0e0fd1e187458db132e8a8d18c97daf9d \
    --hash=sha256:3c5c2ef13797466aa64170cbb66ad98a32351dd4127694cea7199f80f213750d \
    --hash=sha256:3e778bfccd63075167709136afbc251c1f683758d5bf49c803c60ac3f894ce6b \
    --hash=sha256:3f1e6cb402a89457582cd696f982559217d13484a193202c394015297968c86d \
    --hash=sha256:42e82e578c904445d4c8a35b8f28052cf567593215fa5db06266fbc6f77aaa2e \
    --hash=sha256:4408b2b27a95ca8cc48b7411945753773353b5c93b307754781086c99d3a576f \
    --hash=sha256:446654b29bfaa30500d80947eda42cef1449dc8a87f4e3cf061cc8485d3a1f0b \
    --hash=sha256:45010bcfe66df41522d56c9b6114e87ecc597a08970ff6a2ced24415c141ae5f \
    --hash=sha256:49ee178ca31c94621294bf9b8b676a92a2e6bba8af0529591753719e57edb621 \
    --hash=sha256:4d7d93613b01b0199961330e49cfc52d479b3d5776c56c691db31130c0a07d91 \
    --hash=sha256:4fb41211d2333eb930a51e0546a65999761cf1f572a4da56ef9b8a62966c06f2 \
    --hash=sha256:4fe97894d1b306c919b4e50def1e6f6c522f4d03a7283811f4d108f1ce5d3ac2 \
    --hash=sha256:554bffadcbcb6d5f4e5fb10a61cc52084b9a63d1dab5f10bcd2c4343972e8e2c \
    --hash=sha256:59b49507f47479e299a9e1bc41b5cb83a7afda0540625f1dbae886615978acbf \
    --hash=sha256:5eeb8edc6110d9194a4d0d54610f64c37a31c605b5dbb7e407fc6ec7fa34a4a1 \
    --hash=sha256:612b709381c0355b70d89cdb51b7f670591ed5cbbc0e3b5337488019dc667b65 \
    --hash=sha256:61956f074ecd123f55adca68ee3eab46e6a07ad3f8e64e6db95dfacb444f55c4 \
    --hash=sha256:6398d5145689503412cc1748895242598d8846b8967b851133b20dc2ed1e21e8 \
    --hash=sha256:65b408d8fcb273e3499e7ef2ce796810da1becd208c7fb4373692a242d79d461 \
    --hash=sha256:686ac5350fceae63830bb98805fcb8039325bf4c06d9f6f048ff65229d5bffa5 \
    --hash=sha256:6a1a824fbed817e0a891103886b68f063b1e83cc51bc97192a90a60195a9291f \
    --hash=sha256:6abb75ab16bc3281714a5b99548a2225db70dba1f995f6d7f7419b76eb5a8fbe \
    --hash=sha256:6f7121a8914ed13fcfe2099f895341bfb789f004d4c5a0bdece8fa667da10849 \
    --hash=sha256:7020ed44df30b3aa492c00ee3b52d0548c1f30c2c6c5bb13ae897680900d3413 \
    --hash=sha256:720537c7ea6f80dc61913184edb0ce2497a306b39ef19f28505b322553d52bdb \
    --hash=sha256:724184b4aafed865e4f13ca313fdcb43024300c028ec67319cfa16847d84685e \
    --hash=sha256:7c03031610e3e6ed1768a2b7a8fc84637c1257b50c5eacaf094c6e17a84fc563 \
    --hash=sha256:80a5ea3b4fd9d6a5b9a44f7976a9acaaab35aa3c1f6b29e5bd857dfabaded223 \
    --hash=sha256:80c7cadd3fd2bfde5df8aa0787e315812cad0c313a753095d02f4c2b6c01677b \
    --hash=sha256:80ea96f5c1a30bf09007d48466521d9c294bebe197c708c3359096e3e3691632 \
    --hash=sha256:864e9b87ac33c3fb9fb4ad48166d4fdb579c351d5c77deb0d34bccb36a775cd9 \
    --hash=sha256:87fb80cbe3557e27e7b28b995c2b2eedf689b8886f941ab93e0e288f0976518a \
    --hash=sha256:8873c4a11c50b9989168881aeb3f08859f469d809941866aa1feefd8be5431f6 \
    --hash=sha256:888d60953908dcf761aa320c3e390ab8556efbdb551ace63921de90f6ae0848d \
    --hash=sha256:8b5fcc4771732191b2b7d1dd68d8f0353f47f8d90b6150f6dce58bf1112442cb \
    --hash=sha256:8f39588af4731c8923c26810eb3b33f76f17633985e40f59c3cd45a33805a895 \
    --hash=sha256:9173db3be74a35cb6731701094b98120f7ee4876a287882a59cdea1fa7da342f \
    --hash=sha256:92f05c9c42bde5785dc48770bc2194d9f7442544156f951e19cd31b096cec562 \
    --hash=sha256:951733b1bbdb71e377cec567b409f1a7881b47cfcad84121aa74cb575fa425ea \
    --hash=sha256:957bb708e8057ab1649ba566456429d691ec9b90d1c9ad1af1ba7ffbbeaf05f2 \
    --hash=sha256:9916fda742cd4eede63b286f58c06718324265d727ce0856eb1aac86d0d150d6 \
    --hash=sha256:9e1d3a4cb7993b708f0ada8d0c84590efd853f169e7147d2202c9da503180242 \
    --hash=sha256:9e4482589065c8ecd761cff522dcd85f2d39e62f551e37e025d1c7d54772def3 \
    --hash=sha256:a5300757f8a68f5b6cc33f57338d72a0e3589c5cc9ad5f8504ea06f028be582a \
    --hash=sha256:a540abfab208e1b7ef2df231c40ef3b6cbb30a0aad6204e9b6a81c10a6794628 \
    --hash=sha256:a5758353650079898dc1b2b0e95aa51fa23a30d020e06f62c430dd08ee56cdd8 \
    --hash=sha256:a64b85a4760337cfefdb27d42da6ed8b58e8cde3f2d57b6ef43e76ef6ea9ef47 \
    --hash=sha256:a655d34b2a6943af32401f3d94f72e9d731f6ad16285815550bf2b4ee69d420a \
    --hash=sha256:a714befaacbd10092ffe4cea0d3c5f008fb9efe9bc322c715bcdfdee414b9a3d \
    --hash=sha256:a760da040b47767b4b873adfb7c3b691e9ba2fc60f113f9d0b88f1a62f323e85 \
    --hash=sha256:addd736a0547d553283adaf4e05d7104e7f2c7b0b092e9b4d28756825f14531f \
    --hash=sha256:ae4613d7d9dda60fcba95f846cc6f808017f1843f392cf9daad14a6534493d71 \
    --hash=sha256:b11b589e00095ec69cf79841a76360f9b079e95b0368a25b5ebb951ab0c157ff \
    --hash=sha256:b3e445b66c80b4eb4234e855ce94d9adc183eedbd632816228d89930b91b2c5b \
    --hash=sha256:b7b893976e7fe42053da64f2aa27239c24252fd2ec6df471e1be197c0addc3b1 \
    --hash=sha256:b84f186a7f0536fe4ff9a9fa12d06d007b9b71d4b5352ddcc41f59ad6522a312 \
    --hash=sha256:b89efc38431793d28b7cd91227e2f952ad7c48df19132b17f43a5fec3c14143b \
    --hash=sha256:b97a38fb4c732b6832db6bf108963adbcd82ef1268ba2025dce390f45af75efa \
    --hash=sha256:b9d74e4eee9ddb64c2e92d5d61472c59c21684c059eb7b68767be9628e977859 \
    --hash=sha256:bb90e7177944b6684738c1fc36aabd2dd00d1de3be7dbe09f91e196f1bc0dc81 \
    --hash=sha256:bec37990e3d6121f29ecfb594bd8f1bf009e9f7926daba2e50e3b27d3892a783 \
    --hash=sha256:bf3c49863c23a1ad6da9c30351aed6cff8d5ddbeb63c5c8420ae54e98c7d0138 \
    --hash=sha256:bf48516e35cf848390ea68850aba53e7c333720d2945b4d2c25b69fc5171723f \
    --hash=sha256:bfc71e6d970419c1309b3640305298643e2a734cad3f7cfb6d2ddee4175ab53d \
    --hash=sha256:c0094897d7d01f184b2d7fe8c56c66d64efe01b31f4b7d34205b391387df1111 \
    --hash=sha256:c03c6eb6ece86dfdcbb34799efaa339b093132e1aceed491ba5e08fe06cdf699 \
    --hash=sha256:c1a9a6651197fbed6f0212591418b9def774fc3f8324f78d1bf0e6a63e5f8aa1 \
    --hash=sha256:c3589f40749acce747510bf5d589d54e376cb0930ea58b35effac97e5312b0c1 \
    --hash=sha256:c4e38dd8f39c43a91d2410ad2b85610701b0979342c3df1d69eaf8e838c757d8 \
    --hash=sha256:c6c8fabf1dafc1f1ddcbb67896d3f93efb092e8c4b6322d7389b944e76a484e5 \
    --hash=sha256:c90fcf7804ea0a54b896ce0f2b9565350220b8d4890fd0db461a476a4c687963 \
    --hash=sha256:c9b602fae1e00b7c035d661ce85575365719192a7b46784bd71cf64c68053aa0 \
    --hash=sha256:ccb64d887a9db1cd76dbc0f92051a1a478a2a67e7f56c62d915cb881d7734704 \
    --hash=sha256:d06fcdecc10fc7954d7c8f27a03c96055fe525274dc84a7b0dbdc3d6b9e03dab \
    --hash=sha256:d0c3082bf79bcd6a614d55916590ad4b8f93200e10b97f463ea5d9d07c9b5f23 \
    --hash=sha256:d49c18f1ea294cf4adde2e5ac256e98c82ea9d708462ce4bf799dffa7cfe8a2c \
    --hash=sha256:d60030baaa7bfbb02d650c126cdcddcb6e33dbff14d819434c8fa2fdcaeeeba5 \
    --hash=sha256:d7cab119d0df0b9413f106b4d7fc34f2872d3574ed3806fb48959c830b1537da \
    --hash=sha256:d9b77b25b4f395f92de6099ab08e8ae2bc7e51dfe157f22900902243a5cc90c7 \
    --hash=sha256:dabee8f4935e731fb46b2a3091bdda0d3d94b3bbfb907d2b4f12eefce4009619 \
    --hash=sha256:db5e82ba15c142425b8406690032df89e39cca4a2e8afbbb9a3d84edc2373ac3 \
    --hash=sha256:dc79d36d0618752265f0d575915bdc5c5130ecb9c9f6b3bcefeae32e4bdfafcf \
    --hash=sha256:ddfa987262763c3c22a8367d2a49c244b018a74c3a8e3ab1a864119ad45c5633 \
    --hash=sha256:e1172147d28d8fbcf8cb8d26c41506169f5ad8fe9ec969cb116835a19d4d8eca \
    --hash=sha256:e11edba5bc344a32b029a7af9d4b3173982dd79eeafa0b9dbd787364414b0509 \
    --hash=sha256:e2c89e9b762c57f59d5e99ee8b20202adb892e35f8d3485741340999ca55058e \
    --hash=sha256:e31f72490b7c12f7790e1e25c3afffd20503ee1bfb43461d7838b871ff244b19 \
    --hash=sha256:e8c65ef3862a8ad6e86492b6ed9327805dd66904c012bd3649dc67d822ed6c34 \
    --hash=sha256:ebb8912f565b8cdbbf27debfe00df04202c20e2f651b9e32767930c5eace3621 \
    --hash=sha256:ed511a0708e2297e1d6431e7fb217e3402791e491e02da800658ace4973df1bb \
    --hash=sha256:edf06545875f3efa31560d94121e95c7fd70d98b1dfedc0157097d79b13b52ea \
    --hash=sha256:f0fe9834e5aeccaf19a0d8feb296d66a24be1a7c9922002f842a682cd5abb787 \
    --hash=sha256:f1a0d5117230dd46b399a30a38afa44f79c99f3168988fdc4f425c3f928b39df \
    --hash=sha256:f37964e4a5e993d2fd45147741e9dff7f34a2d8c00ab94c4ea0514a4677f959e \
    --hash=sha256:f57dc6b8fef170f105d2cf5cdce254f47b137d7755086cf7050f47e16582abba \
    --hash=sha256:f93bc1c3486ef3747e07c9d7c1d0a147b8fbaab975f80e348aed6f71309dfaca \
    --hash=sha256:fb00027a09a8f9f08028b40dce4c933cf73e4833240ed356583fdc9cfa721566 \
    --hash=sha256:fb99cc9d45f48895d9d67f6a0b8a57f08d39c174d9f25ad97a313e0470267b1c \
    --hash=sha256:fdd88ed5e20b1bcdd234421e454962c971aa44b653bdb7f1ea9ef683e90fb649 \
    --hash=sha256:fe3fa1dd453ed5c7f5ea23a26218329790ed7197a99b90e94330e313959a7f52
    # via transformers
requests==2.34.2 \
    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \
    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed
    # via
    #   huggingface-hub
    #   transformers
safetensors==0.8.0 \
    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \
    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \
    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \
    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \
    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \
    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \
    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \
    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \
    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \
    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \
    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \
    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \
    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \
    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \
    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \
    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \
    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774
    # via
    #   pix2struct-textcaps-pipeline (pyproject.toml)
    #   transformers
setuptools==84.0.0 \
    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \
    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73
    # via torch
sympy==1.14.0 \
    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \
    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5
    # via torch
tokenizers==0.22.2 \
    --hash=sha256:143b999bdc46d10febb15cbffb4207ddd1f410e2c755857b5a0797961bbdc113 \
    --hash=sha256:1a62ba2c5faa2dd175aaeed7b15abf18d20266189fb3406c5d0550dd34dd5f37 \
    --hash=sha256:1c774b1276f71e1ef716e5486f21e76333464f47bece56bbd554485982a9e03e \
    --hash=sha256:1e418a55456beedca4621dbab65a318981467a2b188e982a23e117f115ce5001 \
    --hash=sha256:1e50f8554d504f617d9e9d6e4c2c2884a12b388a97c5c77f0bc6cf4cd032feee \
    --hash=sha256:2249487018adec45d6e3554c71d46eb39fa8ea67156c640f7513eb26f318cec7 \
    --hash=sha256:25b85325d0815e86e0bac263506dd114578953b7b53d7de09a6485e4a160a7dd \
    --hash=sha256:29c30b83d8dcd061078b05ae0cb94d3c710555fbb44861139f9f83dcca3dc3e4 \
    --hash=sha256:319f659ee992222f04e58f84cbf407cfa66a65fe3a8de44e8ad2bc53e7d99012 \
    --hash=sha256:369cc9fc8cc10cb24143873a0d95438bb8ee257bb80c71989e3ee290e8d72c67 \
    --hash=sha256:37ae80a28c1d3265bb1f22464c856bd23c02a05bb211e56d0c5301a435be6c1a \
    --hash=sha256:38337540fbbddff8e999d59970f3c6f35a82de10053206a7562f1ea02d046fa5 \
    --hash=sha256:473b83b915e547aa366d1eee11806deaf419e17be16310ac0a14077f1e28f917 \
    --hash=sha256:544dd704ae7238755d790de45ba8da072e9af3eea688f698b137915ae959281c \
    --hash=sha256:64d94e84f6660764e64e7e0b22baa72f6cd942279fdbb21d46abd70d179f0195 \
    --hash=sha256:753d47ebd4542742ef9261d9da92cd545b2cacbb48349a1225466745bb866ec4 \
    --hash=sha256:791135ee325f2336f498590eb2f11dc5c295232f288e75c99a36c5dbce63088a \
    --hash=sha256:9ce725d22864a1e965217204946f830c37876eee3b2ba6fc6255e8e903d5fcbc \
    --hash=sha256:a6bf3f88c554a2b653af81f3204491c818ae2ac6fbc09e76ef4773351292bc92 \
    --hash=sha256:bfb88f22a209ff7b40a576d5324bf8286b519d7358663db21d6246fb17eea2d5 \
    --hash=sha256:c9ea31edff2968b44a88f97d784c2f16dc0729b8b143ed004699ebca91f05c48 \
    --hash=sha256:df6c4265b289083bf710dff49bc51ef252f9d5be33a45ee2bed151114a56207b \
    --hash=sha256:e10bf9113d209be7cd046d40fbabbaf3278ff6d18eb4da4c500443185dc1896c \
    --hash=sha256:f01a9c019878532f98927d2bacb79bbb404b43d3437455522a00a30718cdedb5
    # via transformers
torch==2.14.0 \
    --hash=sha256:0e7cf18cb0d8bd666b6120932e29c7aef3502b61a08b44da4839580c539a7cdb \
    --hash=sha256:2fb30099be1fec9d163a6dd05ace176305ce8cf914b4d7a5f4f6d7a326aa4b60 \
    --hash=sha256:44b044b9f6f633d982839422a57433d6a1da520037fd88e0c8a47efde589b3b8 \
    --hash=sha256:4dbea8a10d80b10dff2be1ed467d2617b704fe6bf76bbfad4194774c8f6d886b \
    --hash=sha256:553aec938d37d77b783bcf801e638cad068870e7643c47eadac21eb180f551ed \
    --hash=sha256:6981872df75eb5409c439050d36b67cb57d88b4e11080516e2aa7410f6730705 \
    --hash=sha256:731784e3914843c6bcc7aba3987ff7610ac57dbbc816a5d6b9b62e04c240a641 \
    --hash=sha256:731b9ebdea402b8b1996d4c2ae613b16660e559b19e47bdc45d970568bc91c53 \
    --hash=sha256:84bf384779a10c02fc3c6bdbab71a9cb66b0dd93c652d1ed5d6dfc0cb37e5962 \
    --hash=sha256:860423e970f2ce02c4476e8e2d1350131b1c5c5a5e4912180e78b50b53241efa \
    --hash=sha256:8d9e232b6376c62f3090237889fb1cb6887c0ece9f73e6c1052e80fc1481f999 \
    --hash=sha256:96383c62423c3f2767023c4a94774b6435e2c006691daf396e9ddb55d412fbc6 \
    --hash=sha256:9d4b1022a5d9b71282ec67ad0d9e7235870096b8a246dc1c32d6ea1fc83dc998 \
    --hash=sha256:ada340e62591d06a2bcc2d68170f20f45f0b0665d372dc510a8ed7eb3b1d609a \
    --hash=sha256:ae530ddd3f3b94248b77f1fd3313c4f3405bd0d17283d505b81574816767133a \
    --hash=sha256:b2cd92bce63d40bf6fc2e5d840fc2f0063bd180a242daa761cb6088cc2f46e27 \
    --hash=sha256:b985c7defeb8d28691b7513ebaecc64946c5f68ec770e9f4ddba39688864f46a \
    --hash=sha256:c1f844f1c750e87df4b68bc3afbc0e2b0c7ef19d7b8f666e48bdcf6a0c4f0056 \
    --hash=sha256:cad84f41bbdf3dcf333ce394aeeaf25237c4d94fd6b659ba5eb813c829978823 \
    --hash=sha256:cd8cd8f714d511ccdca907282d1da3d9be8322d4e1520b9c3bce39a5c1318a4b \
    --hash=sha256:d2526f71e6638133b97b3cd2881ece3df521460a4727030e8ae2a72a7d3ae31d \
    --hash=sha256:d30207cb89e713dfd6afb5d592b30d9ef252acf80407c5543157ea7b07c4d9a0 \
    --hash=sha256:d70c2b41bde81efde59fcefb147b9247fd062497e778677661b7de1bee5c8c99 \
    --hash=sha256:fecffb58f51fd643d213acd68da21cc3fc19bea05a3bc64b4ee55128f47a4963
    # via
    #   pix2struct-textcaps-pipeline (pyproject.toml)
    #   torchvision
torchaudio==2.11.0 \
    --hash=sha256:00e9f71ab9c656f0abdb40c515bd65d4658ab0ad380dee27a2efd7d51dabd3d6 \
    --hash=sha256:13cff988697ccbad539987599f9dc672f40c417bed67570b365e4e5002bbd096 \
    --hash=sha256:1424638adb8bb40087bc7b6eb103e8e4fe398210f09076f33b7b5e61501b5d66 \
    --hash=sha256:1a07ec72fd6f26a588c39b5f029e0130d16bb40bc4221635580bf8fb18fcbc80 \
    --hash=sha256:1be3767064364ae82705bdf2b15c1e8b41fea82c4cd04d47428a8684b634b6ed \
    --hash=sha256:1c1101c1243ef0e4063ec63298977e2d3655c15cf88d9eb0a1bd4fe2db9f47ea \
    --hash=sha256:478110f981e5d40a8d82221732c57a56c85a1d5895fb8fe646e86ee15eded3bd \
    --hash=sha256:492dd64645e9d0bb843e94f1d9a4d1e31426262ffc594fafecc1697df9df5eb9 \
    --hash=sha256:5847fe2022b17c6580aeb39c8797a443411cc09edfd9183cd50ac1a3b8ccf97c \
    --hash=sha256:6503c0bdb29daf2e6281bb70ea2dfe2c3553b782b619eb5d73bdadd8a3f7cecf \
    --hash=sha256:67f6edac29ed004652c11db5c19d9debb5d835695930574f564efc8bdd061bba \
    --hash=sha256:6ebb59c694909eccb5d61b7cc199d297692012c43286e36d92983aa7bad7586d \
    --hash=sha256:73dab4841f94d888bc7c2aed7b5547c643edc974306919fe1adfb65d57cccf4b \
    --hash=sha256:79fb3cb99169fd41bd9719647261402a164da0d105a4d81f42a3260844ec5e79 \
    --hash=sha256:7e2da1df4f6fe885c46db350a0dc90a0dff4b54541dff8846faa904d255e2bfe \
    --hash=sha256:88fb5e29f670a33d9bac6aabb1d2734460cf6e461bde5cdc352826035851b16d \
    --hash=sha256:986f4df5ed17b003dc52489468601720090e65f964f8bebccf90eb45bba75744 \
    --hash=sha256:9fe3083c62e035646483a14e180d33561bdc2eed436c9ab1259c137fb7120b4a \
    --hash=sha256:a1cf1acc883bee9cb906a933572fed6a8a933f86ef34e9ea7d803f72317e8c1b \
    --hash=sha256:b034d7672f1c415434f48ef17807f2cce47f29e8795338c751d4e596c9fbe8b5 \
    --hash=sha256:bb59ba4452bbbe95d75ad3ef18df9824955625f36698ce9a5998a4a9f3c1ba1d \
    --hash=sha256:bc653defca1c16154398517a1adc98d0fb7f1dd08e58ced217558d213c2c6e29 \
    --hash=sha256:bda09ea630ae7207384fb0f28c35e4f8c0d82dd6eba020b6b335ad0caa9fed49 \
    --hash=sha256:be7ad472acb16d16e98c005f0219b0db06a47dfe8f7b4d177062e1638f871e3b \
    --hash=sha256:cc09cd1f6015b8549e7fe255fb1be5346b57e7fee06541d3f3dbb012d8c4715f \
    --hash=sha256:da2725e250866da42a12934c9a6552f65a18b7187fd7a6221387f0e605fb3b96 \
    --hash=sha256:e3f9696a9ef1d49acc452159b052370c636406d072e9d8f10895fda87b591ea9 \
    --hash=sha256:ed404c4399ad7f172c86a47c1b25293d322d1d58e26b10b0456a86cf67d37d84
    # via pix2struct-textcaps-pipeline (pyproject.toml)
torchvision==0.29.0 \
    --hash=sha256:01a029fa1b2eacac27e1e4f9e0dba24d112bcbd523261b294e4b4a1c7f8330bc \
    --hash=sha256:0e3b99c62f7f095153887330c2c60c9ebc2da84dfbe08eea564a41ec361629cf \
    --hash=sha256:0e631a2f8b24732672d35224d2574ff89a78d56bef5364ef6094a625fa5f1771 \
    --hash=sha256:2428a13706a354ee3901fd63e3f1ef8c2a59f6e20d24cce13210b4e61be08f7a \
    --hash=sha256:28a3964a9e6db34354d4ca440d2c2038deeef24e7219e8d6f49d9d174f44830c \
    --hash=sha256:710df2c4f03ab6da161520c10d9467f4c0201ca314fd1432499a48c9d7a5818f \
    --hash=sha256:784c8b8de9e81e02dea093373bc35040791378df2daec0017e657985183bf862 \
    --hash=sha256:80789f9b50f5277302e7020c8128344e0269b12f6e3b5855034e56bb7081b874 \
    --hash=sha256:83db0299170502b038640444214c99d7b54bad72db447f0554deb8d2c4f02f94 \
    --hash=sha256:85fa54bec1f7d9227b5e4e201ed4bac92247ee10fb19148d92f9836e3e91d5c3 \
    --hash=sha256:942a3b3fb4e981e1abb7846679ee3882dfc733533a30b68df0dadf6c06f238a9 \
    --hash=sha256:994687b818cac0e6d34cb407a41458e2c3262cb7db927465841c9522ba7ebb92 \
    --hash=sha256:9b99a7385da8d706f2dc14cacffb8cbf653d48fc37013a38c41a36659acc3731 \
    --hash=sha256:a292ca7044236a2702ec3d4664de135e01fc9fb3ae9422db3fecc819546e831c \
    --hash=sha256:add88479361b2bf08790338fce4ba11a11fdc1ae508e80a3a013daf1c0d2893d \
    --hash=sha256:b7a736eaa6b2e22476c95ceb62986d195f3e600cdd7d196d7329aa2dfc951994 \
    --hash=sha256:bb74d08d65785b51c61d4d4ec167bfa50a5ea644b57555a33ba2f4435b0d05dd \
    --hash=sha256:bbe15455b59a6c9d822584fd3faaaac50ee972101d7bb7e2887da456e148a3ea \
    --hash=sha256:bd12e152640d1024fca15deaf6f39c53702e12eb398db4b2b038a5a97ab85f7f \
    --hash=sha256:ce3b58631f0b7c8828e8675301bff5cb0623aaed704e3ac91079e89607753205 \
    --hash=sha256:dc5717feef9a0b430b052895db71f1d695939575b74344da981120e9f4da7620 \
    --hash=sha256:e8fdf234d76dca6fc47bc5f4af86ecf1ef57fab6285a7b21681bdcbb494e4c1b \
    --hash=sha256:ebf54f6744556ebd8d7b5f9b6515e2060e5d3156723970dc257bf093f63b2170 \
    --hash=sha256:f9e9627d5036cac8e6de76aeb2f1acc67841ff5ff4b63c4cf78ff954efeb073e
    # via pix2struct-textcaps-pipeline (pyproject.toml)
tqdm==4.70.1 \
    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \
    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4
    # via
    #   huggingface-hub
    #   transformers
transformers==4.57.6 \
    --hash=sha256:4c9e9de11333ddfe5114bc872c9f370509198acf0b87a832a0ab9458e2bd0550 \
    --hash=sha256:55e44126ece9dc0a291521b7e5492b572e6ef2766338a610b9ab5afbb70689d3
    # via pix2struct-textcaps-pipeline (pyproject.toml)
triton==3.8.0 \
    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \
    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \
    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \
    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \
    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \
    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \
    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \
    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \
    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \
    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \
    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \
    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3
    # via torch
typing-extensions==4.16.0 \
    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \
    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5
    # via
    #   huggingface-hub
    #   torch
urllib3==2.8.0 \
    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \
    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63
    # via requests
'''

SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
# One environment per lock digest: a re-run of this cell, or a second Run all in the same runtime, reuses a complete
# environment built from this exact lock instead of rebuilding it; a different lock gets a different folder.
ISOLATED_ENV = Path(os.environ.get('DIMER_ISOLATED_ENV', 'dimer_isolated_env_' + LOCK_SHA256[:12])).resolve()
ISOLATED_PYTHON = ISOLATED_ENV / 'bin' / 'python'
ISOLATED_TOOLS = ISOLATED_ENV.with_name(ISOLATED_ENV.name + '_tools')
ISOLATED_READY = ISOLATED_ENV / '.dimer-lock-sha256'


def _isolated_environment_ready():
    return ISOLATED_PYTHON.is_file() and ISOLATED_READY.is_file() and ISOLATED_READY.read_text(encoding='utf-8').strip() == LOCK_SHA256


if SKIP_INSTALL:
    print('DIMER_NOTEBOOK_CI_PREINSTALLED=1: the pins are already installed; the notebook runs in this kernel.')
elif _isolated_environment_ready():
    print({'isolated_environment': str(ISOLATED_ENV), 'reused': True, 'lock_sha256': LOCK_SHA256[:16] + '...', 'locked_packages': LOCKED_PACKAGES, 'kernel_python': platform.python_version()})
else:
    if platform.system() != 'Linux' or platform.machine() != 'x86_64':
        raise RuntimeError('This notebook needs a Linux x86_64 runtime (Google Colab, Kaggle or Linux Jupyter): its locked environment is built for manylinux x86_64.')
    setup_started = time.perf_counter()
    if hashlib.sha256(LOCK_TEXT.encode('utf-8')).hexdigest() != LOCK_SHA256:
        raise RuntimeError('The carried lock does not match its digest: regenerate the notebook from the repository instead of editing this cell.')
    ISOLATED_TOOLS.mkdir(parents=True, exist_ok=True)
    lock_path = ISOLATED_TOOLS / LOCK_NAME
    lock_path.write_text(LOCK_TEXT, encoding='utf-8', newline='\n')
    for attempt in range(3):
        try:
            with urllib.request.urlopen(UV_URL, timeout=90) as response:
                wheel = response.read(UV_BYTES + 1)
            break
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == 2:
                raise
            time.sleep(2**attempt)
    if len(wheel) != UV_BYTES or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
        raise RuntimeError('The pinned uv wheel failed its size/SHA-256 check: refusing to run it. Run this cell again; if it repeats, the download is being altered.')
    with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
        member = next(name for name in archive.namelist() if name.endswith('.data/scripts/uv'))
        uv = ISOLATED_TOOLS / 'uv'
        uv.write_bytes(archive.read(member))
    uv.chmod(0o700)
    # uv gets no kernel Python path; the managed interpreter is downloaded once and reused on a re-run.
    uv_env = dict(os.environ, MPLBACKEND='Agg')
    for name in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP'):
        uv_env.pop(name, None)
    if not ISOLATED_PYTHON.is_file():
        subprocess.run([str(uv), 'venv', '--quiet', '--managed-python', '--python', MANAGED_PYTHON, str(ISOLATED_ENV)], env=uv_env, check=True)
    isolated_version = subprocess.run([str(ISOLATED_PYTHON), '-c', 'import platform; print(platform.python_version())'], env=uv_env, check=True, capture_output=True, text=True).stdout.strip()
    if isolated_version != MANAGED_PYTHON:
        raise RuntimeError(f'{ISOLATED_ENV} holds Python {isolated_version}, not {MANAGED_PYTHON}: delete that folder (or start a fresh runtime) and run this cell again.')
    ISOLATED_READY.unlink(missing_ok=True)
    subprocess.run([str(uv), 'pip', 'install', '--quiet', '--python', str(ISOLATED_PYTHON), '--require-hashes', '--only-binary', ':all:', '--index-url', 'https://pypi.org/simple', '-r', str(lock_path)], env=uv_env, check=True)
    ISOLATED_READY.write_text(LOCK_SHA256 + '\n', encoding='utf-8')
    print({'isolated_environment': str(ISOLATED_ENV), 'reused': False, 'isolated_python': isolated_version, 'kernel_python': platform.python_version(), 'locked_packages': LOCKED_PACKAGES, 'setup_seconds': round(time.perf_counter() - setup_started)})

# The worker runs in the isolated environment. It executes each routed cell in one persistent namespace and sends
# back printed text, displayed objects and matplotlib figures, so every cell behaves as it would in the kernel.
_WORKER_SOURCE = r"""
import ast, base64, builtins, io, linecache, os, signal, sys, traceback, types
from multiprocessing.connection import Connection

_send = Connection(int(sys.argv[1]), readable=False)
_recv = Connection(int(sys.argv[2]), writable=False)


class _Stream(io.TextIOBase):
    def __init__(self, name):
        self._name = name

    @property
    def encoding(self):
        return "utf-8"

    def writable(self):
        return True

    def isatty(self):
        return False

    def write(self, text):
        if text:
            _send.send(("stream", self._name, str(text)))
        return len(text)


sys.stdout, sys.stderr = _Stream("stdout"), _Stream("stderr")


def _figure_bundle(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight")
    return {"image/png": base64.b64encode(buffer.getvalue()).decode("ascii"), "text/plain": repr(fig)}


def _flush_figures():
    plt = sys.modules.get("matplotlib.pyplot")
    if plt is None:
        return
    for number in plt.get_fignums():
        _send.send(("display", _figure_bundle(plt.figure(number))))
    plt.close("all")


def _mimebundle(obj):
    if hasattr(obj, "savefig"):
        return _figure_bundle(obj)
    data = {"text/plain": repr(obj)}
    for method, mime in (("_repr_html_", "text/html"), ("_repr_markdown_", "text/markdown"), ("_repr_png_", "image/png")):
        render = getattr(obj, method, None)
        if callable(render):
            try:
                value = render()
            except Exception:
                value = None
            if isinstance(value, bytes):
                value = base64.b64encode(value).decode("ascii")
            if value is not None:
                data[mime] = value
    return data


def display(*objects, **kwargs):
    for obj in objects:
        _send.send(("display", _mimebundle(obj)))


try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot

    matplotlib.pyplot.show = lambda *args, **kwargs: _flush_figures()
except ImportError:
    pass

if os.environ.get("DIMER_KERNEL_IS_COLAB") == "1":
    # google.colab only exists in the kernel; forward the BYOD upload dialog to it.
    def _upload():
        _send.send(("upload",))
        reply = _recv.recv()
        if reply[1] is None:
            raise RuntimeError("The notebook kernel could not open the upload dialog.")
        return reply[1]

    import importlib.machinery

    def _stub(name, package):
        # A spec on every stub: importlib.util.find_spec("google.colab") (accelerate does this) raises on a None __spec__.
        module = types.ModuleType(name)
        module.__spec__ = importlib.machinery.ModuleSpec(name, None, is_package=package)
        if package:
            module.__path__ = []
        return module

    try:
        import google
    except ImportError:
        google = _stub("google", True)
        sys.modules["google"] = google
    _colab = _stub("google.colab", True)
    _files = _stub("google.colab.files", False)
    _files.upload = _upload
    _colab.files = _files
    google.colab = _colab
    sys.modules["google.colab"] = _colab
    sys.modules["google.colab.files"] = _files

_main = types.ModuleType("__main__")
_main.__dict__.update(__builtins__=builtins, display=display)
sys.modules["__main__"] = _main
_count = 0
while True:
    # An interrupt only lands inside a running cell; between cells it is ignored.
    signal.signal(signal.SIGINT, signal.SIG_IGN)
    try:
        message = _recv.recv()
    except EOFError:
        break
    if message[0] != "run":
        continue
    _count += 1
    filename = f"<isolated cell {_count}>"
    source = message[1]
    linecache.cache[filename] = (len(source), None, source.splitlines(True), filename)
    try:
        signal.signal(signal.SIGINT, signal.default_int_handler)
        tree = ast.parse(source, filename)
        tail = ast.Expression(tree.body.pop().value) if tree.body and isinstance(tree.body[-1], ast.Expr) else None
        exec(compile(tree, filename, "exec"), _main.__dict__)
        if tail is not None:
            value = eval(compile(tail, filename, "eval"), _main.__dict__)
            if value is not None:
                display(value)
        _flush_figures()
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        _send.send(("done",))
    except BaseException as exc:
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        if getattr(exc, "dimer_stop", False):  # a deliberate stop (e.g. a refused BYOD input): message only
            _send.send(("stop", str(exc)))
            continue
        frames = exc.__traceback__.tb_next if exc.__traceback__ is not None else None
        _send.send(("error", "".join(traceback.format_exception(type(exc), exc, frames)), f"{type(exc).__name__}: {exc}"))
"""


class IsolatedCellError(RuntimeError):
    """A routed cell raised inside the isolated environment; its traceback is printed above."""


class IsolatedCellStop(RuntimeError):
    """A routed cell stopped on purpose; IPython shows only the message (no traceback), and Run all stops here."""

    def _render_traceback_(self):
        return [str(self)]


class IsolatedRuntime:
    """One persistent worker process in the isolated environment, fed one cell at a time."""

    def __init__(self, python, display=None):
        self.python = str(python)
        to_kernel_r, to_kernel_w = os.pipe()
        to_worker_r, to_worker_w = os.pipe()
        env = dict(os.environ, MPLBACKEND="Agg", PYTHONUNBUFFERED="1", DIMER_NOTEBOOK_CI_PREINSTALLED="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
        for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
            env.pop(name, None)
        env["DIMER_KERNEL_IS_COLAB"] = "1" if "google.colab" in sys.modules else "0"
        self.proc = subprocess.Popen(
            [str(python), "-c", _WORKER_SOURCE, str(to_kernel_w), str(to_worker_r)],
            pass_fds=(to_kernel_w, to_worker_r),
            env=env,
            start_new_session=True,  # interrupts reach the worker only through run(), exactly once
        )
        os.close(to_kernel_w)
        os.close(to_worker_r)
        self._recv = Connection(to_kernel_r, writable=False)
        self._send = Connection(to_worker_w, readable=False)
        if display is None:
            from IPython.display import display
        self._display = display

    def alive(self):
        return self.proc.poll() is None

    def _exited(self):
        return RuntimeError(
            f"The isolated environment's Python process exited (code {self.proc.wait()}); a crash of this kind is "
            "usually running out of memory. Restart the session and choose Run all again."
        )

    def run(self, source):
        try:
            self._send.send(("run", source))
        except OSError:
            raise self._exited() from None
        while True:
            try:
                message = self._recv.recv()
            except EOFError:
                raise self._exited() from None
            except KeyboardInterrupt:
                self.proc.send_signal(signal.SIGINT)
                continue
            kind = message[0]
            if kind == "stream":
                (sys.stdout if message[1] == "stdout" else sys.stderr).write(message[2])
            elif kind == "display":
                self._display(message[1], raw=True)
            elif kind == "upload":
                self._send.send(("upload", self._colab_upload()))
            elif kind == "error":
                sys.stderr.write(message[1])
                raise IsolatedCellError(message[2]) from None
            elif kind == "stop":
                raise IsolatedCellStop(message[1]) from None
            elif kind == "done":
                return

    @staticmethod
    def _colab_upload():
        try:
            from google.colab import files
        except ImportError:
            return None
        return files.upload()

    def close(self):
        self._send.close()
        self.proc.wait(timeout=30)


def _is_user_cell():
    # ipykernel transforms a cell before executing it; its caller knows whether this is a silent frontend request.
    frame = sys._getframe(2)
    while frame is not None:
        local = frame.f_locals
        if "silent" in local and "store_history" in local:
            return bool(local["store_history"]) and not bool(local["silent"])
        frame = frame.f_back
    return True


def _route_to_isolated_runtime(lines):
    source = "".join(lines)
    if not source.strip() or "# dimer: kernel cell" in source or not _is_user_cell():
        return lines
    return [f"_DIMER_ISOLATED_RUNTIME.run({source!r})\n"]

if SKIP_INSTALL:
    print('Routing disabled: the notebook runs in this kernel.')
else:
    from IPython import get_ipython as _kernel_shell

    _ip = _kernel_shell()
    _ip.input_transformers_cleanup[:] = [t for t in _ip.input_transformers_cleanup if getattr(t, '__name__', '') != '_route_to_isolated_runtime']
    # Idempotent: a live worker on this environment is kept, with every variable the later cells created, so
    # re-running this cell alone never strands the cells after it. A dead or different worker is replaced.
    _previous = globals().get('_DIMER_ISOLATED_RUNTIME')
    _reuse = _previous is not None and getattr(_previous, 'python', None) == str(ISOLATED_PYTHON) and _previous.proc.poll() is None
    if _reuse:
        _DIMER_ISOLATED_RUNTIME = _previous
    else:
        if _previous is not None and _previous.proc.poll() is None:
            _previous.close()
        _DIMER_ISOLATED_RUNTIME = IsolatedRuntime(ISOLATED_PYTHON)
    _ip.input_transformers_cleanup.append(_route_to_isolated_runtime)
    print({'routed_to': str(ISOLATED_PYTHON), 'worker_pid': _DIMER_ISOLATED_RUNTIME.proc.pid, 'worker_reused': _reuse})

### Record the runtime

This is the first cell that runs in the isolated environment. Nothing is installed here. It records the notebook's source revision and the versions actually imported. Look for a dictionary reporting the notebook's source revision, the isolated Python (3.12.12), `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
# @title Infrastructure: record the runtime
import importlib
import os
import platform
import sys

NOTEBOOK_SOURCE = {
    'repository': 'pix2struct-textcaps-pipeline',
    'repository_revision': '728a24c4227c65070bc131d6f4973ab767e5862b',
    'embedded_module': 'src/pix2struct_textcaps_pipeline/pipeline.py',
    'embedded_modules': ['src/pix2struct_textcaps_pipeline/pipeline.py', 'src/pix2struct_textcaps_pipeline/metrics.py', 'src/pix2struct_textcaps_pipeline/samples.py'],
    'module_sha256': '9026563972dd8cad337fd4f353403555f65d6b80895208f683f13bc18d4c3797',
    'generator': 'build_notebook.py/2.5',
    'notebook_spec': '2.5',
}
import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'executable': sys.executable, 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/pix2struct_textcaps_pipeline/` @ `728a24c4227c`)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells, and a `# @title Infrastructure` comment line heading each cell, which Colab shows as the collapsed cell's title). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/pix2struct_textcaps_pipeline/pipeline.py`

In [ ]:
# @title Infrastructure: carried module 1/3: src/pix2struct_textcaps_pipeline/pipeline.py (verbatim; see the note above)
"""Text-aware image captioning with the pinned ``google/pix2struct-textcaps-base`` checkpoint.

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the Pix2Struct architecture comes from the pinned ``transformers`` release,
the weights are SafeTensors, and no model-repository code is executed. The TextCaps checkpoint reads the
text in the image from pixels and is expected to mention it in the caption; this package measures that
as ``text_recall`` on text the caller drew, which is sanity evidence, not a captioning metric.
"""

from __future__ import annotations

import hashlib
import json
import math
import re
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "google/pix2struct-textcaps-base"
MODEL_REVISION = "61bee0d7e2378e601b68f853ceee4f7cf99f1b88"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "pix2struct-textcaps-base"
WEIGHT_FILE = "model.safetensors"
WEIGHT_SHA256 = "771693f7e310a89d77aa7ee4d1f5ea3e0c4adb0b6112cdf25a20da29cc3d0ea7"  # model.safetensors
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

PARAMETER_COUNT = 282_285_696  # 18,879,744 of them train by default (2 decoder blocks + final norm)
DECODER_LAYERS = 12  # text_config num_layers of the caption decoder
DEFAULT_TRAINABLE_DECODER_LAYERS = 2  # the last two decoder blocks + the decoder's final layer norm
MAX_EVAL_RECORDS = 2_000
# Fixed optimiser settings of `adapt` (pre-registered; read by the notebook's preregistered comparison).
ADAPT_WEIGHT_DECAY = 0.01
ADAPT_GRAD_CLIP = 1.0
ADAPT_SCHEDULER = "none"
MIN_SCORED_RECORDS = 50  # below this a scored set is labelled a small sample
ARTIFACT_FORMAT = "org.valcorza.pix2struct-textcaps-base.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"

# Generation ceilings. TextCaps captions are one sentence that quotes text in the image (the
# checkpoint's text_config max_length is 20); the default leaves room for a long quoted string and
# the ceiling bounds runaway generation.
MAX_NEW_TOKENS = 64
DEFAULT_MAX_NEW_TOKENS = 30
DECODING = "greedy"
# Optional conditional-captioning prefix (the upstream README's "A picture of"); the model continues
# it. A prefix longer than a short phrase is not what the model was trained on.
MAX_PREFIX_CHARS = 128
# Input ceilings. The processor extracts at most MAX_PATCHES 16x16 patches (preprocessor_config.json)
# after scaling the image to fill that budget (aspect ratio preserved), so pixel count only guards
# memory during resizing.
MAX_PATCHES = 2048
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
_PUNCT_RE = re.compile(r"[^\w\s]")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def normalize_caption(text: str) -> str:
    """Simplified caption normalisation (not the PTB tokenizer `coco-caption` uses): lower-case, every
    non-word
    character replaced by a space, whitespace collapsed."""
    return " ".join(_PUNCT_RE.sub(" ", text.lower()).split())


def caption_tokens(text: str) -> list[str]:
    return normalize_caption(text).split()


def unigram_f1(prediction: str, references: Sequence[str]) -> float:
    """Bag-of-words F1 between the normalised prediction and the best-matching reference.

    A plumbing check, not a captioning metric: CIDEr, BLEU-4 and SPICE need several references per
    image and corpus-level statistics. Multiset overlap counts repeated words once per occurrence.
    """
    if not references:
        raise ValueError("references must contain at least one caption")
    pred = caption_tokens(prediction)
    best = 0.0
    for reference in references:
        ref = caption_tokens(reference)
        if not pred or not ref:
            continue
        ref_counts: dict[str, int] = {}
        for token in ref:
            ref_counts[token] = ref_counts.get(token, 0) + 1
        overlap = 0
        for token in pred:
            if ref_counts.get(token, 0) > 0:
                overlap += 1
                ref_counts[token] -= 1
        if overlap:
            precision, recall = overlap / len(pred), overlap / len(ref)
            best = max(best, 2 * precision * recall / (precision + recall))
    return best


def keyword_hits(caption: str, keywords: Sequence[str]) -> dict[str, bool]:
    """Which of the caller's keywords (normalised, whole-token match) appear in the caption."""
    tokens = set(caption_tokens(caption))
    return {keyword: all(part in tokens for part in caption_tokens(keyword)) for keyword in keywords}


def text_recall(caption: str, drawn_texts: Sequence[str]) -> dict[str, Any]:
    """Fraction of the distinct normalised tokens of the drawn text strings that appear in the caption.

    The TextCaps task is to mention the text visible in the image; on text the caller drew, recall of
    those tokens is a plumbing check of the OCR-free reading path, not a captioning metric.
    """
    if not drawn_texts:
        raise ValueError("drawn_texts must contain at least one string")
    expected: list[str] = []
    for text in drawn_texts:
        for token in caption_tokens(text):
            if token not in expected:
                expected.append(token)
    if not expected:
        raise ValueError("drawn_texts contain no word tokens after normalisation")
    present = set(caption_tokens(caption))
    found = [token for token in expected if token in present]
    return {
        "recall": len(found) / len(expected),
        "found": found,
        "missing": [token for token in expected if token not in present],
        "n_expected": len(expected),
    }


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one image as PIL.Image.Image (any mode, converted to RGB) plus an optional caption prefix",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "prefix_chars": [0, MAX_PREFIX_CHARS],
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "decoding": f"{DECODING} (do_sample=False, num_beams=1), deterministic on a fixed device and dtype",
    "preprocessing": (
        "image scaled to fill at most MAX_PATCHES 16x16 patches (aspect ratio preserved), normalised per "
        "image and flattened into patch tokens with row/column positions; the text decoder starts from the "
        "start token (unconditional) or from the tokenised prefix (conditional) and generates the caption"
    ),
    "output": "one caption string (the model's decoded text, prefix included when given), no score",
}


def _check_inputs(image: Any, prefix: Any, max_new_tokens: Any) -> tuple[Image.Image, str | None, int]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``caption`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    checked_prefix: str | None = None
    if prefix is not None:
        if not isinstance(prefix, str):
            raise TypeError("prefix must be a str or None")
        checked_prefix = " ".join(prefix.split())
        if not checked_prefix:
            raise ValueError("prefix must contain at least one non-whitespace character or be None")
        if len(checked_prefix) > MAX_PREFIX_CHARS:
            raise ValueError(f"prefix has {len(checked_prefix)} chars > MAX_PREFIX_CHARS {MAX_PREFIX_CHARS}")
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int):
        raise TypeError("max_new_tokens must be an int")
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError(f"max_new_tokens must be between 1 and MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
    return rgb, checked_prefix, max_new_tokens


def validate_inputs(
    images: Sequence[Image.Image],
    *,
    prefix: str | None = None,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Every image is checked exactly as ``caption`` would check it; rejection is reported by raising,
    and a caller that wants the finding recorded catches the exception and stores ``str(exc)`` under
    ``findings``.
    """
    if isinstance(images, Image.Image) or not isinstance(images, Sequence) or not images:
        raise TypeError("images must be a non-empty sequence of PIL.Image.Image")
    if names is not None and len(names) != len(images):
        raise ValueError(f"names has {len(names)} entries for {len(images)} images")
    checked_prefix = None
    observed = []
    for index, image in enumerate(images):
        _, checked_prefix, _ = _check_inputs(image, prefix, max_new_tokens)
        observed.append(
            {"id": names[index] if names else f"image-{index}", "mode": image.mode, "size": list(image.size)}
        )
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": observed,
        "prefix": checked_prefix,
        "generation": {"max_new_tokens": int(max_new_tokens), "do_sample": False, "decoding": DECODING},
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    results: Sequence[Mapping[str, Any]],
    drawn_texts: Sequence[Sequence[str]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``drawn_texts`` (one sequence of text strings known to be in each image, in order) the report
    carries the mean ``text_recall`` over the images plus one per-image entry, verdict
    ``sample-sanity``; without them it is ``not-measurable`` and says what labelled data would make the
    task measurable. Neither is a captioning benchmark.
    """
    if not results:
        raise ValueError("results must contain at least one caption result")
    base = {
        "task": "image (+ optional prefix) -> caption text that quotes the text in the image (TextCaps)",
        "score_semantics": (
            "the caption is generated text and carries no score, probability or correctness signal; a "
            "fluent caption is not evidence that it describes the image. Greedy decoding makes the output "
            "reproducible on a fixed device and dtype, a reproducibility property, not a quality one"
        ),
        "sample_kind": sample_kind,
        "n_images": len(results),
        "truncated": [bool(result.get("truncated")) for result in results],
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if drawn_texts is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no drawn text or reference captions were supplied for the captioned images",
            "needs": (
                "several human-written reference captions per image from the deployment domain "
                "(TextCaps-style annotations, five per image) scored with CIDEr / BLEU-4 over a corpus; no "
                "such labelled set ships with this repository"
            ),
        }
    if len(drawn_texts) != len(results):
        raise ValueError(f"drawn_texts has {len(drawn_texts)} entries for {len(results)} results")
    per_image = []
    for result, texts in zip(results, drawn_texts, strict=True):
        if isinstance(texts, str) or not texts:
            raise ValueError("each drawn_texts entry must be a non-empty sequence of strings")
        prediction = str(result["caption"])
        recall = text_recall(prediction, texts)
        per_image.append(
            {
                "image": result.get("image"),
                "prediction": prediction,
                "drawn_texts": list(texts),
                "text_recall": recall["recall"],
                "found": recall["found"],
                "missing": recall["missing"],
            }
        )
    metrics = [
        {
            "id": "text_recall",
            "value": sum(entry["text_recall"] for entry in per_image) / len(per_image),
            "normalisation": "lower-cased, punctuation removed, whitespace collapsed; distinct drawn tokens",
            "relation_to_benchmarks": (
                "fraction of the drawn text's tokens quoted in the caption; not CIDEr or BLEU-4, which "
                "need several reference captions per image and corpus-level statistics"
            ),
            "estimation": f"{len(per_image)} image(s), no dispersion estimate",
        }
    ]
    return {
        **base,
        "metrics": metrics,
        "per_image": per_image,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(per_image)} image(s) whose text you drew yourself; plumbing evidence for the OCR-free "
            "reading path, not a captioning benchmark"
        ),
        "needs": (
            "several human-written reference captions per image from the deployment domain scored with "
            "CIDEr / BLEU-4 over a corpus for any quality claim; TextCaps is not bundled"
        ),
    }


@dataclass
class Pix2StructTextCapsPipeline:
    """``_runner(image, prefix, max_new_tokens)`` returns ``{"caption": str, "new_tokens": int}``;
    injectable so the offline tests run without the model."""

    _runner: Callable[..., dict[str, Any]]
    device: str = "cpu"
    dtype: str = "float32"
    source: str = "injected"
    adapter: dict[str, Any] | None = field(default=None, repr=False)
    _model: Any = field(default=None, repr=False)
    _processor: Any = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Pix2StructTextCapsPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import Pix2StructForConditionalGeneration, Pix2StructProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = Pix2StructProcessor.from_pretrained(location, **common)
        if getattr(processor.image_processor, "is_vqa", False):
            raise RuntimeError("snapshot image processor is the VQA variant (is_vqa=True); refusing")
        model = Pix2StructForConditionalGeneration.from_pretrained(location, dtype=torch.float32, **common)
        return cls._from_model(model, processor, resolved_device, source)

    @classmethod
    def _from_model(
        cls, model: Any, processor: Any, device: str, source: str
    ) -> Pix2StructTextCapsPipeline:
        """Wrap a constructed model and processor (every parameter frozen, eval mode) in a pipeline; the
        offline tests use it with a small randomly initialised Pix2Struct model."""
        import torch

        resolved_device = device
        model = model.eval().to(resolved_device)
        for param in model.parameters():
            param.requires_grad_(False)

        def runner(image: Image.Image, prefix: str | None, max_new_tokens: int) -> dict[str, Any]:
            if prefix is None:
                inputs = processor(images=image, return_tensors="pt").to(resolved_device)
                prompt_len = 0
            else:
                # Non-VQA processor: the text becomes decoder_input_ids (a caption prefix); nothing is
                # rendered into the image, so no header font is involved.
                inputs = processor(images=image, text=prefix, return_tensors="pt").to(resolved_device)
                prompt_len = int(inputs["decoder_input_ids"].shape[1])
            with torch.inference_mode():
                generated = model.generate(
                    **inputs, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1
                )
            ids = generated[0]
            decoded = processor.decode(ids, skip_special_tokens=True)
            # Unconditional: decoder_start + caption + eos. Conditional: decoder_start, then the prefix
            # ids echoed, then the new ones (observed: budget 1 yields prompt_len + 2 ids).
            new_tokens = int(ids.shape[0]) - prompt_len - 1
            return {"caption": decoded, "new_tokens": max(new_tokens, 0)}

        return cls(runner, resolved_device, "float32", source, _model=model, _processor=processor)

    def caption(
        self,
        image: Image.Image,
        *,
        prefix: str | None = None,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    ) -> dict[str, Any]:
        """Caption one image; ``caption`` is the decoded text, stripped (prefix included when given)."""
        rgb, checked_prefix, checked_tokens = _check_inputs(image, prefix, max_new_tokens)
        raw = self._runner(rgb, checked_prefix, checked_tokens)
        if not isinstance(raw, dict) or "caption" not in raw:
            raise RuntimeError("runner must return a dict with 'caption'")
        new_tokens = int(raw.get("new_tokens", 0))
        return {
            "caption": str(raw["caption"]).strip(),
            "prefix": checked_prefix,
            "image_size": list(rgb.size),
            "new_tokens": new_tokens,
            "truncated": new_tokens >= checked_tokens,
            "generation": {"max_new_tokens": checked_tokens, "do_sample": False, "decoding": DECODING},
            "device": self.device,
            "dtype": self.dtype,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- adaptation contract ---------------------------------------------------------------------------

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._processor is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return self._model, self._processor

    def evaluate(
        self, records: Sequence[Mapping[str, Any]], *, max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS
    ) -> dict[str, Any]:
        """Caption every record's image (unconditional, greedy) and score the predictions against its
        reference captions (BLEU-4, ROUGE-L, CIDEr-D, unigram F1)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import caption_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        predictions = []
        for record in checked:
            with Image.open(record["image"]) as image:
                image.load()
                predictions.append(self.caption(image, max_new_tokens=max_new_tokens)["caption"])
        metrics = caption_metrics(predictions, [[str(c) for c in r["captions"]] for r in checked])
        metrics.update(
            {
                "max_new_tokens": max_new_tokens,
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "predictions": {str(r["id"]): c for r, c in zip(checked, predictions, strict=True)},
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    def _decoder_layers(self) -> int:
        model, _ = self._require_model()
        return int(model.config.text_config.num_layers)

    def _trainable_names(self, trainable_decoder_layers: int) -> list[str]:
        """The last `trainable_decoder_layers` blocks of the caption decoder plus the decoder's final layer
        norm. The untied output projection (`decoder.lm_head`, vocabulary x hidden) and every embedding stay
        frozen, as does the whole image encoder."""
        if (
            isinstance(trainable_decoder_layers, bool)
            or not isinstance(trainable_decoder_layers, int)
            or not 1 <= trainable_decoder_layers <= DECODER_LAYERS
        ):
            raise ValueError(f"trainable_decoder_layers must be an int in 1..{DECODER_LAYERS}")
        model, _ = self._require_model()
        n_layers = self._decoder_layers()
        if trainable_decoder_layers > n_layers:
            raise ValueError(f"trainable_decoder_layers must be an int in 1..{n_layers} for this model")
        first = n_layers - trainable_decoder_layers
        prefixes = tuple(f"decoder.layer.{k}." for k in range(first, n_layers))
        prefixes += ("decoder.final_layer_norm.",)
        return [name for name, _p in model.named_parameters() if name.startswith(prefixes)]

    def _encode(self, paths: Sequence[str], *, batch_size: int = 4) -> dict[str, tuple[Any, Any]]:
        """The frozen image encoder's output and patch mask per image path (computed once, kept on the model
        device): Pix2Struct's encoder runs over up to MAX_PATCHES variable-resolution patches, and it is the
        expensive half of a training step."""
        import torch

        model, processor = self._require_model()
        device = next(model.parameters()).device
        out: dict[str, tuple[Any, Any]] = {}
        unique = list(dict.fromkeys(paths))
        for start in range(0, len(unique), batch_size):
            chunk = unique[start : start + batch_size]
            images = []
            for path in chunk:
                with Image.open(path) as image:
                    images.append(image.convert("RGB"))
            inputs = processor(images=images, return_tensors="pt").to(device)
            with torch.inference_mode():
                hidden = model.encoder(
                    flattened_patches=inputs["flattened_patches"], attention_mask=inputs["attention_mask"]
                ).last_hidden_state
            for index, path in enumerate(chunk):
                out[path] = (hidden[index].detach().clone(), inputs["attention_mask"][index].detach().clone())
        return out

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 4,
        lr: float = 5e-5,
        batch_size: int = 8,
        trainable_decoder_layers: int = DEFAULT_TRAINABLE_DECODER_LAYERS,
        seed: int = 0,
        min_gain: float = 0.0,
        val_max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded supervised fine-tuning on a validated captioning dataset.

        Only the last `trainable_decoder_layers` blocks of the caption decoder and the decoder's final layer
        norm train (2 blocks by default); the image encoder, every embedding and the untied output projection
        stay frozen. The frozen encoder's output is computed once per training image and reused across epochs.
        Every (image, reference caption) pair is one training sample, so an image with five references is seen
        five times per epoch with a different target each time; the target is the tokenised caption with its
        end-of-sequence token, decoded with teacher forcing from the decoder start token and scored with the
        model's own cross-entropy (padding ignored); AdamW at a fixed learning rate with gradient clipping at
        1.0, no scheduler. Epoch 0 records the frozen model's validation metrics; the epoch with the highest
        validation CIDEr-D is kept (an exact tie keeps the earlier epoch). With `min_gain` > 0 an adapted
        epoch is
        kept only if its validation CIDEr-D beats epoch 0 by at least `min_gain`; otherwise epoch 0 (the
        frozen
        weights) is kept and `kept_frozen_by_margin` is True."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not (0.0 < lr <= 1e-3):
            raise ValueError("lr must be in (0, 1e-3]")
        if isinstance(min_gain, bool) or not isinstance(min_gain, (int, float)) or not 0.0 <= min_gain < 1.0:
            raise ValueError("min_gain must be a number in [0, 1)")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 64:
            raise ValueError("batch_size must be an int in 1..64")
        names = self._trainable_names(trainable_decoder_layers)
        train_checked = validate_dataset(train)["records"]
        val_checked = (
            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"] if val else []
        )
        import torch

        torch.manual_seed(seed)
        model, processor = self._require_model()
        tokenizer = processor.tokenizer
        pad_id = int(tokenizer.pad_token_id)
        started = time.perf_counter()
        wanted = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=ADAPT_WEIGHT_DECAY)
        device = next(model.parameters()).device
        encoded = self._encode([r["image"] for r in train_checked])
        pairs = [(i, str(c)) for i, r in enumerate(train_checked) for c in r["captions"]]

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            return {
                k: v
                for k, v in self.evaluate(val_checked, max_new_tokens=val_max_new_tokens).items()
                if k in ("bleu4", "rouge_l", "cider_d", "unigram_f1", "mean_words", "n", "predictions")
            }

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}
        history.append(entry)
        if progress:
            progress(entry)
        best_score = entry["val"]["cider_d"] if entry["val"] else -math.inf
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        initial_state = {k: v.clone() for k, v in best_state.items()}
        best_epoch = 0
        generator = torch.Generator().manual_seed(seed)
        try:
            for epoch in range(1, epochs + 1):
                model.train()
                order = torch.randperm(len(pairs), generator=generator).tolist()
                losses = []
                for start in range(0, len(order), batch_size):
                    chosen = [pairs[j] for j in order[start : start + batch_size]]
                    targets = tokenizer(
                        [c for _i, c in chosen],
                        padding=True,
                        truncation=True,
                        max_length=MAX_NEW_TOKENS + 1,
                        return_tensors="pt",
                    ).to(device)
                    labels = targets["input_ids"].masked_fill(targets["input_ids"] == pad_id, -100)
                    hidden = torch.stack([encoded[train_checked[i]["image"]][0] for i, _c in chosen])
                    mask = torch.stack([encoded[train_checked[i]["image"]][1] for i, _c in chosen])
                    out = model(encoder_outputs=(hidden,), attention_mask=mask, labels=labels)
                    optimiser.zero_grad(set_to_none=True)
                    out.loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, ADAPT_GRAD_CLIP)
                    optimiser.step()
                    losses.append(float(out.loss.detach()))
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": score_val()}
                history.append(entry)
                if progress:
                    progress(entry)
                current = entry["val"]["cider_d"] if entry["val"] else math.inf
                if current > best_score or not entry["val"]:
                    best_score = current
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
                    best_epoch = epoch
        except BaseException:
            # Transactional: a failure in training, validation or the progress callback leaves the base
            # exactly as it was, with every parameter frozen again.
            restore = dict(model.state_dict())
            restore.update(initial_state)
            model.load_state_dict(restore, strict=True)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            self.adapter = None
            raise
        frozen_val = history[0]["val"]["cider_d"] if history[0]["val"] else None
        kept_frozen_by_margin = False
        below_margin = frozen_val is not None and best_score - frozen_val < min_gain - 1e-12
        if min_gain > 0 and best_epoch != 0 and below_margin:
            best_state, best_epoch, kept_frozen_by_margin = initial_state, 0, True
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable_decoder_layers": trainable_decoder_layers,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": (
                "highest validation CIDEr-D"
                + (f", kept only if it beats epoch 0 by >= {min_gain}" if min_gain else "")
                if val_checked
                else "final epoch (no validation split)"
            ),
            "min_gain": min_gain,
            "val_max_new_tokens": val_max_new_tokens,
            "kept_frozen_by_margin": kept_frozen_by_margin,
            "lr": lr,
            "batch_size": batch_size,
            "n_train": len(train_checked),
            "n_pairs": len(pairs),
            "n_val": len(val_checked),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts ------------------------------------------------------------------------------------

    def adapter_state_sha256(self) -> str:
        """SHA-256 over the adapter's trainable tensors as they are in the model now (sorted names,
        float32 bytes): lets a caller check that the in-memory model is the selected adapted model before
        scoring with it."""
        if self.adapter is None:
            raise ValueError("no adapter: the pipeline holds the frozen base")
        model, _ = self._require_model()
        names = set(self.adapter["trainable_names"])
        digest = hashlib.sha256()
        for name, tensor in sorted((k, v) for k, v in model.state_dict().items() if k in names):
            digest.update(name.encode("utf-8"))
            digest.update(tensor.detach().float().cpu().contiguous().numpy().tobytes())
        return digest.hexdigest()

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted decoder-block and final-norm tensors as safetensors plus a base manifest."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        model, _ = self._require_model()
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "weight_file": WEIGHT_FILE,
                "weight_sha256": WEIGHT_SHA256,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return out

    def _check_artifact_manifest(self, root: Path, manifest: Mapping[str, Any]) -> Path:
        """Refuse an artifact whose manifest is not exactly the one this pipeline writes: the supported format
        and version, the pinned base (id, revision, weight file, digest), exactly one file entry named
        `adapter.safetensors` that resolves inside the artifact directory, and a recorded
        `trainable_decoder_layers` in range. Nothing is deserialised here. The digest check that follows
        detects corruption or drift of the weights relative to the adjacent manifest; it is not authenticity
        against an actor who can replace both files."""
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(
                f"artifact format_version {manifest.get('format_version')!r} is not the supported "
                f"{ARTIFACT_FORMAT_VERSION!r}"
            )
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
            MODEL_ID,
            MODEL_REVISION,
            WEIGHT_SHA256,
        ):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        if base.get("weight_file", WEIGHT_FILE) != WEIGHT_FILE:
            raise ValueError("artifact was adapted from a different base weight file")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1:
            raise ValueError("artifact manifest must list exactly one file")
        entry = files[0]
        if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact manifest must name exactly {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / entry["path"]).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weight path must resolve inside the artifact directory")
        adapter = manifest.get("adapter")
        layers = adapter.get("trainable_decoder_layers") if isinstance(adapter, Mapping) else None
        if isinstance(layers, bool) or not isinstance(layers, int) or not 1 <= layers <= DECODER_LAYERS:
            raise ValueError("artifact manifest does not record an in-range integer trainable_decoder_layers")
        if not isinstance(manifest.get("tensors"), list):
            raise ValueError("artifact manifest must list its tensors")
        return weights_path

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, digest and exact tensor set **before** deserialising, then overwrite
        exactly the tensors it carries."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path = self._check_artifact_manifest(root, manifest)
        entry = manifest["files"][0]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        # The exact tensor set the recorded configuration implies — no subset, no extra, no other layer.
        expected = sorted(self._trainable_names(manifest["adapter"]["trainable_decoder_layers"]))
        if sorted(manifest["tensors"]) != expected:
            raise ValueError("artifact tensor list does not match its recorded configuration")
        model, _ = self._require_model()
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from its manifest")
        state = model.state_dict()
        for key, value in tensors.items():
            if key not in state or not key.startswith("decoder."):
                raise ValueError(
                    f"artifact tensor {key} is not an adaptable caption-decoder tensor of the base"
                )
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(
                    f"artifact tensor {key} has shape {tuple(value.shape)}, "
                    f"base has {tuple(state[key].shape)}"
                )
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})
        model.load_state_dict(merged, strict=True)
        model.eval()
        self.adapter = {
            **manifest["adapter"],
            "trainable_names": manifest["tensors"],
            "history": manifest.get("history", []),
        }
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Pix2StructTextCapsPipeline:
        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 2/3:** `src/pix2struct_textcaps_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
# @title Infrastructure: carried module 2/3: src/pix2struct_textcaps_pipeline/metrics.py (verbatim; see the note above)
"""Corpus-level captioning metrics and two non-neural baselines, in pure Python.

`pipeline.py` keeps the per-caption plumbing check (`unigram_f1`); this module implements the three metrics a
captioning result is normally read by, over a set of records with several references each:

- **BLEU-4** (Papineni et al. 2002): corpus-level, clipped n-gram precision for n = 1..4 with geometric mean
  and brevity penalty against the closest reference length — the `coco-caption` convention.
- **ROUGE-L** (Lin 2004): per image the longest-common-subsequence F-measure (beta = 1.2) built, as in
  `coco-caption`, from the **best precision and the best recall over the references taken separately**,
  averaged over images.
- **CIDEr-D** (Vedantam et al. 2015): TF-IDF-weighted n-gram cosine similarity for n = 1..4 with the
  Gaussian length penalty (sigma = 6) and clipping of candidate counts to the reference counts, averaged over
  references and n, scaled by 10. The document frequencies are computed over the references of the evaluated
  set, so a score is comparable only across systems evaluated on the same records (which is how it is used
  here: frozen vs. adapted vs. baselines on the same test split).

All three use a **simplified tokenisation, not the PTB tokenizer** that `coco-caption` runs
(`normalize_caption`:
lower-case, every non-word character replaced by a space, so `Campbell's` becomes `campbell s` and `T-shirt`
becomes
`t shirt`); absolute values therefore differ from `pycocoevalcap` on the same captions. Two baselines a
fine-tuned model must beat: **constant caption** (the training caption that scores best against all other
training references — the corpus medoid, one string for every image) and **colour nearest neighbour** (the
caption of the training image whose 3x3 mean-colour grid is closest — a lookup that knows the image only
through 27 numbers).
"""

from __future__ import annotations

import math
import random
from collections import Counter
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

from PIL import Image

# standalone rewrite (build_notebook.py): `from .pipeline import caption_tokens, unigram_f1` removed — names are kernel globals defined by the carried modules

MAX_NGRAM = 4
ROUGE_BETA = 1.2
CIDER_SIGMA = 6.0
COLOUR_GRID = 3
METRIC_DEFINITIONS = {
    "bleu4": (
        "corpus-level BLEU-4: geometric mean of clipped 1..4-gram precisions against all references, times "
        "the brevity penalty against the closest reference length; in 0..1"
    ),
    "rouge_l": (
        "mean over images of the LCS-based F-measure (beta 1.2) from the best precision and the best "
        "recall over the references taken separately (coco-caption aggregation); in 0..1"
    ),
    "cider_d": (
        "mean over images of the CIDEr-D consensus score: TF-IDF-weighted 1..4-gram cosine similarity to the "
        "references with a Gaussian length penalty (sigma 6), clipped counts, scaled by 10; document "
        "frequencies from the references of the evaluated set; typically 0..~1.5 on VizWiz"
    ),
    "unigram_f1": "mean over images of the bag-of-words F1 against the best-matching reference; in 0..1",
}


def _ngrams(tokens: Sequence[str], n: int) -> Counter[tuple[str, ...]]:
    return Counter(tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1))


def _check(predictions: Sequence[str], references: Sequence[Sequence[str]]) -> None:
    if len(predictions) != len(references):
        raise ValueError(f"{len(predictions)} predictions but {len(references)} reference lists")
    if not predictions:
        raise ValueError("no predictions to score")
    for index, refs in enumerate(references):
        if isinstance(refs, str) or not refs:
            raise ValueError(f"references[{index}] must be a non-empty list of captions")


def bleu4(predictions: Sequence[str], references: Sequence[Sequence[str]]) -> float:
    """Corpus-level BLEU-4 with multi-reference clipping and the closest-length brevity penalty."""
    _check(predictions, references)
    matched = [0] * MAX_NGRAM
    total = [0] * MAX_NGRAM
    hyp_len = ref_len = 0
    for prediction, refs in zip(predictions, references, strict=True):
        hyp = caption_tokens(prediction)
        ref_tokens = [caption_tokens(r) for r in refs]
        hyp_len += len(hyp)
        ref_len += min((abs(len(r) - len(hyp)), len(r)) for r in ref_tokens)[1]
        for n in range(1, MAX_NGRAM + 1):
            counts = _ngrams(hyp, n)
            max_ref: Counter[tuple[str, ...]] = Counter()
            for r in ref_tokens:
                for gram, count in _ngrams(r, n).items():
                    max_ref[gram] = max(max_ref[gram], count)
            matched[n - 1] += sum(min(count, max_ref[gram]) for gram, count in counts.items())
            total[n - 1] += max(len(hyp) - n + 1, 0)
    if any(m == 0 for m in matched) or hyp_len == 0:
        return 0.0
    log_precision = sum(math.log(m / t) for m, t in zip(matched, total, strict=True)) / MAX_NGRAM
    brevity = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len / hyp_len)
    return brevity * math.exp(log_precision)


def _lcs(a: Sequence[str], b: Sequence[str]) -> int:
    if not a or not b:
        return 0
    previous = [0] * (len(b) + 1)
    for token in a:
        current = [0]
        for j, other in enumerate(b):
            current.append(previous[j] + 1 if token == other else max(previous[j + 1], current[j]))
        previous = current
    return previous[-1]


def rouge_l(prediction: str, references: Sequence[str]) -> float:
    """LCS F-measure (beta 1.2) as `coco-caption` aggregates it: the best LCS precision and the best LCS
    recall over
    the references are taken separately and then combined (not the best per-reference F)."""
    hyp = caption_tokens(prediction)
    best_precision = best_recall = 0.0
    for reference in references:
        ref = caption_tokens(reference)
        lcs = _lcs(hyp, ref)
        if lcs == 0:
            continue
        best_precision = max(best_precision, lcs / len(hyp))
        best_recall = max(best_recall, lcs / len(ref))
    if best_precision == 0.0 or best_recall == 0.0:
        return 0.0
    return (1 + ROUGE_BETA**2) * best_precision * best_recall / (best_recall + ROUGE_BETA**2 * best_precision)


def _document_frequencies(references: Sequence[Sequence[str]]) -> list[Counter[tuple[str, ...]]]:
    """Per n, the number of images whose references contain each n-gram."""
    dfs: list[Counter[tuple[str, ...]]] = [Counter() for _ in range(MAX_NGRAM)]
    for refs in references:
        for n in range(1, MAX_NGRAM + 1):
            seen: set[tuple[str, ...]] = set()
            for r in refs:
                seen.update(_ngrams(caption_tokens(r), n))
            for gram in seen:
                dfs[n - 1][gram] += 1
    return dfs


def _tfidf(
    tokens: Sequence[str], n: int, df: Counter[tuple[str, ...]], log_images: float
) -> tuple[dict[tuple[str, ...], float], float]:
    counts = _ngrams(tokens, n)
    vector = {gram: count * (log_images - math.log(max(df[gram], 1.0))) for gram, count in counts.items()}
    return vector, math.sqrt(sum(v * v for v in vector.values()))


class _CiderIndex:
    """Document frequencies and reference TF-IDF vectors of one evaluated set, computed once."""

    def __init__(self, references: Sequence[Sequence[str]]) -> None:
        self.dfs = _document_frequencies(references)
        self.log_images = math.log(len(references))
        self.refs: list[list[tuple[list[str], list[tuple[dict[tuple[str, ...], float], float]]]]] = []
        for refs in references:
            entry = []
            for r in refs:
                tokens = caption_tokens(r)
                entry.append((tokens, [self._vector(tokens, n) for n in range(1, MAX_NGRAM + 1)]))
            self.refs.append(entry)

    def _vector(self, tokens: Sequence[str], n: int) -> tuple[dict[tuple[str, ...], float], float]:
        return _tfidf(tokens, n, self.dfs[n - 1], self.log_images)

    def score(self, prediction: str, image_index: int) -> float:
        hyp = caption_tokens(prediction)
        hyp_vectors = [self._vector(hyp, n) for n in range(1, MAX_NGRAM + 1)]
        refs = self.refs[image_index]
        total = 0.0
        for ref_tokens, ref_vectors in refs:
            penalty = math.exp(-((len(hyp) - len(ref_tokens)) ** 2) / (2 * CIDER_SIGMA**2))
            for (hyp_vec, hyp_norm), (ref_vec, ref_norm) in zip(hyp_vectors, ref_vectors, strict=True):
                if hyp_norm == 0 or ref_norm == 0:
                    continue
                dot = sum(min(hyp_vec[g], ref_vec[g]) * ref_vec[g] for g in hyp_vec if g in ref_vec)
                total += dot / (hyp_norm * ref_norm) * penalty
        return 10.0 * total / (MAX_NGRAM * len(refs))


def cider_d(predictions: Sequence[str], references: Sequence[Sequence[str]]) -> list[float]:
    """Per-image CIDEr-D scores with document frequencies from `references` (Vedantam et al. 2015,
    the `coco-caption` implementation)."""
    _check(predictions, references)
    index = _CiderIndex(references)
    return [index.score(p, i) for i, p in enumerate(predictions)]


def caption_metrics(predictions: Sequence[str], references: Sequence[Sequence[str]]) -> dict[str, Any]:
    """BLEU-4, ROUGE-L, CIDEr-D and unigram F1 over parallel predictions and reference lists."""
    _check(predictions, references)
    pairs = list(zip(predictions, references, strict=True))
    cider = cider_d(predictions, references)
    return {
        "n": len(predictions),
        "bleu4": bleu4(predictions, references),
        "rouge_l": sum(rouge_l(p, r) for p, r in pairs) / len(pairs),
        "cider_d": sum(cider) / len(cider),
        "unigram_f1": sum(unigram_f1(p, r) for p, r in pairs) / len(pairs),
        "empty_rate": sum(1 for p in predictions if not caption_tokens(p)) / len(predictions),
        "mean_words": sum(len(caption_tokens(p)) for p in predictions) / len(predictions),
        "definitions": dict(METRIC_DEFINITIONS),
    }


def _references(records: Sequence[Mapping[str, Any]]) -> list[list[str]]:
    return [[str(c) for c in r["captions"]] for r in records]


def medoid_caption(train: Sequence[Mapping[str, Any]]) -> str:
    """The training caption with the highest mean CIDEr-D against every other training image's references."""
    if not train:
        raise ValueError("the constant-caption baseline needs training records")
    candidates = sorted({str(c) for r in train for c in r["captions"]})
    index = _CiderIndex(_references(train))
    best, best_score = candidates[0], -1.0
    for candidate in candidates:
        score = sum(index.score(candidate, i) for i in range(len(train))) / len(train)
        if score > best_score:
            best, best_score = candidate, score
    return best


def constant_caption_baseline(
    train: Sequence[Mapping[str, Any]], records: Sequence[Mapping[str, Any]]
) -> dict[str, Any]:
    """The same caption (the training medoid) for every image."""
    caption = medoid_caption(train)
    result = caption_metrics([caption] * len(records), _references(records))
    result["baseline"] = f"constant caption {caption!r}"
    return result


def colour_signature(image: str | Path | Image.Image, *, grid: int = COLOUR_GRID) -> list[float]:
    """Mean RGB of each cell of a `grid` x `grid` partition of the image, in 0..1 (27 numbers by default)."""
    handle = image if isinstance(image, Image.Image) else Image.open(image)
    with handle:
        small = handle.convert("RGB").resize((grid * 8, grid * 8), Image.BILINEAR)
        pixels = list(small.getdata())
    out: list[float] = []
    for row in range(grid):
        for col in range(grid):
            cell = [pixels[(row * 8 + y) * grid * 8 + col * 8 + x] for y in range(8) for x in range(8)]
            out.extend(sum(p[channel] for p in cell) / (64 * 255.0) for channel in range(3))
    return out


def colour_neighbour_baseline(
    train: Sequence[Mapping[str, Any]], records: Sequence[Mapping[str, Any]]
) -> dict[str, Any]:
    """Caption every image with the first reference of the training image whose colour signature is
    closest (Euclidean distance over the 3x3 mean-colour grid; no model)."""
    if not train:
        raise ValueError("the colour-neighbour baseline needs training records")
    signatures = [(colour_signature(r["image"]), str(r["captions"][0])) for r in train]
    predictions = []
    for record in records:
        query = colour_signature(record["image"])
        predictions.append(
            min(signatures, key=lambda s: sum((a - b) ** 2 for a, b in zip(s[0], query, strict=True)))[1]
        )
    result = caption_metrics(predictions, _references(records))
    result["baseline"] = f"colour nearest neighbour ({len(train)} training images)"
    return result


BOOTSTRAP_RESAMPLES = 2_000
BOOTSTRAP_SEED = 0


def paired_bootstrap_ci(
    differences: Sequence[float],
    *,
    n_boot: int = BOOTSTRAP_RESAMPLES,
    seed: int = BOOTSTRAP_SEED,
    level: float = 0.95,
) -> dict[str, Any]:
    """Percentile bootstrap interval of the mean of paired per-image differences (e.g. adapted minus frozen
    CIDEr-D,
    each image scored once on the whole evaluated set): images are resampled with replacement `n_boot` times
    with a
    fixed seed. The verdict reads the interval: above 0, below 0, or no difference detected."""
    values = [float(d) for d in differences]
    if not values:
        raise ValueError("no paired differences to resample")
    rng = random.Random(seed)
    n = len(values)
    means = sorted(sum(values[rng.randrange(n)] for _ in range(n)) / n for _ in range(n_boot))
    tail = (1.0 - level) / 2.0
    low = means[int(math.floor(tail * (n_boot - 1)))]
    high = means[int(math.ceil((1.0 - tail) * (n_boot - 1)))]
    if low > 0:
        verdict = "adapted above frozen"
    elif high < 0:
        verdict = "adapted below frozen"
    else:
        verdict = "no difference detected"
    mean = sum(values) / n
    return {
        "n": n, "mean": mean, "low": low, "high": high, "level": level, "n_boot": n_boot, "seed": seed,
        "verdict": verdict,
    }

**Module 3/3:** `src/pix2struct_textcaps_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
# @title Infrastructure: carried module 3/3: src/pix2struct_textcaps_pipeline/samples.py (verbatim; see the note above)
"""Captioning dataset contract for fine-tuning: the pinned VizWiz-Captions sample, validation, seeded
image-disjoint splitting, BYOD loaders and JSONL export.

The default dataset is **real** and out of the base model's distribution: VizWiz-Captions (Gurari et al.,
ECCV 2020; CC BY 4.0) — photographs taken by blind people, each with five crowd-written captions, from a
population and a caption style (what is held, what the label says, how the picture is framed) that TextCaps
captions do not cover. `google/pix2struct-textcaps-base` was fine-tuned on TextCaps and never on VizWiz.
The sample comes from one pinned parquet shard of the Hub mirror `mm-eval/VizWiz-Captions` (a re-conversion
of the official `val.json` with the rejected and pre-canned captions dropped, so an image can carry fewer
than five references): the four text columns of all 1,550 rows are read **column-only** over HTTPS range
requests through `pyarrow` (about 0.5 MB), and the image column of **row group 0 only** (336 photographs,
about 84 MB) is read the same way — the shard's declared size and SHA-256 checked against the pins before
any byte is read, the decoded text columns' SHA-256 and every photograph's SHA-256 and size checked after.
Images without a surviving reference caption are excluded from the sample.

A record is ``{id, image_id, image, captions, category}`` — the path of the digest-verified photograph, its
one or more reference captions and a category (`text` when the VizWiz annotators flagged text in the
photograph, `no-text` otherwise; BYOD records may carry any label, `other` by default). Every record's image
is its own (one row per photograph), so a split by image is a split by record; BYOD records may share an
`image_id` and are then kept together.
"""

from __future__ import annotations

import hashlib
import io
import json
import random
import re
import urllib.request
from collections import Counter
from collections.abc import Callable, Mapping, Sequence
from pathlib import Path
from typing import Any

from PIL import Image

# standalone rewrite (build_notebook.py): `from .pipeline import (` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "VizWiz-Captions"
CORPUS_REPO = "mm-eval/VizWiz-Captions"
CORPUS_REVISION = "c4a6d897836e7885d0095134f92d392e4e770539"
CORPUS_RELEASE = (
    "VizWiz-Captions v1 (2020) as re-converted on the Hugging Face Hub, dataset revision c4a6d897"
)
CORPUS_LICENSE = "CC BY 4.0 (Gurari et al. 2020; vizwiz.org/tasks-and-datasets/image-captioning)"
CORPUS_TEXT_COLUMNS = ("id", "answer", "question_type", "text_detected")
CORPUS_IMAGE_COLUMN = "media"
CORPUS_FILE: dict[str, Any] = {
    "path": "data/val-00004-of-00005.parquet",
    "bytes": 392_245_504,
    "sha256": "4492465a41d32b3c12b8b7b6a0cf7e0a0e202a5b825b006ca0c85dcdf24efd3e",
    "rows": 1_550,
    "text_sha256": "9799ebb13cf6a7e7c76afdd180892e21499ecefff697212ce4e505c7fc207d6e",
    "row_group": 0,
    "row_group_rows": 336,
}
DEFAULT_CACHE_DIR = Path("weights") / "vizwiz-captions"
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 208, "validation": 40, "test": 70}  # the 318 row-group-0 photographs with a caption
MIN_RECORDS = 8
MAX_RECORDS = 5_000
MIN_CAPTIONS = 1
MAX_CAPTION_CHARS = 500
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")
# image id -> (sha256, bytes) of the JPEG carried by row group 0 of the pinned shard (336 photographs).
IMAGE_PINS: dict[str, tuple[str, int]] = {
    "29631": ("635042d060e1f42bdebf7d16af7b7b1f78763482b21959dd6ab3e642fb131522", 219537),
    "29632": ("595732108a90de2ebb653d8bac8ab1d88d837624103c0a3344161f57ff84bd7b", 208376),
    "29633": ("93ff2bd506c778c53715a965a83d8d55a6d3a46b4b06dd91108689284ba3aa5f", 14682),
    "29634": ("b7bfa7344d17b18dcc03f3fdcecb874963a3584d14b92210717f351155e42d0a", 232493),
    "29635": ("f807878774e35d16396c8d6600c67b8bd6cb93a89328da8d29bfd8c34386ec85", 268910),
    "29636": ("fa7f75a23aaedfeb26032a7bce86bdfaace2d204695efafc17b87a0907bae611", 82416),
    "29637": ("407efaaabffc133f142c0c690318a16812c39fc30902ac95d86b3b8640b511c9", 506359),
    "29638": ("486f02412386b81757936e3e9782efe234fe69760d24fc70734a0b60d7b284b0", 443910),
    "29639": ("37cfae24bbfb0faf1d6fadda974b2824bf8976ffe7fac02f740eee75027d95b3", 349992),
    "29640": ("da467dffb38a6d407cee8fe1b1b690a3d8af70f378941e0dd992284f9d2d8f5e", 269887),
    "29641": ("4f2e2bea724a6b3b3ead65718da3b3d17bcd34f56913c0652ba91d89acd6967e", 257024),
    "29642": ("afbdef671c1faa2f191466f7320b07a18c583afe2c3e856eccefec46532f29db", 225839),
    "29643": ("cdb9b4f804e4ee61754d82560940c72fe21a3747fac6c75648faa613231da76a", 261053),
    "29644": ("632a0a1662b58ad6b13cefef34a51898b93fcdf88abebc9217e312691128f569", 193354),
    "29645": ("889242e35ee151e13fe6243567126d0476c9221a3985ecf8dce4f6b415f2b59e", 258707),
    "29646": ("0454ddcec779bfbfafd093518f67165b2a7a092e81917e78c213aa18f6dbb8cb", 346643),
    "29647": ("d0ef537515306a3df5c77d85dd0d2666c4aa0da2f72502cc82500cf6524d1e19", 271956),
    "29648": ("ceb4c0b7b00bbb1dfbdacec67ea4d8fc8d59754a9856c0eb87a52a554a570664", 321526),
    "29649": ("dccb89f70f8be06bdcff3e4947422fb5a69cf5069b00e707c22e92090fe91397", 272027),
    "29650": ("f5e41db77978764a79010a335c6e21d5a053892517dd34a4dfd315ae17eeeb1a", 434003),
    "29651": ("72be2c699582d963754a4986c56c22701511cef3c802c48e0d7aa558f9107a09", 292153),
    "29652": ("7db7f94532355552e80fbeb015d9e4db978398cc57512a9c96bd0db44a50c0c4", 338389),
    "29653": ("0a84818b7f47bf36c5e6e4001481dfc8faec6bf232792ce521b75db2cf50ec86", 188622),
    "29654": ("15eca622fe300527a6c92df119b8a2f3947df676d627047ea50e96de39303d19", 49771),
    "29655": ("b2e54c27133737e21ca33e858ef52dd66b7a68bd9144f947094702295511a891", 39778),
    "29656": ("719f5852b6092754ac26f3feb6d6d974da19a848146ea3bced950e71dd55cc0b", 144496),
    "29657": ("568dd4519806b64e63d40f063205e485c1fc9f42a6719fff7acbe59d94c01148", 199012),
    "29658": ("dac90383b9f44fbff68426da47687c124f2511152ce4d3e94c307855a43e3871", 337908),
    "29659": ("61be884a769cb69f89e847cb643c4a78c01e747934d56186a53a8b3e6daf244f", 30681),
    "29660": ("08fac9a7c1c30e0a1aa92f7295f3366454d4629a78bd3678b86090ffff8fc373", 205850),
    "29661": ("72e87e97b362c7fe3412676ca230cb5765cd6d9091d530e4f1f894458344c97d", 330913),
    "29662": ("61bb302f64ec861c9628b1efc04c065a1fffe4c9fa6688b16d06ce946eec0c1f", 182155),
    "29663": ("a417180d3f937c01b9269cab5e46e096a05714c0c2fdf630bf7430cb351e44e9", 258175),
    "29664": ("3913b82cecc6a49c057c9cc4dde219920f62aa9c43f2b1843c58cebdefa5ee20", 217507),
    "29665": ("e4b02231db046fc3b6b25cfd434ba4ed25ba74f98941cedc7127fa3d5579c0e5", 673261),
    "29666": ("1d6e584c39a7b469a97f7125ab91dbfed50ce151c76c0037365754b3da1935ec", 415821),
    "29667": ("994e2c3cf86c546c10480b01064a9af915add972553ad8b4645a3e5a183969fe", 459367),
    "29668": ("f6c0fc8835171e91cf8368be8c33e8af565c037e89fdd61293e92dc61f7a57ca", 283625),
    "29669": ("3a465d5426ce6c766a9aaee0aa135ff5b438197587a21111bb06f32d827a4229", 155717),
    "29670": ("e60a335c69895e52db1f1265533f773977117819d10cffc7c7a6fe5b77fd911b", 218339),
    "29671": ("45a785cec7d063f545f72b8e5f1c3ce1820c43f62bd725fdaff2070db47468ff", 301005),
    "29672": ("20df609ab7455f25c26a8ad7e79224e921dd3656de31c8e9757b0bcda5a3e6e8", 271522),
    "29673": ("9009def8616940b157d2133306edd1a2f7f39b129a158a9a19ebd88069f246d8", 36083),
    "29674": ("5160d7be46d40cd6cb3b122d94b48e8791ed0a939408884b130b11ce3eff64aa", 311110),
    "29675": ("ea0dcee84b2b3ccaf9371e8ffecc779f6a7acf8a1303b3c3438c3b507a3fde8a", 331823),
    "29676": ("799e06916b505131dbc20db33c6aef34f9c325a347819de063c3faa599c1e607", 209338),
    "29677": ("67fe5425d3f0e7aa2a260d854cf25ea1c1c41f9cb30dc3691b16700966011c85", 176798),
    "29678": ("80727a1e760ecee07214bfec0448fda106fc07f3c849a4248b347514b0fcfa91", 591377),
    "29679": ("80e29692260e0ade581839baec5d5a4550e5822cab0aa15115cd3c96a1569f3e", 33215),
    "29680": ("7c836719c64e5d97ba83b99a6fba966062a180a1e3aaffa30dde4d2117daef89", 373575),
    "29681": ("2416fea174cfd4522953171bbcaa38900aa38d3e9671c6b4ff9dbe314caa5f13", 293047),
    "29682": ("05c5971c974f1c9843ce2bb0d08c1500e380596107ac76a24b057b0efa658573", 466144),
    "29683": ("f3f6e73a1d1dd7ba7449ba0634b09decfd2350dfbe8f3fdcf3fbbdbe2719b5b7", 185222),
    "29684": ("92a3f9f7bcd352a6f1c03d305f98aec1f1d3cc315f3d6c2bab46f13bca7817b6", 325758),
    "29685": ("2db1c9bc24a7a12af81c2d8a5539e422d210e03eda0ecc0af6944e98bdea1305", 304402),
    "29686": ("fd3ebb92d81317be6db45ad6326d3227f095edd2aa93c2f3e571806b5a4301b9", 379396),
    "29687": ("8be2b3264792a2ac22abc9a613d5759e9f31e79b3659f78039a70c08d449860a", 233109),
    "29688": ("d0862ed97636b9cb77577c5ba6a82cb75176548231568a1a50c00bb6c47700d1", 301273),
    "29689": ("31b332bfc7af3903b1f6091cc92627ca299dbe9c2e2330f730228e505e573296", 129869),
    "29690": ("d39dbd910bd9fbe3b36faa99b6580cf371b1aff87230066f91f46d1a55dd124e", 224998),
    "29691": ("3012921370798079afe212db4bc22cfbf284bd7584e74cf33297956bd9189f8e", 385230),
    "29692": ("903718f7a211747b1677bc680f72facb3a4d582abd03131dd98c88ece06d0c54", 280343),
    "29693": ("d44c2877651a8dba7cf2943265f0f6353ff38697cf921fc14c33a5a1afcbcd73", 196010),
    "29694": ("0a20818fe43942fd653591cbed494c38a2550e27e3892753c278142b462f900d", 165487),
    "29695": ("98650214fe93da1f94e6ad9d0fc19bb79e2485223e706e1ba292a50af85c988f", 63581),
    "29696": ("ddfd96ca16ada6259061247926d6bda0d302b3ec7adbda44e66f0a31c27baf11", 490864),
    "29697": ("7e68d03f88d8e6a65ed883ccf45f5b2e1c7b1668e5b168355650311601fb3979", 276114),
    "29698": ("dc8bf976fc17cebf01fbf6779c8400ccf27a12ac7124cf7490a560051c18b50b", 520196),
    "29699": ("54e6417e77612a9b8444e4febb025e12e1488b05aa72fbcf63e8bd69a8bf6652", 52962),
    "29700": ("b03833951a0c568edc3a1568ad14769b58615b57a32d50b297fe12df05989128", 178206),
    "29701": ("d1d5c7795f4c794645bc728e02efe447bc8a4c896aa397fb843925d510262e70", 204765),
    "29702": ("d175d905034ca667379229035051f90abcd49d42d9aedd17cfc19bf6dc4ed7db", 329707),
    "29703": ("429e3c4a9c822847f9670251c8027485505686724caa0cf467fecb4b46afd7e7", 98889),
    "29704": ("1302773b4d1d13c8242e9b6fe283f85bdc6331b01de77f795eded9d2a399d334", 208973),
    "29705": ("beb66040a0e41e4f3ae26db0cb2d78321bfcb71b6f2fe3198671ceda10e3ffa6", 305340),
    "29706": ("ac55c0fd7e334f1a5cdf41006c33cd1c699ac71a9a95556e27f6ea564ae8a587", 260218),
    "29707": ("f86d9785862905e412f37cc2dbd49836c5aa6ead03c586a4f9a76555f1894305", 397788),
    "29708": ("f69d115f7a11814d05ab5e8e2b90070938a71edd890dbc4ae6eb4c62a96d6b8c", 339030),
    "29709": ("dd65b40eee8a23e0bdf47b67d5a3be54b31913ba9918fabe4a9d4aaf068f63aa", 218005),
    "29710": ("741594d8849df32c38f1898498573c18da785e19dea9481723d770b7d47bfad0", 480990),
    "29711": ("b4cb496aa54857053256ea084a4390bc9a157c923ac601c77057ef257439b7d7", 429787),
    "29712": ("8473b6193e761171dd28d3c2cb4894dfeee51b014330c2289cfd40b8d8a53c92", 315871),
    "29713": ("569fa73a25181a278b420556c8bfcce2e2abeb8e36e844c53c977ac8c3aa3e23", 254682),
    "29714": ("c55af0d333f124114fe440ca756b5d39ead1c14b07ebacedbd52300622e46454", 322112),
    "29715": ("d3007697c34a7ef94452e5ef3fd260c549fbfe37473d015c25f880e83f8bd284", 208474),
    "29716": ("a53d0db806788cc105ebd5ab4749eb7a31617a95b0394cc30a808bc5d9f79667", 375964),
    "29717": ("37d835e42fc5693772402122065dbbf8bb7da4642e63c26a97609146dfe64cb2", 30875),
    "29718": ("1682aeae89a9d92c6484dd7ba787922846f463103c4b58fcf3251e91c5190a16", 189651),
    "29719": ("ad695d09ad35c7c2d528d26b30f602e7883e5018007a4d6109c18d3020d3c382", 279840),
    "29720": ("4af2ab447a764814d215712afcbfb12b42de8e9ba29a1aa9b02ef1550240d2d3", 188421),
    "29721": ("574d2175f929da7be1a8d1c2849939be185945aa0565670d8036854e553dd718", 504332),
    "29722": ("88136aa35f1b1bf1ba2caf69fd12c466364712822bace72e5d188f7f0b41ce0b", 234812),
    "29723": ("91f501ed03e17b9860515fb8602956a17d569f927da5b4f61228116b5e282578", 22300),
    "29724": ("81db3e3d3276b80cc910cd207bbe43fe347cd300f52e32f6659cd965590716a7", 173636),
    "29725": ("8901239345b4524db707c92fecb542042d2c44c293ba35efb6b86c60d1feceeb", 267482),
    "29726": ("917694384bbd0debc698d56d6cdd38d18e9a6b4b8d1adef6faf2f4dd3858f83a", 227891),
    "29727": ("864c8141cca16977f599cdcef0d4e1313b96d51044c116f765c7bf226b08468a", 289964),
    "29728": ("2ea6ece248a3fbb0b7a8c0a2143daf36a429c4934b8ee72be014c95a8cecd868", 407475),
    "29729": ("480630ab0b3bc6ebade59d97f2c0f37a99df9b76a88dcf699c22cbfa6b5098eb", 525377),
    "29730": ("ca6c42ddd9afe5d55fdd8f47ea6feb26ba4a5a6600afdf6130b212b352aaf9b6", 96112),
    "29731": ("1cff0ee25e06c7c68ca657cca15d383cc831bcb4734577611afc676eb264d393", 325968),
    "29732": ("f44b95a2c2e643066d75410a7d860d2c12fcd7a890e06d4f9a226044554ceee7", 416985),
    "29733": ("7d72c78e4ca19ac4b71455aa5c7445f1b4f46ceced6580ee57f63345514d7ea4", 456935),
    "29734": ("abee412faa3a0c7c9ec64a65ee533cc717ff5d5aba34532fa1a71c091693dd15", 364966),
    "29735": ("041c0e34e313709ed82a7183eb122e04f588379d70e1688ce4f0fae78748bdca", 5674),
    "29736": ("3837381d2a37dca6c1aff7916a20108ab791a3c609c25adcaaac0ddc4eaed079", 254531),
    "29737": ("a4f9e99838d4cae4fedd6c7327f1f056d0c766ad3b71b3aafd13f1aa2ebd633e", 279112),
    "29738": ("e437e827d89a0e7f948b1130c59e14498b48d794542e6a290f156cdc75e6c8a2", 438833),
    "29739": ("eca903ff72aaa2e6d28f8ba09ebdc11a8870c04ba69d358e4bbdd9c9295cf2e5", 462594),
    "29740": ("aa23104e6d3451495da8f4c4b4697e14195ce91f534b05a07e16df13abca50a8", 209903),
    "29741": ("66b435428d6f56531272b5126d8253d8bf3a5dcebbbe2fede4192a492d531e1a", 396402),
    "29742": ("e06a6f405a309164c69e5b1efe16308cc6a73dd730156f25bf6dcccdde0d40ac", 291934),
    "29743": ("5f500f75a9b89f5046bfe71fac33d3166198f0f54e4fcc8783fbf210c06786cf", 199415),
    "29744": ("370e3f0dfb6fcb8ca4eb460a3f3bbf1a88e6b49ed6e12080043504c1bb583e75", 94905),
    "29745": ("90bc34c278369315792bbdd555b6a28b59c2c05577268cfa8c06fa62634dd61c", 278846),
    "29746": ("ddafbfff05e0bbf6fd951fa47ad23e82221e4bf967f8257f33c0608dd276eecf", 476441),
    "29747": ("fee198facb8b7c32f51d0a71dfedce1b9e5774cfc53292805d46ad07fbcc4c09", 240146),
    "29748": ("d45aeb61c6b1325d67d9e4bae1306b41757e2b6603df737fa4e706aac4c72717", 437285),
    "29749": ("d8d1bea0ad92686a2a81cb9e64b176fcbb42359764037a08e490639ebc0e9101", 378157),
    "29750": ("7c0104f71e089e9f6c3deec5931cc9914ee5cab5e20dadc521d6ab6a9f01d8dc", 153136),
    "29751": ("8110503bbb60bcd584dbb654257501dd629d2cb67ac0c5efba506ef359fd263e", 198878),
    "29752": ("06660e9b67661ddccd6e5f6e29c33b3f7ed277438beb2fe535dac8a4dbc95d75", 178611),
    "29753": ("70d821652187f89d4722c35f2a6fe242ce10fd5ba19d0a20659eb712cc36f7e0", 223117),
    "29754": ("ce8741810135d13631f820b05c6236b5cc59a8c4b32ceb15d954b6c903ed10db", 388709),
    "29755": ("4e48a4ccf5778b08f72e880d1ffcc75c55754e959793b449749f492a4b861fdd", 25809),
    "29756": ("3e8c187144179fe615c4d65f9552796df6db854bbaf452ee04bf2b79a6b46f64", 249967),
    "29757": ("d0c5c9c2841255304d132cfcd21dff3f46a4a1fd949cb46d13290b078c3fc6ad", 404469),
    "29758": ("c8da6129ba6741fe9ab3aec3d1f2a98295a1a3ac0e39e681e0f4c959c50ba57c", 557828),
    "29759": ("ff17439b1a4254ec0ac56e56eeb3a4a79efdab180d3aacb719cada411a86db88", 189077),
    "29760": ("a35bfb5aab5ccda3e08664a1144afc6fe4aacd8d50dadfaad1299176cdba2b81", 41846),
    "29761": ("b605f111974843799b6893c7a676edacedb9c4605a22e9b4bb2481df1685385d", 145351),
    "29762": ("2551862d131925ecc320ca175eb500a6b5f14f034689494521b8f4ad0788d100", 490759),
    "29763": ("a5a65eaa03b4ac5be3d19393de923a3a75a47c0b91e24286ae672d2c8d72410e", 128025),
    "29764": ("4899ac930c06390744285b8eb9d4561e9fdfed9600ff2282d8e88662dd24de57", 299517),
    "29765": ("de3aecd40c5e88d2631d4b6579a6bb05cc3d17a0ca78e4c14b731ce72da9aa7a", 49944),
    "29766": ("8e8192b45a95a651037fe46ba0a376aefc8c416fc613273b451441ddd77f16d8", 10213),
    "29767": ("e1549a8a8a4df8a7adde1d7eda4267141cfea21d500c59585e83633e3c72bfdc", 164821),
    "29768": ("d0df77108a4e1b4c6795ecc2a20b88b278fd2eb1bb1cb2832634ffe212c1601d", 44980),
    "29769": ("1272ee672e32c91ad19f9ccc88bcb0cdd669b4f0a0aa59208c47050f27815b46", 131144),
    "29770": ("1b4b19ebee58cfe4c47c5151d605b478cc25e67cd551d2eb5aec7e9a90850745", 393641),
    "29771": ("6caa00c059b3c0328a4548a205b3df0f48c73ea7b7b3a28481a6be5a133ffe29", 313637),
    "29772": ("58dbfcfd30b1430aa729c308e894f854d3b1760517e57e84fe066310db571806", 45750),
    "29773": ("1a88906df09163fda64f2ab6657d17f40a51bd607c45a54d5e7b1df0fc28f269", 377194),
    "29774": ("aa3f4192358443d4c1a1dafa9df61bbed5d0b4a9819f39666981e05959fc47f6", 311439),
    "29775": ("d9e74a559870f3e71306bf4b308e3b191089a53be2537cd1206091c7496485bb", 422151),
    "29776": ("62fba88862012fccdbe5a546af02ede17a8dae4ef5da0530fd8075b11f7a81ca", 170851),
    "29777": ("a25d5bccdef331fb7d7f4069d4aff6bcb3fd2b5978fcbc786cb63dfc69f35a2f", 192365),
    "29778": ("e6c63a4e9ab1d42cc99b828c401b2df40440f4b98017d73be591cacbbe6d15c6", 35073),
    "29779": ("de04222ff62a9c7d608661adb83048bfa480187d9d370be96ce0c868fb986c73", 353379),
    "29780": ("b18fddc7fb6781eee1362d66f7ebb0b46356a7126228c5a423a6dd2a31b0821e", 191319),
    "29781": ("4e5d80215bcee8645c5a2004fb4e9344b98a1a89eef2cea52b5a391c0471ad23", 7013),
    "29782": ("b810bf11ab589b948767fde0ca1fe359a0fd95cdf479ff562de5e1c2ee08ab91", 177385),
    "29783": ("6a76d12335bc9865e3e6a590e0680a348bed174fe77ee88c17f1b87cc23dc74b", 362709),
    "29784": ("a52552b4845de621251adcbef62ed34ff858bc348bb67957d1ee70a4d8af1c63", 643614),
    "29785": ("b5ba4b328f41ce244cae7d121ea793db6628911c5a19c0236b6600da27e808a0", 438950),
    "29786": ("adad10c2b55b075c88840b83a93d7d9d76191560d250ed50f76c5feea72dbebe", 311019),
    "29787": ("5f34951d0ba8669cee13b5041e3829a0d4add8558c35f286ed364b6645e03509", 408758),
    "29788": ("9dc12128ac87915441da9f6c8e77442b0f8cad7c711d1d292cfaea40fced0819", 510613),
    "29789": ("dc8abffe7b476f2c50800a8ce313a1f9b38590fe772094f7c9913fc00582a02a", 114300),
    "29790": ("87a5d02fd6cf49995ebc927b78165f9b094734a75ec67e33e1403c90d0e0a938", 208011),
    "29791": ("3d7157161b22d66f2f13fbf4bee430ccd0f2b96209ec3294bd81718702ea1ea1", 269717),
    "29792": ("a7b2b807b937331c404f47d7cddedc0294598dc31829c00616cf0859faa2fca3", 248637),
    "29793": ("55f462ab11c6bd2984ba207b28bb3991a613d342924020f36898524d22f42eac", 34816),
    "29794": ("13fe979b44d3dabc50acd949923e90fc92c8155fcb5862072a491b4bccb56702", 231414),
    "29795": ("b0b4e84d43bb1126c219207815189746e573ca79fc6dcd974cb919c98dac6114", 28979),
    "29796": ("e1d375fb269a2f7af483e9642006687bbf48b13de6449eba269ed1ea301ead0b", 309491),
    "29797": ("e06bfad920e6141e63b310614a675666ee208e7b107747d7a57b0f52d03b5ecf", 232256),
    "29798": ("100c4e5ef4a5754fd8a396fd91fa953703fbf6ea049d14ffa67212eec2079a84", 525169),
    "29799": ("718923abc7b871ec6a3283107c9433315051f3871aa4b25c04478bd250e2305c", 47642),
    "29800": ("3abbfd82dfc8dc604ce731bcb4edd93e676981bd13f0156a5b72df58c61ecca3", 181879),
    "29801": ("c752d233318384092edd28eb738f8b53a4cf02877847d23d8f8a10d48e0d5fc8", 275995),
    "29802": ("d04441eaa445a65be7c41b9300424b8b0e997424a8f1a2ad6784e55993c9ae1e", 158578),
    "29803": ("9a49b8fc4b1cf9da61adce45cadc6c009fe1711906f5e65ea574031b5586917d", 201658),
    "29804": ("e3b1536cc038ead8f04be37c5cf66eeafa86061ae1e8b05f600965ab32ef5444", 44618),
    "29805": ("1ba19fc1161fa0643d394e7bdf98b574fe26c1a09ae0e6d76856ee5fb47e168e", 217746),
    "29806": ("48760b1f9389f4b2b2b249abf76f748b8a21ff4481a68ad0c0b6861a0f33d379", 290530),
    "29807": ("4bd808167d6c60b90ecf542efe9a62c4805259e606c7b959243fd54a57b5145b", 406162),
    "29808": ("39171d5f029c46839b3b7d823a4acdc93d24170fed93ec77afbdea8ad9bef061", 377411),
    "29809": ("b9839b6409a51d502ff51af56e467b6c782d41df94cc09be6d28a5872d797aa4", 232030),
    "29810": ("f4d37575e2a94b37f18f2fb886d7ebfd545d18ebdc5bac75198d14e8bb562416", 311279),
    "29811": ("3738f8d7b3480ac63a0f58799e2ade217b107fac31f86b242ccc3e1769c2bf50", 36891),
    "29812": ("3d10acf022e0f2d31dec9058c29fbfe6ea27a32d43d69fc48b2aa2b5a8d5e12f", 344695),
    "29813": ("c1c11c48d2dd6aa79dc22b6af2814e9dee43853de682d4cc0b59e913dade0ca9", 338234),
    "29814": ("d6a59d15a9118b09db6e03727a068cdaab917b70ef1c9cc60f1950a7f8701909", 152910),
    "29815": ("4b97dc8d91ba86407d26b6f0dc7bd744203d654f6f9fb0d21204abe357de3cb8", 363993),
    "29816": ("d9a92bc5ae4876c0276bfa66858df931d2016753c22c369d9c617eadee3de400", 341561),
    "29817": ("9a43c390d00d9456a25302b58924e3d9e561247ffcbd1493511625731dc84105", 45291),
    "29818": ("a645d2d4cf8f14dddc8ac101087fa86d48eece80fd787bf489916153af526565", 66253),
    "29819": ("8d84a9fd5e104bda8726dd9438b2bee542560471ede28d829b2ecf12e66f515c", 42837),
    "29820": ("f7f0770e298746744e1a21651265c5cbb41bac1f48138ff6265b27162e495157", 70105),
    "29821": ("cff1b37a175fd949f3335220b0dc3fa7f2ca349a161f48eb7cb8ac5205c82a31", 373794),
    "29822": ("054fa8b6e3edd5441f6310a6fb1239f42a7a58c02eefcfdba5b639d9e52274ff", 300882),
    "29823": ("cc037b4c8f0229735412f689becbcb5aaf8f364dbee0d9ee689bc597e7a90086", 157341),
    "29824": ("c67f671153473749741612116e3face19345176d972fc62c5e6757c12fd6790c", 424572),
    "29825": ("1bd5b519e8a29fba5a2475b65e37a62cf6b1c7910a4a224775e1175ebebf8a00", 34935),
    "29826": ("9f6dfcdd509f2974dffbf152bcdb6104fe0095de2ef3a6f706f222007cb22673", 219081),
    "29827": ("e54d8a9765a3a3191e8bd92fd5a2307993f6ff06cac731c8d264458567ed737b", 151335),
    "29828": ("156754312c694379b1deef15f0403a60c8c28de6da21445d073fa92ba28da370", 135069),
    "29829": ("ce98997aefae475337dac34196db3f9d2120dea9b78d953da4afe91feb7b05f4", 312500),
    "29830": ("c140df5d30319a0c79f5c2eb6a07bb54499385e90a254e332189b243475ae548", 158682),
    "29831": ("e0c3a6d74e62cb1fe9d44d9183f52e77dc48c9cf3258da8981895a326c9c4d68", 124324),
    "29832": ("7dc3e23e9a6051058bf9cb5f3ccf30b8909028811727a89a8230b35695f6049a", 157619),
    "29833": ("a9d972896c339964e8e6ae847b4f0f8ed92927104dc06c00fd52cce46a14c140", 314791),
    "29834": ("bbda9f55ffecdab5e915a9ca121c8896afdd35a439dcb3b979379ba78ebb27ef", 112718),
    "29835": ("0dc449131efa13e262f3f6e53e67a8408c78f4caba1b5ad13de3a3ed92e2979a", 650725),
    "29836": ("6a72975ad1bc60225a0fa4ea2eadec0cd8e136625550d56d6d9c0205b73a7844", 56931),
    "29837": ("c6a9c7f61bf6b08c3d124a9cdcb833f3e63764f4cc9c948b9dc92de3223d8f7c", 9447),
    "29838": ("df029de24901a88df81a5d75a61638c978628fbbe0bdd52aaf795907af86175c", 351697),
    "29839": ("72256894647c036c6edd98503efd475f736eca87274fc6e092f39d78e1ca7ca8", 278412),
    "29840": ("063a7e05c87e9efebdfb31056553e7f8ccb2e3673cb6ad52bbdc5da38fbea535", 458885),
    "29841": ("91d641b2899d01febc290a363b0838e8013ccc1eecc480756fe223a453f8bceb", 415795),
    "29842": ("10a8c31503f56618f0e0f81792722b0cab77e815007a0cc23200e3559bd4e611", 235645),
    "29843": ("b95db2d4e7f4633047b7dd86fcb59e5b0f898ad75a9395257b41797c7831123c", 365666),
    "29844": ("d8d25e573eed3cba58725b7c473e5abdc44c0b09544cf5eb2a33bf724b0f9811", 41258),
    "29845": ("78399ca0a32c0b0e7b66f9ad68848964d7336c9564d7b4aaa88b4d5d3e655625", 31828),
    "29846": ("dd31f05d384e7379f4fa1c0f4a879076b5a25b924dc29836dcf486d2113be8bb", 296024),
    "29847": ("cc8ca7ef3f31dae8782a03e8cffae4de07a5aaea8f5ce359c3a4bd2fc7e26a65", 20607),
    "29848": ("63fccd507b10d898f7c4a1e2fa208c77aac87b8c97a6bcfdf66c041090283c69", 384334),
    "29849": ("9057d56158295da1508c65889db2f5aded71bd40c5c5a94f44d92f072774253f", 203331),
    "29850": ("8c620f36df7d85fa3451c743f1ae33376cfe413cc42e371abc9c95e515a89a90", 13873),
    "29851": ("8da0593ca8d40e5a8323ae8b5b22947557aa12dec00d21552806cafed9d2f15b", 157156),
    "29852": ("312027bd87f112ff69bc5fc4b3ed8640ec3a4c35ba27f9c06e16b54c367c950a", 293040),
    "29853": ("cee16b0f4a24739f8886d93ce33da9b119a149e2f723535f639e65560d8f82e1", 30224),
    "29854": ("e45dc7b39fe592e8fcc056577ae07883fdd291aae485ea61eed99d2378a91651", 202810),
    "29855": ("ea863bf064573d9027395bf66a8823bd4b77850cc41e32a07995036d939f35f4", 155005),
    "29856": ("ebe6fb96e634982d52e1de04ec7a892f650b32f9a898ebf4570a51670c7966cb", 321606),
    "29857": ("39cad3a848dd8770b9f0b811f3fe6f44bea6e8a4dd3f49f11e23a47684f62c8a", 442161),
    "29858": ("ac486c1dd789b23ce83893275d06c940420e49352cc1e0f94679efc45f1e2155", 44835),
    "29859": ("e6ecef0f813ead568921e94c1ab0cae1d65b7ac741652b13da6ce352f9b2a65b", 360024),
    "29860": ("cd1b20b16b1f1750a6094fb7a5295801f267146548f585c795ecdea9cf31d2c5", 112705),
    "29861": ("9547e001963fc5f40799dc525b4ee735abe2c3bed26ff50b14fdd89999d25484", 350187),
    "29862": ("4a95fe7afc5cbb350a2b7ab7aa54e893734e004193253dde6c68df941eff309a", 176608),
    "29863": ("f13482011b51bac98c51333191dc49ca77eddeb57a06401a4c34e41bb29584a3", 21920),
    "29864": ("75f3f8bcbf488a3e68638d2b2f7e42232d9b6938aa96361cdd95fe5ec6073fd3", 399486),
    "29865": ("e0b2882dd26ddd86c2a4a8c1cdc4766979f5fb3528c83091cde8710809a2b260", 181975),
    "29866": ("10b10a831107a71b081d23d0102470397b76fb7ec3451c9e45ffaa872ad20cb3", 387122),
    "29867": ("e581aaddea9210301d461b6fd949296981f2dc2f235501d591ae8749976e9488", 36464),
    "29868": ("519f3f44fd0b1072f358cfeb38c0b0094947b0af68d5bad90411dd0e092bba82", 470945),
    "29869": ("5c27308a717e13f28d752e0a1f37053f7e4361854f4aeb05cebf7819b18b4e08", 38444),
    "29870": ("bd5b2fcb7772118dd3dfb618cc7fb107ab4ea8147a2703aff6778372a276dfab", 255456),
    "29871": ("b20f3dd88d003dda575298fefd59beab1ce7c0bd0169681c5b350df699ddd82a", 136816),
    "29872": ("8854df3cc3d65677dddde8c34abb35be691bfb2cecaed8b4618ca353089e72cb", 26384),
    "29873": ("08d6a427a414fa439976321eebc132893bef9f41fff477ac771f61aba6dd62d8", 343919),
    "29874": ("20203cf3e30e598c5ad2715613bc430630b64789a1e7fdc5538c668a7bc99356", 26686),
    "29875": ("ddce3e2f191ca4029f9d3751aa4cac699e0b906472582399e2594868c6c80a8b", 433027),
    "29876": ("64e08f36a55fa80ac7556753bcd3f687861fdb4ca6951e81433501fa12935785", 146029),
    "29877": ("eae555f3e43c9f2d291c329d0b9be7ada3105ebf8b3df4de2ed82ebdfdde6d8d", 363644),
    "29878": ("2c412687c2ebddc26eac9f9d6ea026cd5a46ad1751e0545e5ca61f4b62904081", 158283),
    "29879": ("f0ae8aacd8aed12f0650dbacfd940fee193eefc974d7e578023851df8e94ff2f", 320358),
    "29880": ("ad69e5243f84affd6874579c4f4e40b96d81c78e3f7e257321b808bb49323a1b", 425143),
    "29881": ("7d6c80fe0530cfc6fd22d9b1d02f60219208ff55f3b052fec9d116fe4493eea3", 355016),
    "29882": ("d7806ec349d8c4032b4cdc768f65dfc47a0cb89bc9f099bebdf301f3df52b9ba", 217695),
    "29883": ("0e54abb0080e25d7fd7b4ec403acfa7d801c96d37d1a45b1e329e2b0e8a77e09", 401414),
    "29884": ("b247ba5a815df0e394d90c67454b8f066019272d64852e8d2d1807a5e806209c", 266379),
    "29885": ("565796d60d5346a57b019f46473ade092378d9a39d968c49456587b721e1e5ae", 45163),
    "29886": ("51e9732afaf804f16a96683010680b532a4660db16a1e30c6d900580739b12ef", 423880),
    "29887": ("3b3685f5932f166e1b5aaf9609ff88d3c8289cae53b06f32463fd90813537333", 343056),
    "29888": ("36ee713e2670edee4ce73d7321fc0bb9486c7754f80b99975515766a71bc4224", 176362),
    "29889": ("1dc949bf65a6df8b790784cb519710a0aece208413875656bb972e4be8aa6f14", 486220),
    "29890": ("3938eb523b39444c900c2d1e579a8f201e43b5e58ff315b6f87567b044ca0397", 256699),
    "29891": ("6e285eea02e05516fad2b8b2416d21c3cfbb99b8a36071c04edff4cc5f01443c", 250683),
    "29892": ("adfa33e978312772159d20f59992451705bac376662a5e82dd2f1ff096bf8b4a", 221409),
    "29893": ("79ceae4e26eac4471eae61b153fa82c041c2b55e3cdb20de30bf8055fb5783c4", 30338),
    "29894": ("80b60ab1b670764121eb7cfe9527b21d742d65fdd60ed8d0a3532708b4b6a411", 171884),
    "29895": ("a6c0d677758a319fad37d82ff4e72ae7cd6c082ee3ce0f5f3c4e070904c87763", 329105),
    "29896": ("c68f5844a7bb9eef2a58349d9aaf413d5e08a5c32bcbf07a6e571e2992201155", 54418),
    "29897": ("7b84e61344bdac82853bd92b078f40a66aafef7feafd616d8aaf982b007d6a3e", 210137),
    "29898": ("e5d9e2bfdc9b834048b27482465de42eaceafbca929ccd8737241ca04322035f", 47292),
    "29899": ("f5daf4e2a9801bc398ab3969e45f2c06e17e4eca27773a233b81e4686af649c2", 193799),
    "29900": ("d37a8cb22ca826b2f93065fdb6197417745b3e4d621f3d63d215c31d52d43b9b", 269812),
    "29901": ("26ae57492fc829277d1c563592546613bb8654f2c316a2864c8995833ead5e9f", 354840),
    "29902": ("d6b51939865c7d6447282d00fff885d64499ea45667e32ec728928ffd34593e6", 308796),
    "29903": ("1001c263bd5f61fc77ba29f23fc5a6110302b915778570f0e56265d0d80bc4bc", 327179),
    "29904": ("be8572affcc124958453fe2dd9563a8dddc3f26c7eefb835c7588edfc0ca4150", 350369),
    "29905": ("e9a7c429b75feb89f00bcbc04890d7ab4ebb1c899fb529cb1bb0dd4fa2058276", 666604),
    "29906": ("50bc6e01f65c13f1ba5be935dabb08b594ce0b7b0792b2da18c1ca58fd31f74c", 315611),
    "29907": ("492557dcffe67f2f77bae127aab297b8fb1377ffe23479ecadfbaab0eff7e842", 50758),
    "29908": ("d3f4ad2154d2e561d904b9b40e60d4170fa5e96ae82c0d17408d20a7cb4e18d8", 391628),
    "29909": ("77cfe3c7ace2dbf5dbc11eb161851a623795d9c8863b812b3a55054bd3e84ca9", 33810),
    "29910": ("773d24c3642af73fd594a94aea0a2279fafd7703e6158f8b57990921fa29a8a6", 143512),
    "29911": ("35f622289ea354b103d06b32faf7f246b4dcda578a750863fd280f1c9ccc2ad0", 588105),
    "29912": ("cda1b5197a0b4c4ae52e294f874d7d58b4134cb7212ae9c322c32e129e661d62", 95025),
    "29913": ("489740b2b6591287ee9c59144611df6d6b6742d2b970c9a7ef02a74b374bf7bd", 369235),
    "29914": ("2fb2b04c3509315246ae70492258bc64d9613623d3792f046e6aae8979625af6", 433789),
    "29915": ("1fea659557baed69f07cba2635d02ebe76d2570a6e9f93aa51daa9d1fc1ad3a0", 356727),
    "29916": ("fb7789e515f353d2de64fae91c73e809c57bc96e4078a8c5e120d384157ed55e", 320303),
    "29917": ("289b1ce17b4ad8c960f1da809285bfc3034d82d674e3dbedda403e48272aac3a", 17564),
    "29918": ("6725fb1e23e0e72cbf9bc450d0dd3cb175c3be1444fda811332bcc5309fe62fc", 408785),
    "29919": ("894dd00881dd040fa05e5074277ee2c34ef55e9eda2d4ad1e903d0d92f6312af", 51338),
    "29920": ("8f05a064c40d7c4d3b81dae0b0b21c9be66f09f035e8e96c5280e39aba9a2ddb", 340034),
    "29921": ("be1b76828fb49acac70aa634ee09b59ab78a22a4f357df655aa19d5fd976cc84", 216071),
    "29922": ("6d39c79bb7902a6dfebf79d080386e15c9af4be9bde68ed123b296d923af1309", 314600),
    "29923": ("588cbf95b3306e3cad45bd1a933cf0f23b1e2d7090c4db2b73e788bff43c3d06", 228564),
    "29924": ("7124eab0c351c49769c9aadbc371b5554b1baba046e175d5e5f877de7606d1fb", 193119),
    "29925": ("c211e7abdc83cf9ddaabddb5905147620805b378d4429b70cef0d9398fe2d4e7", 394759),
    "29926": ("76b3d18ab768faba062faf82e940fd69f844551eed2e8eb9db03cc4f98ba1ec6", 28284),
    "29927": ("f9e9947c7ec95d28dcebcc29f80bee61463ef61f5ecf77de5037c8ef3eec7f42", 14048),
    "29928": ("881864d6d1bde3ab2934d1ab8f66cac3d206d5229f9d7acd4fb4a7748fb69018", 215683),
    "29929": ("607b254ff8a6b18152f7a13db476ed0a969ffca5c06c20f21c47b9f2d6c78640", 251318),
    "29930": ("25b665a789ea24b1764b20e1c10bd37a86f5cb5b7ba9e31ab9c3493c003c2c8a", 255222),
    "29931": ("a8cd3f2809c4a73180b9e182408e22cfd68ca66f0ad343f366458f3595a91d37", 88801),
    "29932": ("a81e1534adbdcbda38a5b33998387d1d09d3780be37a521e80be6feb25a80de6", 178374),
    "29933": ("a427850dc4e04ae512ac8f643c8836e839e61124c20bb47c18156fc332b27cad", 438440),
    "29934": ("d50b65f99b28f2a37a875ec5bd2681c00a593d469c6cde01959b9c84acf5f7b4", 102557),
    "29935": ("61e6a46b4196e4ca8d4025c498802eb07bcf2e3b13e65acc8444c301cb1b1b13", 271692),
    "29936": ("5e1c02e8a715c4fab1316409550d402be74ef05e371bf2b3c75b8ad85f9ab4e7", 45024),
    "29937": ("9c7ebeb9cba5790c1a90edb4f6fb713db68709c9fcfdfa934454bda0760744db", 88587),
    "29938": ("513bcef293345224d54b67d964b7ae9b4227a4b1e06dc2d916b0144103bff839", 19210),
    "29939": ("613d13d83aa482adda4e812c2cf68234d2f3ff749589bc50a32603f181fafa7b", 349382),
    "29940": ("fc4c8864e9f88b01206fc2b83b1eeb6ba4c60a2146d5c26c1caf1bb8c85f54b1", 294794),
    "29941": ("b301240f69917690ab11066031fafe6021e9c837b7bd9bad913dadb97e239cf8", 428491),
    "29942": ("d4fa60eae71db77cde7e1adefb908ff26dd2ef4b64f840d5f6cd2d17ad3ff19e", 472554),
    "29943": ("e83a270164e3f37761adf3bc12362e9d1d5a8a4389f666233a967f1b28a8112a", 509787),
    "29944": ("92f6e9353634b2eb411affb00445dd4c7bbdbe3d83eb40fe777c5d387dfa9d31", 234212),
    "29945": ("ce237c3bec5cc7cf2bc16fbdcaa9b35e35267f350698c338fee6dcaa0a3374ed", 144498),
    "29946": ("f41d3c5b282d309860be51d9dc6f69c26eaae73bb7c042428b006e8fde8f536a", 213113),
    "29947": ("1c2725952f9119319b19403a3208a409925d97e077caae3fa6e4856ad2360280", 189683),
    "29948": ("83ec0cda3a11e39eb7ac9b326c5f5315e22c246417d7767ab9dfb49d71c19934", 329393),
    "29949": ("201e9f182576d6050a29a8a3a7604a4c8075b38d1dcd288221284cdce073fe47", 250680),
    "29950": ("d6acd6861639dfda3e06c85ad0cd8991548d18216ca8f062b425c2f64962a701", 215597),
    "29951": ("ba278857bebb0431a3423a63e77430ae6df6e13eab15fef61edbd06152a5c64b", 162039),
    "29952": ("44e808a799fbd557410d8abe27d5fc09a3fb21b80313567ce5769c95d027355a", 169532),
    "29953": ("f8895274b0ce60e90dc0ef04e43d343376a25bd37690812c1af2127ed5b3dd58", 180269),
    "29954": ("657c4225719cc8c4114db0629c7e572daa533e9081934a697fd6461a7630f1ec", 354907),
    "29955": ("7ec84d11b3ff20f52cb0e67bfc1e95961e06a673614e641b37425ec1945900d4", 238628),
    "29956": ("340f80df2b31c41399f65ec4a63fe352942604c61472537b2018e296e5d513f1", 59146),
    "29957": ("3c3eaae89011a58996420067f8e9d54ef3add5953e02b7946fc853281d849c79", 38775),
    "29958": ("0631ba0a1efca02b1ab476450e9fe8a9694a56d7c15683a9e60d11609821782b", 625637),
    "29959": ("72cc08bf673ebf2ed661130114b9c68f2e118bb9f09119415288a383166a4e76", 250745),
    "29960": ("5e70073f05770966965359cf8ccd0003422734e5f81285401d75539d52e52644", 297808),
    "29961": ("90f12d2e00663cfe64f1d2c8a74b991631eb21867e7c2c743111ba139b154530", 254933),
    "29962": ("6e6e0ea8a9a0c07cda332e0504bda15681305f152df65f3027edca7793202685", 283494),
    "29963": ("6ba4719e1b0c9a9b1ea5445c9864ab7de000f58346e42e200a2caf70b232c2d6", 175496),
    "29964": ("27954bbf156be5dc3c555484c4baa405f31a4b5b57ed33a7ecce330f33a060c0", 39077),
    "29965": ("857520da4ed5efa089022beee3bf9c606185d2a8f332efb681344f26930ba035", 11193),
    "29966": ("dcfc5bed185a039c4c1d5a2df4bab33aa671cf147f6c31c209be737432b886e0", 489420),
}


# ---- Fresh test (review T64-M1; owner ruling 2026-10-17; docs/learning-rate-grid-preregistration.md) ----
# Row group 2 of the same pinned shard: never fetched or scored by any fleet row. The shard's footer (read
# 2026-10-17) holds row groups of [336, 336, 336, 336, 206] rows; row group 2 carries ids 30303..30638, 324 of
# them with a caption.
SHARD_ROW_GROUP_ROWS = (336, 336, 336, 336, 206)
FRESH_TEST_ROW_GROUP = 2
# image id -> (sha256, bytes) of the JPEG carried by row group 2 of the pinned shard (336 photographs).
FRESH_IMAGE_PINS: dict[str, tuple[str, int]] = {
    "30303": ("0093c8a6a1e18d53ae7d1051a34e84ac60dd582a1784ee0a6ee209fe5a115f8d", 363918),
    "30304": ("75fcc404643cf355464033cb1c90dba619e3ead8d181f4f18752721a18f84c64", 349346),
    "30305": ("c7003e37a0f102e467396341466bdf75585b10ecdbad0d72043085b77ab30e53", 275669),
    "30306": ("fe3d57293cf1ea4699eadf1335d1450358c69ef424a6290862c9134c15d4cce2", 356910),
    "30307": ("9ca93456c29282eb97aa12084f40e301a5c310a01baa7074e51b06610abc23f0", 375702),
    "30308": ("f166b8334b3b1404a49eb21e4b0560cdc24f91f98e70dd8b35fb9517702b3a3e", 214162),
    "30309": ("d4305693a7593d1098118f8728438624d67dc4eff5b6404778edde01eea639c4", 307679),
    "30310": ("106640aa5713e828304cfe95248e560740cf0550ed218f24f04c2a4d04ceb4ba", 39507),
    "30311": ("f519302a12e8d9472177d67ee20813a6547b7d8a37b99881305d04db56d07c02", 25999),
    "30312": ("00a618348c797580df7441e518690bbf81be031ceedb815c92b4540203aaf22c", 63593),
    "30313": ("398fce37cabbec0e05bed1d151ee21b2b3a5bd4b0a18b65ae2c06aaad0c03ee2", 232723),
    "30314": ("a7c0029433cb179dbe7cf8f884d2be84c29ffd4c68ef835849cfdd98375b58de", 118487),
    "30315": ("d3bf0f06996b0386b67e25272b6865f9ed8db1f4641cd14079fd0937d5ae9de3", 224334),
    "30316": ("a0917eafc47cd134a12d5a7ee3e532d0b1c445c57759942205b564eac05b02f7", 374615),
    "30317": ("c20e9471895a61d40800ef07a10168a425c949e5ffd3e9797ee8255f31369d56", 15068),
    "30318": ("0471173330955805e4f2fd6719c4d36b2e1dd7c65e7a77616fb5307f86158d81", 391058),
    "30319": ("2ae6880078b0a89813270e3ec3e1f076879a74030f28ce5f7c3a84b715b3fec4", 293087),
    "30320": ("068abd68430624b8ea71fb398060f0543f74f405ae8fded03eebab904d817004", 135715),
    "30321": ("e9713ece77731de4d315e9220f74c1f9a370c090ec3e1fb7f545006ea69d093b", 254225),
    "30322": ("23a8900aefe924df388de2840414e219cf8553cfc2849781ef5806627d816d62", 272396),
    "30323": ("a4cb968f3e8441703ee99e95c1b7a6555e57a60edb0f2f4ea6c3e6a6d0d250c0", 20062),
    "30324": ("4e0f3c0232ca2352bf83b95bb9eaecafbe26b7d9f271ab3906576e94badb5765", 170220),
    "30325": ("36236a0406530591dc889e807cf3bc228a3c1dda4541e7df4c1003a8ea6cc275", 500259),
    "30326": ("726f990571582378e97d9a80cd91d99f9674af3d35fa0c37a7cd9baeef277d15", 37127),
    "30327": ("170234c3c6f39e0454a3578d6fa9ddb1d614aa6005f357220f82677b7a9f4ba1", 277058),
    "30328": ("475d7a2f0646ad252a506cf59583a607484837e251c729a970a66ec9014971c5", 123529),
    "30329": ("59ec9ef3ed667318bf0a5d5f8b8db1aed27648f12e5b2a5ab494e20d4d7e9311", 182021),
    "30330": ("a38102d4176b7519b344bf0c786730ceb7c5a493470fa207a6700104809bc936", 292797),
    "30331": ("841279c816d9afb0ad51d6575fc4e022f92096c773b4aee96ffc6fe84244e4a5", 1153265),
    "30332": ("f0da692777355cc0a01560f9a86e3f2703c2c7ff5b12e6798ceb2b2828de7209", 346431),
    "30333": ("56e3fda30b8e7c57415d7fde43b429d6c57a099c82e11d5cb40f7f07c6db7139", 155291),
    "30334": ("1ea5199ef90f9f0fbfd516131489597d9f85698b0a5e3bd654beaa6dbbfac995", 137396),
    "30335": ("716f30096ee1951a6edd20e009a4a8c031381944a8e226d2da4e487f4b9311a9", 201909),
    "30336": ("e427a6ebbb21d0078147cceae3debed3fcf76d76794e7c6d57d868cb4aa479a8", 29055),
    "30337": ("ebd62784508ba7241ec2db7614583a82856c74059c27541b8c5a515244f34482", 324349),
    "30338": ("893e69607842306a582d0ad2248b72ba54d767b1f656f9220646bb27e8ff4e28", 268245),
    "30339": ("cd3b7f8e68affb375286b8e6a73b129d7606d037d52c326031590bd92622cbe2", 181954),
    "30340": ("643b468ac904fca26b3ac2c0796dfa3a691977d6e4bccec667a9acc40804e187", 54280),
    "30341": ("c76c48193ca42a14a9161876eb9131668719bfdba1dfa4340234a3b5be3aae82", 239123),
    "30342": ("6453636be093cbfee0af6a8030adb083fc4cf2efa49a7acece1513ba7b94a8e8", 301472),
    "30343": ("7667ca04ed1748192454a9cf3d8cc52f09faa01c920cf1ba2e2f6159d7546cc7", 35790),
    "30344": ("b915859e10954c811d26875c2d622263d47578e3a34c97ad89d0de0a767b6c80", 298987),
    "30345": ("1e264206f841d6de6c582bf6faeaf74a5a7b1347f849ec04abb077f1106aff25", 377352),
    "30346": ("3a843f134e36f195993df2855823f5aee042115cd51ab24249f666c06987f0bc", 51692),
    "30347": ("524fa9e2192d0d104f713c305fee0678de8fb067d20a9ecec350b8189b0d2d48", 278729),
    "30348": ("86eff871f9a862346b26b63f702a922348205c0548e36376fb363d55cd284640", 217057),
    "30349": ("24040b946c1fd25229ee079e65cbc6f8efab900067f5c7a2f21fe3918293ebb6", 303346),
    "30350": ("eda1d5ba24a093843b50de4398f58aa3390b12ba0cc633f1416235fe06495561", 240111),
    "30351": ("0a472b3d24780c5ee770bc765ddbe71d3c63d11677f33454dc7e2ac14cfb50d5", 320411),
    "30352": ("b03212a4f45e331b170e1f895d860f57a3926c08f8e6b36d6c99902569486f63", 46485),
    "30353": ("e846c2d0c12aca6da0a7dd4340f21b233c94bc09bab925be54db740fb07cda99", 31562),
    "30354": ("f1f5a8f896d53b11e4282d2feaf7639eb8c8333db1757cf6d9bb8101c90d32f0", 372725),
    "30355": ("f0ad80363d61686685d4a8936dfd89ac15ffdc6ffd20075f33154ee161b4d18a", 504215),
    "30356": ("e1affa279ab2911dd00296e781313e0cc865534810d2103b766ecf112c774061", 205292),
    "30357": ("6414997fc45a652fda7c59c8dc077befd491a84e5076a04ef1a0c2ed99ec6fe4", 47422),
    "30358": ("41bfda35579ee949c11baea78c9015fbbf00b2248bdd905ec797e577e824e402", 150518),
    "30359": ("a8fba8f3c01ff61818799c7c7d105a74e49883452a5fa526a6bd8c2826977076", 284257),
    "30360": ("98c6947971534cf76a8e6628abc3090f55a85198ce2f79c8947c2a383969cd5f", 247637),
    "30361": ("27f60ee66b14b058c782013985d68e1e0fe034a23b6d909e43c513aff89f93c9", 285116),
    "30362": ("f5f3675612b3347820c6b30aa1d09622f70d62a5e5eac7ca0e5a1f71c9e91a27", 169531),
    "30363": ("bcbc1c53ee6b13883e60a496ae2ea8cccd9328c1f823b33d1f280e31ee11fffa", 121698),
    "30364": ("3c471621e20f7364cb763958322a8a7e684ea13d258952487e8e5fb1e0afaf32", 294754),
    "30365": ("c3ca918615a7521b579ed0dbcb170b7a5939bc66e3d2c7875907b8093de2d410", 226511),
    "30366": ("4903d02f0adfb48b7c05a66902e6336c6b2d9eb6a4109fa7557d8f37182aa927", 57784),
    "30367": ("e53c60d92412e4f84b277cf6d732ba9ae50a8dee6198b4ac61d749cfde7ea610", 465836),
    "30368": ("7eba7992add37e23cdd36f865b2c488cfde854b661ae4d524c95b3533e53fe19", 29822),
    "30369": ("93ea00b0f4a0f339900fc1ffbdb564853329bbe2a324a63402465b0043375807", 367605),
    "30370": ("f2adde807fdc14b96bc8011dc95a36fc6eaa23e8cbbf7701188da78d33a3beff", 255826),
    "30371": ("f329d32e8f2da83289dd7ebe88817760305efd70ad045eb6584ad99e7ef863fc", 265175),
    "30372": ("dacb99a4543a07cba5627efcfebe4e357f451efe90373f9ae9866bf7ef456c25", 103915),
    "30373": ("a1c681213689e20997d89f961b315bf7de84f3bf56634c459a46d0810fdeaaa6", 82947),
    "30374": ("3e0e54b4466893b753796e3fc16f9b2f99a44ed254f987ae1ca23f80a92920e7", 370020),
    "30375": ("a06897fafe540e36244a13d5ef38fe58efc8acd4bf76ea6dfbb0d28a83ddd8a3", 268419),
    "30376": ("4b3f1faed55e4a5b025c4493c50414f5d0e88ba1679c242f79d6034961ce552b", 242669),
    "30377": ("0313baedad16a67f35c43f17dc74f622dc1dcaa5738ab7456beb05a0fcb2baf5", 320999),
    "30378": ("f86e00f45e76fc5c2839762a2283d1bd3f7119c5821be5df0b8f188fe1b1830e", 363581),
    "30379": ("c70274a25e44a3e22bf69e789c7d6fc17b7f421fe189a09cd86a23215c78949e", 167291),
    "30380": ("3727dfd4552d6ca5b17c1c9333b80b5eaede49b700f2db21cdae399869c56792", 148506),
    "30381": ("68a39e3528bae34571bb73f45bdc35c45f6c61d8a4050f001b2f307738bc3468", 260473),
    "30382": ("d125dedcfe0b66cbbed81306aab502396ecf1dc8009edde9b8050996fa0e552a", 303255),
    "30383": ("5e484a3c7794843d68c7ee3c402c4b0d1bf1d1c4f5e3ee3a4f4dd6ed8e69982b", 150779),
    "30384": ("5fedfdce234787570d48eff026f510b78c9bc3e05604c5f3baa8927ae28c0966", 219152),
    "30385": ("7a98958c9b24a0c6f85ea98afaf592df67eeb0bafcb9d4d86a66a89f3b47e0d2", 304454),
    "30386": ("67ac3c4da08fb99c80c1f98fd0787f6a12b6c0a87595a5835e96206a904cc5d9", 340868),
    "30387": ("510a363c905a2c1fd1f19497fb23bad18fa92eee21398260f3919bfc4b3ca739", 394723),
    "30388": ("b1a37ea1f3da64283972a091fc0d9301ac6557b661e660b9eb55117973b0300d", 192018),
    "30389": ("0e2521bd9487e9707fa29bcd950c764b917e3dca994e56ef3dc9b2b6831dcb78", 124738),
    "30390": ("466c2845d888a1081a00b036b818658aa0dbcaf5a4f0c97dddfecbe5f51030d5", 237675),
    "30391": ("5833aeb2c1f6749f178fdff432cb0cb4760eeefed826cb6c53af231703f88416", 503799),
    "30392": ("4396c9094fe2a0e725d79ade0f31ab15abad137a52c462c481b81a45036c8840", 353606),
    "30393": ("ee473e4f061d1e34bdbe67b3a35c32459c391cfc68c3a8ce901be5c3aa609dbe", 266576),
    "30394": ("e1892662cac27cc522d60066c9687aa94c7fa2f37bd2d574336b699a45bd796b", 457875),
    "30395": ("a8ad68913896698375aa46c211c898c7996394cfa7aeb7f539396a4471efdaf3", 139201),
    "30396": ("e9ff45daed388254eae4956f9ff6347b7bde469f90d16f4b89f0046fd57a8978", 282745),
    "30397": ("e8b96d524ff4c396233b6743c57fb3bede489bc75a6752d560c0f4f3ac8ddc8c", 212289),
    "30398": ("6e0c520c00bbea936ef6b66fb60cda35fd9b856815bb1756beb53032cb119190", 368737),
    "30399": ("ecb791a0e0a060e04a8559016524cca5d3e1e57388e1027d255a3ec9d0c7192c", 368659),
    "30400": ("fb99d26cd94374a445fb0eaaf33e460c55fabd4f4d92f502bb76e6e96041d113", 270170),
    "30401": ("ac76b6b71181f72aa0f9cdabc1f82f7b8dbdf74fc0bda2466239c7280e6c5e5f", 195130),
    "30402": ("bdb6e551f3fcde9add9846006e67b8e1cf08ba88bb9f544e595fd4d9018c8671", 10183),
    "30403": ("bd45194b05401622e818ca929e459a13d838ede2dd0b46070b70fb68290f507a", 268846),
    "30404": ("0f0e99ca092b7e3ae8bb208199aa5aa6102b6247215aea60e13084e233a8d0ca", 195589),
    "30405": ("17b09e73601db67c7f95caf1a4072422c2ed05fd5d7dfb3ae6b40534d61e56b0", 33826),
    "30406": ("75b03ed5d1ab4b58d9222754d72319f4cb39454dece8c661486d0b494225902d", 39527),
    "30407": ("06fad9b688b057ad84d6f35888749f4fd4cc34ab8a5f9f165d1a4024f67ea4c2", 125439),
    "30408": ("1d95f758c0b8641795822ba2df5983c2cb345124437eaee22fc3596baca45a3c", 243627),
    "30409": ("8eb822b84f3df5d00684f2fc1170f005130f21faf74afa7b6cfc85e16bb97377", 223188),
    "30410": ("83b638b8a6a2173a3ba7a2c6a3570904cb6ca06c46770d168a89624c8c0b2b8b", 436692),
    "30411": ("f317c24d9418a888603fd30d62fef8a221d3b6f7faec5d194a91fc9a06bdac35", 178862),
    "30412": ("0624c9c467ad556b31c687d40d56354f5c37a89120fb2fb791d7bfb52e084a18", 463097),
    "30413": ("99cadb0126d1f019e870dfee836c7190dbddbb928235179a2d1f3509b5bfe584", 128711),
    "30414": ("7242431ca9b3f0839599b44b87c71250fb0e91a4a557c6d1e6728b6377822d7b", 268911),
    "30415": ("d1822b1e7de49e645aac345a28d62157c4fa60ebf4b908e3874a5696b9495406", 288946),
    "30416": ("e5bb1240bf83c3cc08901758f5bc2ac42eacb14d71349cb517c4d111e891e14e", 239872),
    "30417": ("10933aab6302380f2e5e8266fc0a5314b0a688f6bb5c574dae688a53b4d6c3f7", 279411),
    "30418": ("56d13913a81ab9f20b1c30da1fac7d7c84ed0844137f0cf21958caf703fd6f38", 182793),
    "30419": ("784292d82adf75363ac199eb5b7b7aa394e6e9b02eaf61438646dbc6102c2ec6", 284974),
    "30420": ("979246ca65aa759f2bb864209cab1b165904e5b2b9cc45c8e5f817e35334f69d", 414833),
    "30421": ("bb2bdd5cdee0d6b41483e57a9684aaccf75ae0ead791691adaffd42d46efb2be", 221575),
    "30422": ("f7bc26fb911aeb51f1399b7850d07864380db175480d75227cec7bfe46437188", 270050),
    "30423": ("21cf365ad51f2efd445bfad85ce7983c75ea44fb115ea047abc351823783407a", 264261),
    "30424": ("0cd108384dd52de699b88992719fcd53e9289c779da580fdd8bbbb7866e9259b", 244740),
    "30425": ("2eb59c774a3f90e2ac56482299f1dd3bc864e3c4572e8183cb0607013c380db0", 417832),
    "30426": ("d7fcfd574dbb5abd33828c6f8a0c8a88fe3676bbd140396a1a5ebcb08bb82c21", 380835),
    "30427": ("8770de31afb21cf5ef5426f8ccca1fda6d88171190dd3b874e392ec6dfbb7a7c", 464067),
    "30428": ("8ac8fa0ee2e23551b452f31663898e5350bf1dfb37b57fb27da6a7830ae2e1ac", 123403),
    "30429": ("adb76372342dfdf41a637e0d1ca0aba88045ec7dbcd2e50f3167695e797ed685", 307998),
    "30430": ("8b371dd869ea0fb69abbda7ccf95840c8d1ddc6c6d2f04cf911af40678a938c8", 245637),
    "30431": ("b966b75ca0e7f305928e1ec2638545c6ea136f34119c9e86db0e0e57f5078380", 572745),
    "30432": ("f0fa801524c4996c8150b28949ac23b2a3cdce9a3149037017eb19e808b95f0f", 234588),
    "30433": ("21c273c629fac36d78c2683ecd3ff75621d69a401162585d3f6a58eac8536cfd", 328606),
    "30434": ("280b894c063d7e7aefafa6c8b0c05a31e16c614ce39210f83310a7b48acc5eeb", 239336),
    "30435": ("31c3fa987585b86d8f36118a26070537cf9c44c640adfc1f50dbd9df5ac6ed36", 447165),
    "30436": ("9aa79d1360846135367527d6d7f05ea1924a81e353e7795bddf331da7778c1cd", 172588),
    "30437": ("ba7b80be02d9c630d2094845a751a8775e7bdebba02d4c56f0471bb3547540b3", 297003),
    "30438": ("3703128788083408720b55255b8908cd0d8f03dc462e15acb89f97f01890d5a1", 247316),
    "30439": ("26dd1a5529af6faf0a782d7c54d331dbe07e4c72d68dbb9dcf4110087b702b9a", 446670),
    "30440": ("6e34105d41fa8d1326cb3496d537b98be1f11449f2dd121dcb160d2022e6947b", 30791),
    "30441": ("509cd52d3631db5a0dac60f6b4186b0dd3a11ddc420aa1f2b69995b265611c7c", 282882),
    "30442": ("313512c584c57b9ba10e00158f5b7052bfba431a3ca959fa14668e2f8871f7b9", 360456),
    "30443": ("56b7018d07550c2ee3531e2b118ca92f55e697e4b40701c5e6ca5f337c1a4396", 279603),
    "30444": ("acc92a3ea839c739629f5bc62cd80516241113ab9d8f9c407e21f4fe3b0bd1a6", 414682),
    "30445": ("d6f51124683464431174f172a55e95ed29618f34f496a5fec50b6bd6935b23e5", 197271),
    "30446": ("8a4e9a6828e9eea81025c85b41978001c4d4c4104f9cdd1b19c5dbc99e8f0156", 24560),
    "30447": ("bc1c8bc0dc56c8c3c6b89be8b18f9509b7366b6d22d227acc4ebe82b2190eb80", 158539),
    "30448": ("d6b69e7a4616d9cae3c04bde56e7a8892939f0790ff9ca0772b60ea1c2807914", 283471),
    "30449": ("a9c3e3b741a8c6373337f2d6fb9e28941de850b325add038f100950e82c2e214", 172305),
    "30450": ("fe85bef1b77ac63f55ae656c99f785f079b790a90c237b0f697f4f9bc4593633", 35170),
    "30451": ("ea86b754028cac2a9f02bce44fc977ff9840333906bbc45586db58ffc0760c2c", 324365),
    "30452": ("457684f436edf98615a6404ffdc4698e4a385011e228cee315fa0ecf23999ff0", 41465),
    "30453": ("650343b662b98d324fb237d55edb3bb789687344c0973f97b43506b884f4d447", 206610),
    "30454": ("3c147f0c3e4bd03cc3d268923489c66d7b16a95f5011596e491c075c7a80f819", 364202),
    "30455": ("eadd8ae90ef64771a46c80c47d7e92cf0dc809ac190b76cab856073c855d7942", 42262),
    "30456": ("5efb9525ff7d9488bab6c090e0a8ffe65ea20655dbcc153f882a2841dc125ea3", 467744),
    "30457": ("b592ff51d03732edeaba4401f1a1f70a1bd5741124583ddbb11497cfbbfa61b2", 111017),
    "30458": ("b6f65f4ea61f65e5e8d63df5d1d8aa13f747e180b5f754598bc3cccd7c52eeb1", 514823),
    "30459": ("efdc5af5cebcaf73a76f7ba352064ff0973fe96c3d8da017acdb22e7c9d20603", 238652),
    "30460": ("8e613767b2123618abf5658bb3791e97bd1881840f4d5de5d4b196e2ab32db9d", 259370),
    "30461": ("4b6655aadfe56ad1c9b08624f5f5e619118557881bd85f29024365f701c5b40f", 19894),
    "30462": ("b22b32199df7e1415443da01c71ad3c8489cb3eba9a0fc524b6b0a46d5fa4d0b", 38505),
    "30463": ("fdde1099232ad4ebf417595d1a47dd734f944c8fe971d6560f53955d92dd61c7", 427014),
    "30464": ("e5d7ebcfe0b929fe493133bc54aed250f0a030be3ee2cfbf5ae5558b18ff4aa5", 564652),
    "30465": ("c5d5016170cbd9ede8e175cb26fa7d89e9df4cc38a281023bed7c84c0cfb249b", 214088),
    "30466": ("e93b3e9b12ab9491fd0b886426bc6d17e48ae77a86f976142be60601c4eea777", 250851),
    "30467": ("4312d3d4750fb015ecbe7b3840a1b5ab0b0d1fae56b61da2377c27b492b39c06", 138720),
    "30468": ("c4ff1639baa9614ff7b37b25bcdd047fe3c019de0beedd3222aac7500a10fa1d", 42382),
    "30469": ("cc8374ef6a70b193c6f4310ae981c46c6cb96ecc4669fc5c167af8400db96ea2", 481185),
    "30470": ("0f594bcce79cdb8b9f5bdc8ccf4adce21d498acba0b4763c9427c7523b057f94", 388976),
    "30471": ("0b22cb0968d749c60d383f0f31aa392b8d841842c600d0e1e0c26aa344d8a02c", 270227),
    "30472": ("dc43267820d7d2004a2f97f615ee2273ac6e05903f6bdf46626f8a31700781cc", 252274),
    "30473": ("426eb4458fdd2a362a54c78b1f32d58749431da444599fdb10dd28f5f35f6e79", 299738),
    "30474": ("6b4585ad7065ec0271f8916e10922a03bcb32013fc163e2ebbe751c14e976468", 232720),
    "30475": ("4dba7f2bb3b32881acffa53e080cf47946d83a2044b26c72524984acb4d949b0", 141997),
    "30476": ("47083cadc9c8ce3037c076d5edc448ba19eb999657334f7dba6f082b784211ff", 187872),
    "30477": ("6896dbcc5154be57d163bf6fe4da826f193136f803823232235f1a3b43c9dd64", 42655),
    "30478": ("725e38aba00af493cfe42cfe3855c0010551409d265bf6ab32f208740fe47b10", 48397),
    "30479": ("df1597a0025924c1569f8ac3e59c50ee6ffb9a7f0344db66c1fe0adbf53a33ad", 134725),
    "30480": ("0d2f8b964e8a0bb4a3e4c1976c7f45fb3874e967b9ab4faa5f5bf067ca129b2a", 417238),
    "30481": ("695007fbca479965b23121930263c1ed96a29992c55e292f047136ee00a440bd", 217029),
    "30482": ("99af752f2dbe98e25bdfe0878cb6fddcf5433788fac2713ba6b3dcf1c3a10085", 908385),
    "30483": ("bc9aa62a86a089f88f389449722e951a9c602c2dddec89b3c722b96f3ecb8381", 404110),
    "30484": ("2da4b329ca3d417eee0e831ba319ac5ecd140d96c46f9e99d5b69c14f4ab98c3", 395717),
    "30485": ("876ce0462540c4097ee16600e791861fe04a72130d0579f34668e1e7fc7b6721", 269787),
    "30486": ("6c969965e453ebe8d9c91a09774ca9de1731c9623dacfcc44e074fa34866c823", 441810),
    "30487": ("01b86f03c49c44ad1fced4135094cb32c26c1c4fab10d7004f2b1a89512e6630", 281665),
    "30488": ("2768eb55a2e7e9e06df2a8f69b4b2c1825e68c457d0619698bbbd98877eb9eca", 93684),
    "30489": ("f5fa71be676623f527211918d334bcbc81f4e4160394e84e2578d7522d82aae7", 247484),
    "30490": ("a86b6f4d6547edaff005cf4c083a0a63e45f329f716251801a0e57ff40b11a08", 346301),
    "30491": ("8c6f32eaa368712b1b2eec9a21e54fd2b58a1e3dd28f1b9ebc03435f5589bcfa", 174740),
    "30492": ("d9d1fd19fbb197d15ed6a9d9f5a4f61d60d02b473e8d0f569b5f2d6f9e31dd00", 268724),
    "30493": ("da6e4da10f8571b9cc419252c03d8622a4602bda153b850a8a25ee57a6aa8e21", 358981),
    "30494": ("ba4844c6295c3cadd499904dd2d3ef9d76bbec8b7de5841ecb826d50204b6071", 45909),
    "30495": ("3d232ae15a3dbb5149e222c6d4ffed42a875ae4565ee001e7db4e66476d02c77", 259461),
    "30496": ("57ef5030ab46816392c4780bae5f00c021f24ecb2c67c2a5e31a782d03c5d9c0", 95900),
    "30497": ("7455ffa798c1be0d54e4474a4cb6689e520a91999465040be0eda2baaf79deda", 265829),
    "30498": ("934650a60ed5c23f7c4edc27e3f8088276cbbb3c21fb43a09d7f899756087220", 341388),
    "30499": ("ed125a61556aa628effa95a7fef5b2b02c1640b8c379022cebd5486e3a1ac142", 138489),
    "30500": ("e7757ada27e59e23a9bc407988be919c1195f0e15114fcbeabc723c32ec2f0e2", 33367),
    "30501": ("89a6a9619d544a947a24fee7a989466b8d9120809079b3e4325952e33f7b8595", 217667),
    "30502": ("359671e4539f620ea17afee6f5d5456a72415597746f6509e1c5f75bbe464501", 274761),
    "30503": ("23722a02cce27f2c553552e0322d647b1800d99cebc58b2a542b35a883a35466", 197503),
    "30504": ("7c8a199da71b484cd579884e64149b0fb73f624abf390140642daad0217ed153", 413014),
    "30505": ("a6e301d3674d88b32226500e8f1ca8c36f741d6064ddd40fc1b6acdf185bef24", 445319),
    "30506": ("abfc017c9c530998fccf631bb3d51337ae7d027541e83c88ac8fb91a97680699", 187735),
    "30507": ("fe48b75a47fa3d87030466756757ad55c4756ffa7a6a39f9c8734d1932a3e7ec", 142234),
    "30508": ("89e5672e83537a653d878fe5f0ba00209031cd48d440cfc95aa5f99b3e9183b1", 313596),
    "30509": ("1cc5af1c55375e9097bc33d4c09d93b25636e6baeec320cda1b404a903782d44", 264571),
    "30510": ("a95bb2add0193bc38f3a8668950bf83e3774e40f3a2acf5a10a0896633816a9e", 33578),
    "30511": ("391ebad4a4b8fc4d604067d128c0fe06a416af9fb5ec15232ed8e89be64193e5", 255261),
    "30512": ("5c8861aada630ae60e857285836ea436eca5d922d08b1aa80fc712725d99b11c", 349313),
    "30513": ("79451a927761c62ac98ec83058c1f5a4cebc98f0c062cbee0e9e1a92d62d0e2f", 46278),
    "30514": ("e44e60b94d4009ca1f45a47015b437187b1967152a04800948310428ca54b968", 323745),
    "30515": ("083b237e0ed33001ef3e025041e2a9e197bcac703151e0cd3a81c62a93d00712", 45618),
    "30516": ("612d11ff16b898c0e5ffa05a8373fb075b00bdc448604f89009595d2a62f41fa", 392827),
    "30517": ("c0ae9258011de1e845962aba04713d199bcad2e6db1f9e14579d6b599dffc313", 164482),
    "30518": ("5f290965f952e12ae290cbe06a33aec2ce293049dfb69534e7a902e1c8d8359d", 45601),
    "30519": ("fe118b65d0dd74ecdb8d2b84995c31b0de7975f89aef04cf49282f8f7e7e5388", 330863),
    "30520": ("d877eb8068926faeed289df2531ee6a96fd8ebbaaad2c55dcbbc2d015cf8a03c", 56896),
    "30521": ("45e35cf88f3cd07761d6df6afaa36f960f274ce8fa810fdae0be58a334554033", 101129),
    "30522": ("7dfdcc32fa7c678106adbed7b3a65f6b7674fc1275bbe8248f305830f4bcc01b", 197081),
    "30523": ("c227fe7bf315c8103c3c5d82f2c5f8f42b2836b96176dfce71c59df9f2573956", 156612),
    "30524": ("474f51205a34cf34b988f92f5c6295527ac4fd74f084e17d37b5cb51eca99eb9", 511473),
    "30525": ("3d2c996d01a8394c0b45c012bd887bb0060632570b5d1b0e280ba1537e323716", 52156),
    "30526": ("b04cfeb88f52b54105eb81fd1364d5e7b5907e9806ffa7a04d4ab53a374b4076", 127218),
    "30527": ("2cf29b8b8820ca2ed9c23e3a586be0887bc48a0bba907f7540bec9ef42af9355", 189649),
    "30528": ("ead81f3d07288766ddeb721c8ebe53db5716c4f057a373db43d40bfe4229ba17", 212896),
    "30529": ("ca19930131aea25d16938ea874b9b39e12e85009aa7f675fa217cec28e48198b", 281108),
    "30530": ("adc0666a197234991f8a39162080c9500add1a85ae1bcfc33b13019af0d180cb", 406968),
    "30531": ("26b93ba73def517ab381c8457880e2f51fa91215e2b5582adcc9e56d9628792f", 335963),
    "30532": ("e2117714b8266a1813df537b782aea426b3302db18d23617a644c46c941ee168", 331027),
    "30533": ("a9bf8dc46267ab9e257691a3fe4f71c8d9d460be1f35b813ad4b335a4f3ee2a4", 332362),
    "30534": ("76c6ed17b2ee6f4db465b616cf5f4edd97c0ecccb7e079830bbaeb5605fb5cd5", 298568),
    "30535": ("a358a031ddffde6f5657addae9ca71a682365ac45b1ca415a89b1259521ae73f", 230594),
    "30536": ("6c363441b8a15dc0389d73fa4dedec576dc3fb849828226b067ce099562d0bfc", 397864),
    "30537": ("c1aa7011c9f5bcfeade0fdcc36572e2b3b3356d009e6ad1ec1a9b1707ad73f3a", 640604),
    "30538": ("a7ceac18d5c504aa1e6bd8a5ca003f2fedd4376f56ef281234d4c9fb47d533bf", 133817),
    "30539": ("763e896115b87b2444b6f1e2388d8952146355062069e5a4f209b7226d146a44", 472828),
    "30540": ("f9ec0799f15c9f09df2bbf8b1bfa56b31971134ee82cb21932a566746b9e19b6", 408524),
    "30541": ("9268b86bba7e397ff0a6926e688a9be8d4d86ae9e4bc62a76aae7dba687205b0", 419356),
    "30542": ("dcc5a96d132c9b21f0dbe4f7d17c7d9870fe4bbbc63f568c5190692d31b4c86d", 352371),
    "30543": ("d400d8dbe38e93f8b42dec3fa8da7989026b812838124abc7fb515dd96fa33e3", 313523),
    "30544": ("9083b36c28f3b8115f9cadf152ecfe0e6b38ce27ce347e5d74fef736aa84268d", 194759),
    "30545": ("f3d44384a178ae00515729f8d98ecee71872f2ac650c6d57df12ec3900832cc0", 295380),
    "30546": ("7f86ead2d73ac970775054aafd27812449e073f5f320ed56ca603de08b2c96cd", 311597),
    "30547": ("5558ac2dd3d423be812035a60e94fa9a2f68fe68c3930e8fe9bd14160531c5a7", 332232),
    "30548": ("bab7481b42450bf5fd67b266d0e79c4090c3ddd5802a77b24fb990d655493df1", 244247),
    "30549": ("2ad2b1ad4c0998c22339f112c42ae947ddb73bff63eeb7ad8c7d64ad358db11d", 301679),
    "30550": ("ab33119fca9a99be71d896117c6aa9fc327536cbe8bcd5b09d3dbec1c369668e", 202364),
    "30551": ("7d403d9a65da0de189d5a56a511b25eb8b7ef64a227be3dccb0afeb3af9f2a1c", 200926),
    "30552": ("a3339de5783cc1972df454ed0b18d6fd3f10c82fd985606bc36ff8059b0c2adf", 353769),
    "30553": ("12cb10eb1674f64dc8da4903c971bb63b27a10670d32496f0db17ee20a66fea9", 116336),
    "30554": ("0fe4c0f763d0203db2afef452b457ea2e48f2026d20f378939214c2834069608", 340437),
    "30555": ("16cda594663dc404ee926b1f8f27334ba2848d36e0738ac2d566a8262fa65bcf", 136961),
    "30556": ("4d09244a16cb2f340609b31b6260ce89913f1d22bee6987f36d5e0fda30f9fa6", 362283),
    "30557": ("2bdbc12db4cc9598caec925766f674d028fd5630c39962d2624270eca5a99409", 38839),
    "30558": ("451b2e0f3ef8568e0b9203b58b92c01a28abb97da02cde617d682a0d9a7762ee", 225083),
    "30559": ("7cbb0fc3764d59c8ed1746ecf715bc73a327c66a77373968b8710b0ddde17f51", 250438),
    "30560": ("78459a563dfdde2f86b51d8ccf946a4aaccf137d3aced84b038b905123a54fb2", 229278),
    "30561": ("db9fbeb24193296b526b4b488f937c2bd464563f19b62a5576f31e206a2a2ac5", 251736),
    "30562": ("5c78e819e03c75bfd2aefdc5858ce2d584beb96fd244f92586ae26759ac3d6bc", 357331),
    "30563": ("d6027703f29f91e23ac66e2679b72219d7ab20b57bdde82c5b1fa04b3f8697f1", 238493),
    "30564": ("32a0d62c55d0eb4da9713801a1576ff413f51664781a8e64a380f6320a6d5175", 157914),
    "30565": ("769271a9988e9e6f31a5b7af212e16fdf44d6160921c290c569b010a967ec0f8", 14607),
    "30566": ("f9c799555db89751093fb26fadd79f7c44e9f3e1c4c3f56e7e8f1a74252ba81b", 329098),
    "30567": ("56f0b1b6568e406b3859d014a7e7eab540422774646573de598690add05c38fd", 894641),
    "30568": ("f363159d8e713aff5d868d926cf4dd32195747b202e662cd18e028a814e13fb4", 393176),
    "30569": ("b0d5371ad07875e20f0e6b085eaa7997921f2df168f777b7d9cc46c1d4c0edf0", 38775),
    "30570": ("cc48d7d89f794bf07ea3fc4355c443fe78a3658bfab2cd1be1fa79b1d1cf236a", 299312),
    "30571": ("67421da3417b277487104421f7f2612868f8ccd340408c0eaed0bb1dbeeeb099", 202900),
    "30572": ("266538037b15e2c4ca7c4842c8b11ae6aef81ea271ae6efb60b36f67e4dd7d39", 171336),
    "30573": ("799ea84b2a19322556dc0baf34449494e49bc944d0674c9767989ba9101190e1", 516273),
    "30574": ("bc21c93172bb85ee2aa028f3ef7c5a502c07290448453e3a60f6368ed84a4f9b", 243634),
    "30575": ("02f9909724d4b22046004a296040b0f880da5ec900d19146be9b763d372341d3", 381217),
    "30576": ("15f63e68643c901ab59c02f8e9df13006b89a473da7ddc5f9fba44e8112a8b5f", 179040),
    "30577": ("1a241afe67555e0fc5760b898d01035b8714a45a34d08888d9cd1342bda34bb9", 308170),
    "30578": ("2c20366504269874afe9f1535660308676f6572f5260602621c9ca1a5f9f5af6", 405153),
    "30579": ("0f8eeb6da75b17b52ac18af4cf210e1a4e3c803e616c00fa87074361641268cd", 176616),
    "30580": ("736f82357cc709bfcce38d936383cede3d425b5c7ab44558c746dc30b1c248b9", 20198),
    "30581": ("9466b42b34bff6da75f4b400b2b70241ef531f4f726ea68e52b3a722c8c8d2af", 284076),
    "30582": ("665168d9fdbc28e1c44b25089aad48fbd402d446b9ae2f6b0f234580619a44ca", 229909),
    "30583": ("cfd893175f18622d4fc84774accdca08eaa07101bc1471b29a091add5724a42e", 301940),
    "30584": ("b721da155fd7f05f27b8c5b9c83d8ef7ab6ef967e51577c785c359dd26d0d23f", 36920),
    "30585": ("cc309075a3d395867fb6d3c11974260d5a408fd143ca7a49fba0ce37dac81f9c", 20464),
    "30586": ("b0fad12c6fc716881a677a903e478a00fbfbf0c466d49a75e50d6ac2c45bb72a", 373665),
    "30587": ("d7856849f002c181b51079fe56d059991c71c6aff167067ada1c810034c701b0", 134648),
    "30588": ("e0ea54e96e3ef90b229e0c2b0ddf2969d8a6e417aeab40e4230cca8e9fa0f0ac", 318010),
    "30589": ("e74bdf5292fb175da6fb2999c4459890f75c0e3f655a9e073205e93b9f1e7d70", 299348),
    "30590": ("fd5338dc93a71be9249f5d2ff2fd534f4a70d66ca395f448816c47b1f7b5a991", 30960),
    "30591": ("f98718f8b7aff763bb45570b2cc68453d1ecbddbd05f3cbf634171e95a126f7d", 393755),
    "30592": ("ccc853293bb88fcb85c3d5dcb5d63c2df1e95a49db841a5a274fa05b80c7b7a4", 233293),
    "30593": ("3e4f6ef21cb5851e78603d900a8e43a54d207ded9e414878265634c268be7a02", 180498),
    "30594": ("22e0cdcf86600ca0eee226d1c4380b7981c5c51bcb1556d81e910351f7045391", 197628),
    "30595": ("af110bad2526d906bdeed3ccb9e0ba9078bf5501ca50509209301c4872b1e9a4", 297991),
    "30596": ("22021b78b0aea430a434073e0a365250d4b3857c7007ed56ca15d3b70600ce40", 146876),
    "30597": ("92638189925b2833608c183ad258ad2386b9893ac2b65d7b0a7986e725cefead", 560584),
    "30598": ("e4a5d0710da1619c8e37539ec1a86acf959b0ab0f6db0b0c54b326efb8d65d02", 441822),
    "30599": ("41aad165d948964558d10ce0f409654994eabc1614042bc257342362ea16d5a7", 234615),
    "30600": ("c09a2203c0578736b5541b8511b91def3ab16a7db3e709f384e5ea3c2a59b092", 395250),
    "30601": ("dc0da350f13a1ecdd14905830beb5dbdf6846bb1a9eea407b1e7fee05abb7ee6", 586362),
    "30602": ("90e78229878fd1e05b249fcc0d958376466a248e7d10159baa71418e96f02f56", 235299),
    "30603": ("99b6fb8e0d4a04f5cc8f6025b817085cfda55e7f413596c28c93fc3ce6837ecb", 359878),
    "30604": ("0ba17d9fef6f1d431337651d84d89422fed4c58b7f7880826fe5595636d16582", 37193),
    "30605": ("a1d3e5a829032229c372fe26f2db2d9314709fba959f734bd22e254f033fe4f3", 394056),
    "30606": ("10b6ec544510ea2ccb6dae638f2b5c38fc872608672131dcab79be25293cc1c7", 436160),
    "30607": ("41d933e12191ce510e09b1a16bd9cc254e7e8911c4af3307fe8fbcaccfd6f163", 347641),
    "30608": ("62a3599e2c44b66a92a7107a805f5b6b5a3bae5264282cb54d3b59554e779971", 216190),
    "30609": ("dc8bedaf7e77f45f45edfb0502a6d97d9cb6c76a4a558c4691d7347081efd141", 387147),
    "30610": ("89921bd84440185872b2a0cbb7952c95b5e60713c8c8fc9cd332a243d36267c8", 334472),
    "30611": ("1ba80c8e100426b463567dedf6fe82648dc95a9c54d9b96677d4b11fbc3f28c6", 76207),
    "30612": ("bb8a67950bce2b4ddc33c76f78c54f3ffa5e9bc0efad6ed30e6d29d308473a69", 302297),
    "30613": ("c82c1ad80ca42c31f4faa39a30f6a87c16ca0818fb1d76d8be5539e85e78435d", 20956),
    "30614": ("8487c07e7739cba0f981b9b40bf00c320d8408726cadad52fa0b56b3f372ca18", 316914),
    "30615": ("fb78f236721f357d99245ae3c5d78074a2c5edbe3e2c47255629ab8b7e6efdbd", 235831),
    "30616": ("e99a3b9f31b0a279bddc7ea9b1bb3d02976d4d7f653ee1903e5c2d92a7e9ab8b", 301164),
    "30617": ("da7b80182e0a88e4b767cc8bf9f0ecd56fd1e12c260cb52ef99ae3a05189d4e0", 419867),
    "30618": ("a6dd0f2b68552716f93d6344cd5c4d810e7832b7d777f7ef9b2441ec7e829503", 45054),
    "30619": ("b1baa1adb4c0b32df8a221af5af1ae0ae308930fe127c8154a32d8cc0879355a", 235240),
    "30620": ("2ff2fb9b4f6e27e574194f4e4a5312060c73c2452f351e38bd7cb619ae8a06f2", 222916),
    "30621": ("b94b38101518c6ebda950c60b2d4eff3e16e127ea286d9259a8658277e6f9e1b", 289570),
    "30622": ("809c794e9dfb968c932ff17368b5c524870dc6ed8d88f03c2761ad1b55ff9b49", 51595),
    "30623": ("ad457939f89aee3131212ed4852a0d04f0cd021005b2c2c5c165ca68110ec56f", 323783),
    "30624": ("facd7f129b141b955721036765c7a7f1ba0be43f972b80267dfdf5e2567dfd0d", 101924),
    "30625": ("ee5c30d64207b43d7ff2b8673e4382cb84ca27567232c96b4c680d3006a849bc", 296372),
    "30626": ("18759313f42d75fcbdecf3b2c2242ce68a35acdbc787e40bf544335bc514684e", 409482),
    "30627": ("5a854affae38b034f12ee2f36ec0c3b8da7ab0018fc23611b5efaa680d5d6647", 245267),
    "30628": ("7af35b4688d80b2ae928bc0223d3713e7af7d12d15dadeaed58916bd09595a29", 272176),
    "30629": ("e29fccaa219ecdd59d80dadcf4006b13cd9d008b720f1678b70b8cc4d314a955", 41415),
    "30630": ("718abc6b645ca929069d77e8285626386f7b22237cc1bd71262434f5e1515948", 333161),
    "30631": ("40cbb4db8c2305f20e3fe0e2c55cf38553fa6127927655f466c313dece826278", 172277),
    "30632": ("10070b5ba6e6a95dd9cf5174e277bf3de86741939717784e4c22e46ba6b60125", 393134),
    "30633": ("8c3645c70ff0f6db726afe7b4351f145b0b526fafce120bc2bad230b4fb9152a", 315215),
    "30634": ("4ac2603b1e418d1f22a2f80d99d6e363e72a43ef83e2e88f3aeeb8dec0223ae9", 277483),
    "30635": ("2a58a9268d57baa0479ebb045bb4e4e7f7d4fa9362feceeb7ab813b8c5432f0c", 181760),
    "30636": ("0a4a1f3f6fe8f06952bee0ee7423032f58d063c27ddac087a1b282c52a009394", 10687),
    "30637": ("fb6945097d1fa6289be16b4c41aa78a395c3d38447f54b53e314fcc96a62161b", 323442),
    "30638": ("679925b3fc15c578c6b3b1ec9ec2dd80ac308ae791747abcac4c93ab1059bf7e", 459902),
}
# Near-duplicate screen (pre-registration Amendment 1, 2026-10-17). A fresh photograph whose 64-bit dHash lies
# within
# NEAR_DUPLICATE_MAX_DISTANCE bits of one of the 208 TRAINING photographs (pre-registered split) is only a
# CANDIDATE;
# it is excluded only when a recorded manual review confirms the same physical object or scene. Blank, black,
# blurred
# or other low-content frames are never confirmed. Candidates, decisions and reasons are pinned here.
NEAR_DUPLICATE_HASH = "dHash 64-bit (greyscale, 9 x 8, one bit per adjacent pair), candidates only"
NEAR_DUPLICATE_MAX_DISTANCE = 10
# The 12 row-group-2 photographs with no usable reference caption (every caption rejected or pre-canned
# upstream):
# they cannot be scored, so unrecognisable photographs are not measured.
FRESH_TEST_UNCAPTIONED: tuple[str, ...] = (
    "30317", "30328", "30380", "30402", "30450", "30461",
    "30462", "30553", "30565", "30605", "30624", "30636",
)
# candidate id -> (nearest training photograph, Hamming distance); computed 2026-10-17 before any fresh
# caption existed.
FRESH_TEST_CANDIDATES: dict[str, tuple[str, int]] = {
    "30315": ("29952", 10),
    "30316": ("29677", 9),
    "30331": ("29927", 6),
    "30334": ("29897", 9),
    "30355": ("29667", 3),
    "30359": ("29897", 6),
    "30365": ("29815", 10),
    "30370": ("29897", 7),
    "30371": ("29664", 8),
    "30374": ("29851", 9),
    "30379": ("29874", 8),
    "30389": ("29677", 0),
    "30400": ("29851", 10),
    "30403": ("29874", 10),
    "30404": ("29758", 9),
    "30410": ("29667", 9),
    "30413": ("29677", 7),
    "30419": ("29661", 10),
    "30431": ("29937", 10),
    "30448": ("29936", 10),
    "30449": ("29851", 10),
    "30463": ("29927", 10),
    "30467": ("29677", 0),
    "30470": ("29730", 9),
    "30489": ("29937", 9),
    "30491": ("29924", 9),
    "30501": ("29913", 10),
    "30526": ("29677", 0),
    "30551": ("29874", 5),
    "30552": ("29658", 10),
    "30556": ("29667", 9),
    "30574": ("29740", 10),
    "30576": ("29917", 10),
    "30587": ("29937", 9),
    "30599": ("29738", 8),
    "30602": ("29878", 10),
    "30608": ("29851", 10),
    "30613": ("29846", 9),
    "30631": ("29917", 10),
    "30635": ("29927", 4),
    "30637": ("29790", 8),
}
# candidate id -> (confirmed near-copy?, reason), from the recorded review of each candidate beside its
# nearest
# training photograph (2026-10-17, photographs only; no caption generated or score computed on the fresh
# test).
FRESH_TEST_SCREEN_REVIEW: dict[str, tuple[bool, str]] = {
    "30315": (False, "different objects: a $5 bill on a table vs a pasta box"),
    "30316": (False, "different scenes: hair and fabric vs a blank red frame (low-content reference)"),
    "30331": (False, "different objects: pink fabric vs a cartoon on a screen"),
    "30334": (False, "different objects: a blurred card vs a printed page with glare"),
    "30355": (False, "low-content frames: blurred denim vs a plain green surface"),
    "30359": (False, "different objects: a laptop surface vs a printed page with glare"),
    "30365": (False, "different objects: a wristwatch vs a garment edge"),
    "30370": (False, "different objects: baking instructions vs a printed page with glare"),
    "30371": (False, "different objects: a paprika jar vs a mayonnaise jar"),
    "30374": (False, "different objects: a digital display vs earphones"),
    "30379": (False, "low-content frames: dark fabric vs a plain red surface"),
    "30389": (False, "low-content: blank light frame vs blank red frame (hash distance 0, not a copy)"),
    "30400": (False, "different objects: a meat package vs earphones"),
    "30403": (False, "different objects: a CD cover vs a plain red surface"),
    "30404": (False, "low-content frames: a plain light surface vs a green card"),
    "30410": (False, "different objects: a chocolate bar on a table vs a plain green surface"),
    "30413": (False, "different objects: a jar label vs a blank red frame"),
    "30419": (False, "different objects: a laptop logo vs a packaged steak"),
    "30431": (False, "different objects: a book page vs a blank light frame"),
    "30448": (False, "different objects: a medicine box in a hand vs a TV remote"),
    "30449": (False, "different objects: striped fabric vs earphones"),
    "30463": (False, "different objects: a bottle label vs a cartoon on a screen"),
    "30467": (False, "low-content: blank light frame vs blank red frame (hash distance 0, not a copy)"),
    "30470": (False, "low-content frames: a blue blur vs a black frame"),
    "30489": (False, "different scenes: a floor and cabinet vs a blank light frame"),
    "30491": (False, "different objects: a wooden ceiling vs a gas meter"),
    "30501": (False, "different objects: a can on a table vs denim"),
    "30526": (False, "low-content: blank grey frame vs blank red frame (hash distance 0, not a copy)"),
    "30551": (False, "low-content frames: a blurred light garment vs a plain red surface"),
    "30552": (False, "different objects: a hi-fi cabinet vs a phone on a floor"),
    "30556": (False, "different objects: a keyboard vs a plain green surface"),
    "30574": (False, "different objects: a coffee pod vs hair"),
    "30576": (False, "different objects: a snack package vs a worksheet"),
    "30587": (False, "low-content frames: a blurred light surface vs a blank light frame"),
    "30599": (False, "different objects: a seed packet vs an audio receiver"),
    "30602": (False, "different objects: a poster vs a dark sofa"),
    "30608": (False, "different objects: a turkey package vs earphones"),
    "30613": (False, "different scenes: a blank glowing screen vs a laptop showing a web page"),
    "30631": (False, "different objects: a plug-in device on a pillow vs a worksheet"),
    "30635": (False, "different objects: a Nespresso package vs a cartoon on a screen"),
    "30637": (False, "different objects: a plastic container vs a keyboard"),
}
FRESH_TEST_EXCLUDED: tuple[str, ...] = tuple(
    sorted(k for k, (confirmed, _) in FRESH_TEST_SCREEN_REVIEW.items() if confirmed)
)
FRESH_POOL_SIZE = 324  # the captioned pool (secondary set)
FRESH_POOL_IDS_SHA256 = "02e15128fc50ed47a9c5e38f8f0d5c4774d72477fc23974b6efaa11def5316d6"
FRESH_TEST_SIZE = 324  # the confirmed-screened set (primary)
FRESH_TEST_IDS_SHA256 = "02e15128fc50ed47a9c5e38f8f0d5c4774d72477fc23974b6efaa11def5316d6"

# ---- Learning-rate grid and score-once rule (pre-registered 2026-10-17) ----
LR_GRID = (1e-5, 2e-5, 5e-5)
LR_DEFAULT = 1e-5
LR_MARGIN = 0.02  # a challenger replaces the default only if its best validation CIDEr-D is higher by >= this
# an adapted epoch is kept only if it beats epoch 0 (frozen) on validation CIDEr-D by >= this
FROZEN_MARGIN = 0.02
PREREGISTERED_DATA_SOURCE = (
    "pinned VizWiz-Captions sample (row group 0 train/validation, row group 2 fresh test)"
)
PREREGISTERED_SETTINGS: dict[str, Any] = {
    "data_source": PREREGISTERED_DATA_SOURCE,
    "byod": False,
    "split_seed": SAMPLE_SEED,
    "split_sizes": [SAMPLE_SPLIT["train"], SAMPLE_SPLIT["validation"], SAMPLE_SPLIT["test"]],
    "caption_max_tokens": 30,
    "epochs": 4,
    "batch_size": 8,
    "trainable_decoder_layers": 2,
    "adapt_seed": 0,
    "learning_rate": LR_DEFAULT,
    "run_grid": True,
    "lr_grid": list(LR_GRID),
    "lr_margin": LR_MARGIN,
    "frozen_margin": FROZEN_MARGIN,
    # Amendment 1: settings the pre-registration fixes beyond the form fields.
    "validation_max_new_tokens": 30,
    "weight_decay": ADAPT_WEIGHT_DECAY,
    "grad_clip": ADAPT_GRAD_CLIP,
    "scheduler": ADAPT_SCHEDULER,
    "decoding": DECODING,
    "model_revision": MODEL_REVISION,
    "fresh_test_ids_sha256": FRESH_TEST_IDS_SHA256,
    "fresh_pool_ids_sha256": FRESH_POOL_IDS_SHA256,
}
# A runtime cannot know about other runtimes (SCORE_ONCE_RULE item 8): a pre-registered scoring is
# labelled for this runtime; docs/release-verification.md names the hosted run of record.
TEST_LOOKS = ("pre-registered evaluation (this runtime)", "reproduction", "not scored (exploratory)")
# The release-verification record of the first pre-registered hosted run (None: none recorded yet). While
# it is set, every later run with the pre-registered settings is labelled a reproduction of the pre-registered
# evaluation.
PREREGISTERED_EVALUATION_RECORD: str | None = (
    "docs/release-verification.md: Colab CLI Tesla T4 run of 2026-10-17, commit e86f270, blob 07064ffc"
)


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def text_digest(rows: Sequence[Mapping[str, Any]]) -> str:
    """SHA-256 of the decoded text columns as a canonical JSON list of
    `[id, captions, question_type, text_detected]`."""
    payload = [
        [str(r["id"]), [str(c) for c in r["captions"]], str(r["question_type"]), bool(r["text_detected"])]
        for r in rows
    ]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


class _HttpRangeFile(io.RawIOBase):
    """A seekable read-only view of one HTTPS object served with `Range` requests (what `pyarrow` needs to
    read a parquet footer and a few column chunks without downloading the file)."""

    def __init__(self, url: str, size: int) -> None:
        self.url, self.size, self.pos = url, size, 0
        self.fetched = 0

    def readable(self) -> bool:
        return True

    def seekable(self) -> bool:
        return True

    def tell(self) -> int:
        return self.pos

    def seek(self, offset: int, whence: int = 0) -> int:
        base = {0: 0, 1: self.pos, 2: self.size}[whence]
        self.pos = max(0, base + offset)
        return self.pos

    def read(self, n: int = -1) -> bytes:
        if n is None or n < 0:
            n = self.size - self.pos
        if n <= 0 or self.pos >= self.size:
            return b""
        end = min(self.size, self.pos + n) - 1
        request = urllib.request.Request(self.url, headers={"Range": f"bytes={self.pos}-{end}"})
        with urllib.request.urlopen(request, timeout=300) as response:  # noqa: S310 (pinned https URL)
            if response.status != 206:
                raise ValueError(f"{self.url}: server ignored the Range request (HTTP {response.status})")
            data = response.read()
        self.fetched += len(data)
        self.pos += len(data)
        return data

    def readinto(self, buffer: Any) -> int:
        data = self.read(len(buffer))
        buffer[: len(data)] = data
        return len(data)


def _open_shard() -> Any:
    """The pinned shard as a `pyarrow.parquet.ParquetFile` over HTTPS range requests, after the file's
    declared size and LFS SHA-256 have been checked against the pins."""
    import pyarrow.parquet as pq
    from huggingface_hub import get_hf_file_metadata, hf_hub_url

    url = hf_hub_url(CORPUS_REPO, CORPUS_FILE["path"], repo_type="dataset", revision=CORPUS_REVISION)
    metadata = get_hf_file_metadata(url)
    declared = (metadata.etag or "").strip('"')
    if metadata.size != CORPUS_FILE["bytes"] or declared != CORPUS_FILE["sha256"]:
        raise ValueError(
            f"caption shard: the Hub declares {metadata.size} bytes / sha256 {declared[:16]}…, "
            f"pinned {CORPUS_FILE['bytes']} / {CORPUS_FILE['sha256'][:16]}…"
        )
    return pq.ParquetFile(_HttpRangeFile(url, CORPUS_FILE["bytes"]))


def _hub_text() -> list[dict[str, Any]]:
    rows = _open_shard().read(columns=list(CORPUS_TEXT_COLUMNS)).to_pylist()
    return [
        {
            "id": str(r["id"]),
            "captions": [str(c) for c in r["answer"]],
            "question_type": str(r["question_type"]),
            "text_detected": bool(r["text_detected"]),
        }
        for r in rows
    ]


def _hub_images(row_group: int | None = None) -> dict[str, bytes]:
    """The `media` column of one pinned row group only (row group 0 by default): image id -> JPEG bytes."""
    group = CORPUS_FILE["row_group"] if row_group is None else row_group
    table = _open_shard().read_row_group(group, columns=["id", CORPUS_IMAGE_COLUMN])
    out: dict[str, bytes] = {}
    for row in table.to_pylist():
        media = row[CORPUS_IMAGE_COLUMN]
        if not isinstance(media, list) or len(media) != 1:
            raise ValueError(
                f"row {row['id']}: expected exactly one image, found {len(media) if media else 0}"
            )
        out[str(row["id"])] = bytes(media[0]["bytes"])
    return out


def fetch_annotations(
    *, cache_dir: str | Path | None = None, fetcher: Callable[[], Sequence[Mapping[str, Any]]] | None = None
) -> list[dict[str, Any]]:
    """Return the shard's 1,550 caption rows from the cache or the Hub, digest-verified."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    local = cache / "annotations.json"
    rows: list[dict[str, Any]] | None = None
    if local.is_file():
        cached = json.loads(local.read_text(encoding="utf-8"))
        if isinstance(cached, list) and text_digest(cached) == CORPUS_FILE["text_sha256"]:
            rows = cached
    if rows is None:
        raw = fetcher() if fetcher is not None else _hub_text()
        rows = [
            {
                "id": str(r["id"]),
                "captions": [str(c) for c in r["captions"]],
                "question_type": str(r["question_type"]),
                "text_detected": bool(r["text_detected"]),
            }
            for r in raw
        ]
        if len(rows) != CORPUS_FILE["rows"] or text_digest(rows) != CORPUS_FILE["text_sha256"]:
            raise ValueError(
                f"caption shard: fetched {len(rows)} rows with text sha256 {text_digest(rows)[:16]}…, "
                f"pinned {CORPUS_FILE['rows']} / {CORPUS_FILE['text_sha256'][:16]}…"
            )
        local.write_text(json.dumps(rows, ensure_ascii=False), encoding="utf-8")
    return rows


def fetch_images(
    image_ids: Sequence[str],
    *,
    cache_dir: str | Path | None = None,
    fetcher: Callable[[], Mapping[str, bytes]] | None = None,
    fresh: bool = False,
) -> dict[str, Path]:
    """Stage the pinned photographs into the cache: cached files are re-hashed; anything missing or drifted is
    read from the pinned row group (one range read of its image column) and refused on any size or SHA-256
    mismatch. Returns image id -> path. `fresh=True` stages the fresh-test photographs of row group 2
    (`FRESH_IMAGE_PINS`) instead of the row-group-0 sample."""
    pins = FRESH_IMAGE_PINS if fresh else IMAGE_PINS
    if fetcher is None and fresh:
        def fetcher() -> Mapping[str, bytes]:
            return _hub_images(FRESH_TEST_ROW_GROUP)
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    (cache / "images").mkdir(parents=True, exist_ok=True)
    out: dict[str, Path] = {}
    missing: list[str] = []
    for image_id in image_ids:
        if image_id not in pins:
            raise ValueError(f"{image_id} is not one of the {len(pins)} pinned photographs")
        digest, size = pins[image_id]
        dest = cache / "images" / f"{image_id}.jpg"
        data = dest.read_bytes() if dest.is_file() else None
        if data is None or len(data) != size or _sha256_bytes(data) != digest:
            missing.append(image_id)
        out[image_id] = dest
    if missing:
        payload = fetcher() if fetcher is not None else _hub_images()
        for image_id in missing:
            digest, size = pins[image_id]
            data = payload.get(image_id)
            if data is None or len(data) != size or _sha256_bytes(data) != digest:
                got = (
                    f"{len(data)} bytes with sha256 {_sha256_bytes(data)[:16]}…"
                    if data is not None
                    else "no bytes"
                )
                raise ValueError(f"{image_id}.jpg: read {got}, pinned {size} / {digest[:16]}…")
            (cache / "images" / f"{image_id}.jpg").write_bytes(data)
    return out


def build_sample_dataset(
    annotations: Sequence[Mapping[str, Any]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
    image_paths: Mapping[str, Path] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The pinned photographs that carry at least one reference caption, shuffled with `seed` and cut **by
    image** into `sizes`; `image_paths` (from `fetch_images`) fills each record's `image`."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    by_id = {str(r["id"]): r for r in annotations}
    missing = [image_id for image_id in IMAGE_PINS if image_id not in by_id]
    if missing:
        raise ValueError(
            f"{len(missing)} pinned photograph(s) are absent from the annotations: {missing[:3]}"
        )
    pool = sorted(image_id for image_id in IMAGE_PINS if len(by_id[image_id]["captions"]) >= MIN_CAPTIONS)
    if sum(sizes.values()) > len(pool):
        raise ValueError(
            f"{len(pool)} pinned photographs carry a caption; the split sizes need {sum(sizes.values())}"
        )
    random.Random(seed).shuffle(pool)
    out: dict[str, list[dict[str, Any]]] = {}
    offset = 0
    for name in ("train", "validation", "test"):
        chosen = pool[offset : offset + sizes[name]]
        offset += sizes[name]
        out[name] = [
            {
                "id": f"{name}-{index:04d}",
                "image_id": image_id,
                "image": str(image_paths[image_id])
                if image_paths is not None and image_id in image_paths
                else f"{image_id}.jpg",
                "captions": [str(c) for c in by_id[image_id]["captions"]],
                "category": "text" if by_id[image_id]["text_detected"] else "no-text",
            }
            for index, image_id in enumerate(chosen)
        ]
    return out


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    annotation_fetcher: Callable[[], Sequence[Mapping[str, Any]]] | None = None,
    image_fetcher: Callable[[], Mapping[str, bytes]] | None = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned annotations and photographs."""
    annotations = fetch_annotations(cache_dir=cache_dir, fetcher=annotation_fetcher)
    paths = fetch_images(sorted(IMAGE_PINS), cache_dir=cache_dir, fetcher=image_fetcher)
    return build_sample_dataset(annotations, seed=seed, sizes=sizes, image_paths=paths)


def image_dhash(path: str | Path) -> int:
    """64-bit difference hash: greyscale, resized to 9 x 8 (bilinear), one bit per horizontally adjacent
    pair."""
    with Image.open(path) as image:
        small = image.convert("L").resize((9, 8), Image.Resampling.BILINEAR)
        pixels = list(small.getdata())
    bits = 0
    for row in range(8):
        for col in range(8):
            bits = (bits << 1) | int(pixels[row * 9 + col] > pixels[row * 9 + col + 1])
    return bits


def near_duplicate_screen(
    candidates: Mapping[str, str | Path],
    references: Mapping[str, str | Path],
    *,
    max_distance: int = NEAR_DUPLICATE_MAX_DISTANCE,
) -> dict[str, Any]:
    """Compare every candidate photograph's dHash with every reference photograph's. Returns the flagged
    candidate
    ids (nearest reference within `max_distance` bits), each flagged pair, and the distribution of nearest
    distances."""
    ref_hashes = {k: image_dhash(v) for k, v in references.items()}
    pairs: list[dict[str, Any]] = []
    nearest: dict[str, int] = {}
    for cid, path in sorted(candidates.items()):
        h = image_dhash(path)
        best_ref, best = min(
            ((rid, bin(h ^ rh).count("1")) for rid, rh in ref_hashes.items()), key=lambda x: (x[1], x[0])
        )
        nearest[cid] = best
        if best <= max_distance:
            pairs.append({"candidate": cid, "reference": best_ref, "distance": best})
    return {
        "hash": NEAR_DUPLICATE_HASH,
        "max_distance": max_distance,
        "candidates": len(candidates),
        "references": len(references),
        "flagged": sorted(p["candidate"] for p in pairs),
        "pairs": pairs,
        "nearest_distance_histogram": {str(k): v for k, v in sorted(Counter(nearest.values()).items())},
    }


def fresh_ids_digest(image_ids: Sequence[str]) -> str:
    """SHA-256 of the fresh test's image ids, in order, as a JSON list."""
    return _sha256_bytes(json.dumps([str(i) for i in image_ids], separators=(",", ":")).encode("utf-8"))


def set_profile(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Size, `text` share and mean reference count of an evaluated set: CIDEr-D is comparable only within one
    set
    (its document frequencies, category mix and reference counts differ between sets)."""
    n = len(records)
    return {
        "n": n,
        "text_share": round(sum(1 for r in records if r.get("category") == "text") / n, 3) if n else None,
        "mean_references": round(sum(len(r["captions"]) for r in records) / n, 2) if n else None,
    }


def build_fresh_test(
    annotations: Sequence[Mapping[str, Any]],
    fresh_paths: Mapping[str, Path],
    train_paths: Mapping[str, Path],
) -> tuple[list[dict[str, Any]], list[dict[str, Any]], dict[str, Any]]:
    """The fresh test (pre-registration Amendment 1): the captioned photographs of row group 2 (the secondary,
    unscreened pool) and that pool minus the review-confirmed near-copies (the primary set). Re-runs the dHash
    candidate screen against the training photographs and refuses to continue unless it reproduces the pinned
    candidates; checks image-id disjointness from row group 0 and both pinned id digests. Returns
    ``(primary_records, pool_records, report)``."""
    by_id = {str(r["id"]): r for r in annotations}
    shared = sorted(set(FRESH_IMAGE_PINS) & set(IMAGE_PINS))
    if shared:
        raise ValueError(f"fresh test shares image ids with row group 0: {shared[:3]}")
    pool = sorted(i for i in FRESH_IMAGE_PINS if len(by_id[i]["captions"]) >= MIN_CAPTIONS)
    uncaptioned = sorted(i for i in FRESH_IMAGE_PINS if len(by_id[i]["captions"]) < MIN_CAPTIONS)
    screen = near_duplicate_screen({i: fresh_paths[i] for i in pool}, dict(train_paths))
    found = {p["candidate"]: (p["reference"], p["distance"]) for p in screen["pairs"]}
    if found != dict(FRESH_TEST_CANDIDATES) or tuple(uncaptioned) != FRESH_TEST_UNCAPTIONED:
        raise ValueError(
            f"the candidate screen found {len(found)} candidates, pinned {len(FRESH_TEST_CANDIDATES)}: "
            "the photographs, the training split or the Pillow build differ from the pre-registration; "
            "the fresh test is not scored"
        )
    kept = [i for i in pool if i not in set(FRESH_TEST_EXCLUDED)]
    pinned = ((pool, FRESH_POOL_SIZE, FRESH_POOL_IDS_SHA256), (kept, FRESH_TEST_SIZE, FRESH_TEST_IDS_SHA256))
    for ids, size, digest in pinned:
        got = fresh_ids_digest(ids)
        if len(ids) != size or got != digest:
            raise ValueError(f"fresh test: {len(ids)} photographs with id digest {got[:16]}…, pinned {size}")
    records = [
        {
            "id": f"fresh-{index:04d}",
            "image_id": image_id,
            "image": str(fresh_paths[image_id]),
            "captions": [str(c) for c in by_id[image_id]["captions"]],
            "category": "text" if by_id[image_id]["text_detected"] else "no-text",
        }
        for index, image_id in enumerate(pool)
    ]
    primary = [r for r in records if r["image_id"] not in set(FRESH_TEST_EXCLUDED)]
    report = {
        "row_group": FRESH_TEST_ROW_GROUP,
        "photographs": len(FRESH_IMAGE_PINS),
        "uncaptioned": uncaptioned,
        "pool": len(pool),
        "candidates": len(found),
        "excluded": {k: FRESH_TEST_SCREEN_REVIEW[k][1] for k in FRESH_TEST_EXCLUDED},
        "kept": len(primary),
        "ids_sha256": FRESH_TEST_IDS_SHA256,
        "pool_ids_sha256": FRESH_POOL_IDS_SHA256,
        "shared_ids_with_row_group_0": 0,
        "screen": {k: v for k, v in screen.items() if k != "pairs"},
        "profiles": {
            "primary": set_profile(primary),
            "pool": set_profile(records),
            "excluded": set_profile([r for r in records if r["image_id"] in set(FRESH_TEST_EXCLUDED)]),
        },
    }
    return primary, records, report


def _canonical_setting(value: Any) -> Any:
    if isinstance(value, bool) or value is None or isinstance(value, str):
        return value
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, (list, tuple)):
        return [_canonical_setting(v) for v in value]
    return value


def check_preregistration(settings: Mapping[str, Any]) -> dict[str, Any]:
    """Compare a run's settings with ALL of `PREREGISTERED_SETTINGS`. Returns ``{"preregistered": bool,
    "settings_differing": [names], "settings": {...}}``; a missing or unknown key is refused, so no setting
    can be
    left out of the comparison."""
    missing = sorted(set(PREREGISTERED_SETTINGS) - set(settings))
    unknown = sorted(set(settings) - set(PREREGISTERED_SETTINGS))
    if missing or unknown:
        raise ValueError(
            "run settings must name exactly the pre-registered settings "
            f"(missing {missing}, unknown {unknown})"
        )
    differing = [
        name
        for name, fixed in PREREGISTERED_SETTINGS.items()
        if isinstance(settings[name], bool) != isinstance(fixed, bool)
        or _canonical_setting(settings[name]) != _canonical_setting(fixed)
    ]
    return {
        "preregistered": not differing,
        "settings_differing": differing,
        "settings": {k: settings[k] for k in PREREGISTERED_SETTINGS},
    }


def plan_test_look(
    check: Mapping[str, Any], outputs_dir: str | Path, stem: str, *, recorded_evaluation: str | None = None
) -> dict[str, Any]:
    """Label a run's look at the fresh test and name its report-file suffix. An exploratory run (any setting
    differs)
    never scores the fresh test and writes ``<stem>_evaluation_report_exploratory_<n>.json``. A pre-registered
    run is
    the pre-registered evaluation unless one is already recorded (`recorded_evaluation`, default
    `PREREGISTERED_EVALUATION_RECORD`) or this runtime already wrote the canonical report: then it is a
    reproduction,
    and a same-runtime reproduction writes ``_reproduction_<n>`` instead of overwriting the canonical
    report."""
    recorded = PREREGISTERED_EVALUATION_RECORD if recorded_evaluation is None else recorded_evaluation
    out = Path(outputs_dir)

    def next_free(kind: str) -> str:
        n = 1
        while (out / f"{stem}_evaluation_report_{kind}_{n}.json").exists():
            n += 1
        return f"_{kind}_{n}"

    base = {
        "preregistered": bool(check["preregistered"]),
        "settings_differing": list(check["settings_differing"]),
    }
    if not check["preregistered"]:
        suffix = next_free("exploratory")
        return {**base, "test_look": TEST_LOOKS[2], "suffix": suffix, "fresh_test_scored": False}
    if (out / f"{stem}_evaluation_report.json").exists():
        suffix = next_free("reproduction")
        return {**base, "test_look": TEST_LOOKS[1], "suffix": suffix, "fresh_test_scored": True}
    look = TEST_LOOKS[1] if recorded else TEST_LOOKS[0]
    return {**base, "test_look": look, "suffix": "", "fresh_test_scored": True}


def best_adapted_validation(history: Sequence[Mapping[str, Any]]) -> float:
    """A grid configuration's score: its best validation CIDEr-D over the adapted epochs (1..N)."""
    values = [h["val"]["cider_d"] for h in history if h["epoch"] >= 1 and h.get("val")]
    if not values:
        raise ValueError("no adapted epoch carries a validation score")
    return max(values)


def select_learning_rate(
    scores: Mapping[float, float], *, default: float = LR_DEFAULT, margin: float = LR_MARGIN
) -> dict[str, Any]:
    """The pre-registered selection rule: the default learning rate is kept unless another grid member's best
    validation CIDEr-D beats it by at least `margin`; among challengers that clear it the higher score wins,
    and an
    exact tie goes to the smaller learning rate."""
    if default not in scores:
        raise ValueError(f"the grid must include the default learning rate {default}")
    challengers = sorted(((lr, v) for lr, v in scores.items() if lr != default), key=lambda x: (-x[1], x[0]))
    chosen, reason = default, "default kept (no challenger beat it by the margin)"
    if challengers and challengers[0][1] - scores[default] >= margin - 1e-12:
        chosen, reason = challengers[0][0], f"challenger beat the default by >= {margin}"
    return {
        "chosen_lr": chosen,
        "default_lr": default,
        "margin": margin,
        "scores": {str(lr): round(v, 6) for lr, v in sorted(scores.items())},
        "best_challenger": None
        if not challengers
        else {"lr": challengers[0][0], "gain_over_default": round(challengers[0][1] - scores[default], 6)},
        "reason": reason,
    }


def _check_record(record: Any, index: int, *, base_dir: Path | None) -> dict[str, Any]:
    label = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label} must be a mapping with id/image/captions")
    for key in ("id", "image", "captions"):
        if key not in record:
            raise ValueError(f"{label} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label}: id must match {_ID_RE.pattern}")
    image_ref = record["image"]
    if not isinstance(image_ref, (str, Path)) or not str(image_ref).strip():
        raise ValueError(f"{label}: image must be a file path")
    path = Path(image_ref)
    if not path.is_absolute() and base_dir is not None:
        path = base_dir / path
    if not path.is_file():
        raise ValueError(f"{label}: image file not found: {path}")
    try:
        with Image.open(path) as handle:
            handle.load()
            validate_image(handle)
            width, height = handle.size
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{label}: {exc}") from exc
    except OSError as exc:
        raise ValueError(f"{label}: image cannot be decoded: {exc}") from exc
    captions = record["captions"]
    if isinstance(captions, str) or not isinstance(captions, Sequence) or len(captions) < MIN_CAPTIONS:
        raise ValueError(f"{label}: captions must be a list of at least {MIN_CAPTIONS} reference caption(s)")
    checked_captions = [" ".join(str(c).split()) for c in captions]
    if not all(checked_captions):
        raise ValueError(f"{label}: every reference caption must be a non-empty string")
    if any(len(c) > MAX_CAPTION_CHARS for c in checked_captions):
        raise ValueError(f"{label}: a reference caption exceeds MAX_CAPTION_CHARS={MAX_CAPTION_CHARS}")
    prefix = record.get("prefix")
    if prefix is not None and (not isinstance(prefix, str) or len(prefix) > MAX_PREFIX_CHARS):
        raise ValueError(f"{label}: prefix must be a str of at most MAX_PREFIX_CHARS={MAX_PREFIX_CHARS}")
    return {
        "id": rid,
        "image_id": str(record.get("image_id", rid)),
        "image": str(path),
        "image_size": [width, height],
        "captions": checked_captions,
        "category": str(record.get("category", "other")),
    }


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_RECORDS,
    max_records: int = MAX_RECORDS,
    base_dir: str | Path | None = None,
) -> dict[str, Any]:
    """Structural validation of a captioning dataset (every image opened and decoded); raises ValueError
    before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):
        raise ValueError("records must be a list of {id, image, captions} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    base = Path(base_dir) if base_dir is not None else None
    checked = []
    ids: set[str] = set()
    images: set[str] = set()
    for index, record in enumerate(records):
        item = _check_record(record, index, base_dir=base)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        images.add(item["image_id"])
        checked.append(item)
    return {
        "records": checked,
        "n_records": len(checked),
        "unique_images": len(images),
        "categories": dict(Counter(r["category"] for r in checked)),
        "captions_per_image": {
            "min": min(len(r["captions"]) for r in checked),
            "max": max(len(r["captions"]) for r in checked),
        },
        "caption_words": {
            "min": min(len(c.split()) for r in checked for c in r["captions"]),
            "max": max(len(c.split()) for r in checked for c in r["captions"]),
        },
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [[r["id"], r["image_id"], list(r["captions"]), r.get("category", "")] for r in records]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def reference_captions(record: Mapping[str, Any]) -> list[str]:
    """The reference captions of a record."""
    return [str(c) for c in record["captions"]]


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no image id appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = str(record.get("image_id", record["id"]))
            if key in seen and seen[key] != name:
                raise ValueError(f"image {key!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
    base_dir: str | Path | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded split of a BYOD dataset into train/validation/test **by image**: every record on the same image
    lands in the same split, so a test image is never seen in training."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records, base_dir=base_dir)["records"]
    groups: dict[str, list[dict[str, Any]]] = {}
    for record in checked:
        groups.setdefault(record["image_id"], []).append(record)
    order = list(groups.values())
    random.Random(seed).shuffle(order)
    n_test = max(1, round(len(checked) * test_fraction))
    n_val = round(len(checked) * val_fraction)
    splits: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    for group in order:
        if len(splits["test"]) < n_test:
            splits["test"].extend(group)
        elif len(splits["validation"]) < n_val:
            splits["validation"].extend(group)
        else:
            splits["train"].extend(group)
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(
            f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required"
        )
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read records from a JSON array or a JSONL file of ``{id, image, captions}`` objects; `image` paths are
    resolved relative to the file's directory by `validate_dataset(..., base_dir=...)`."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"dataset not found: {file_path}")
    suffix = file_path.suffix.lower()
    text = file_path.read_text(encoding="utf-8")
    if suffix == ".jsonl":
        return [json.loads(line) for line in text.splitlines() if line.strip()]
    if suffix == ".json":
        data = json.loads(text)
        if not isinstance(data, list):
            raise ValueError("JSON dataset must be an array of records")
        return data
    raise ValueError("BYOD datasets must be .json or .jsonl")


def write_dataset_jsonl(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """One record per line in the shape `load_byod_dataset` reads back (image paths as given)."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    keys = ("id", "image_id", "image", "captions", "category")
    with open(out, "w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps({k: record[k] for k in keys if k in record}, ensure_ascii=False) + "\n")
    return out

## 3. Pin, stage and verify the model

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `61bee0d7e237…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `Pix2StructTextCapsPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
# @title Infrastructure: stage and digest-verify the pinned snapshot, then load the model
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "pix2struct-textcaps-base",
  "modelId": "google/pix2struct-textcaps-base",
  "revision": "61bee0d7e2378e601b68f853ceee4f7cf99f1b88",
  "files": [
    {
      "path": "README.md",
      "bytes": 7676,
      "sha256": "e5dc83622c2ce6ce1df491db93f6ddc6a7ac27f5988c4a36c432a69e91de0bf0"
    },
    {
      "path": "config.json",
      "bytes": 5009,
      "sha256": "aaebb1cbc1eb952676e1b874b09ea28b2519f654ebf261807b22eaebc43f2023"
    },
    {
      "path": "model.safetensors",
      "bytes": 1129177976,
      "sha256": "771693f7e310a89d77aa7ee4d1f5ea3e0c4adb0b6112cdf25a20da29cc3d0ea7"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 217,
      "sha256": "885e6dd8946290a61b0a808e90cc74dd1a33a011b6482fd213bc7327945305f8"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 2201,
      "sha256": "5c87151ef0f72a99d1f766a4c418bd2a1f90aaa30a8e22fe5eca9641daebb64f"
    },
    {
      "path": "spiece.model",
      "bytes": 851388,
      "sha256": "7fd650335add59bed55a432186ca0437a09e185c2d241faab468a538fe6bcf94"
    },
    {
      "path": "tokenizer.json",
      "bytes": 3265159,
      "sha256": "0af109b23840545ef2c286073f4373959badba1faa73c8557881d5126f6287c9"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 2583,
      "sha256": "5fdb6767a49aca48fdfa43d0279321918185fc4997bdb3ea72bf3a6301a1b43d"
    }
  ],
  "totalBytes": 1133312209
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = Pix2StructTextCapsPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. VizWiz photographs, captions and split

`fetch_annotations` reads the pinned shard's four text columns (or the cache under `weights/vizwiz-captions/`): it first checks the byte size and SHA-256 the Hub declares for the file against the pins, then reads only the parquet footer and those column chunks through `pyarrow` over HTTPS range requests, and refuses the decoded columns unless their SHA-256 matches. `fetch_images` stages the 336 pinned photographs of row group 0 — each cached file is re-hashed; anything missing is read from the shard's image column in one range read and refused on any size or SHA-256 mismatch. `build_sample_dataset` keeps the 318 photographs that carry at least one reference caption (the Hub re-conversion dropped rejected and pre-canned captions), shuffles them with `SPLIT_SEED` and cuts them **by image** into 208 / 40 / 70 training, validation and **exposed development test** records (the 70 were used to choose earlier recipes, see the introduction), each labelled `text` or `no-text` from the corpus's text-detected flag. `validate_dataset` then opens and decodes every image and checks every record against the contract, `check_split_disjoint` asserts no image is shared, and the training split is written to `outputs/pix2struct_textcaps_train.jsonl` in the shape BYOD expects. The **fresh test** is staged the same way from row group 2 (`FRESH_IMAGE_PINS`): `build_fresh_test` takes its 324 photographs with a usable caption (12 more have none — every caption was rejected or pre-canned upstream — so unrecognisable photographs cannot be scored), re-runs the pre-registered **candidate screen** (a 64-bit difference hash of each fresh photograph against the 208 training photographs; within 10 bits makes a candidate, 41 of them) and refuses to continue unless it reproduces the pinned candidates. A candidate is excluded only if a recorded review of the two photographs confirmed the same object or scene (`FRESH_TEST_SCREEN_REVIEW`, one reason per id); none was, because a whole-image hash also matches different objects and blank frames. The primary fresh set (screened) and the secondary set (the unscreened pool) are both pinned by an id-list SHA-256.Look for: 1,550 annotation rows, 336 photographs, three digests, the fresh test's 324 photographs, 41 candidates and 0 confirmed exclusions, the category mix per split (a little over half the photographs contain text), captions per image between 1 and 5, and four refusal probes — a duplicate id, a missing image file, an empty caption list and a dataset too small to split — each rejected before `torch` does anything.

**Predict before running:** why split by photograph rather than by caption? About how many of the test photographs will have readable text?

In [ ]:
import collections
import hashlib
import io
import json
import shutil
import time
import zipfile

USE_BYOD = True  # @param {type:"boolean"}
BYOD_PATH = '/etc/hostname'  # @param {type:"string"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)


class RunInvalidated(RuntimeError):
    """This run was stopped on purpose (a refused BYOD input); the message says what to fix. Shown without a traceback."""

    dimer_stop = True

    def _render_traceback_(self):
        return [str(self)]


def require_valid_run():
    """Every later section starts here: after a refused BYOD input nothing is scored until Section 4 succeeds."""
    state = globals().get('RUN_STATE') or {'valid': False, 'reason': 'Section 4 has not run yet'}
    if not state['valid']:
        raise RunInvalidated(f"This run is invalidated ({state['reason']}). Fix the input named above and run Section 4 again; this section did not run.")


def byod_file(path, kind, suffixes=()):
    """BYOD path first (works on Colab, Kaggle and Jupyter); on Colab an empty path opens the upload dialog."""
    if str(path).strip():
        source = Path(str(path).strip()).expanduser()
        if not source.is_file():
            raise FileNotFoundError(f'{str(source)}: BYOD_PATH does not exist or is not a file (relative paths start at {os.getcwd()}); give the path of one {kind}.')
    else:
        try:
            from google.colab import files
        except ImportError:
            raise FileNotFoundError(f'(no file): BYOD is on but its path field is empty, and the upload dialog exists only in Google Colab: copy the {kind} into this runtime (or attach it as a Kaggle dataset) and set the path field.') from None
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError(f'{", ".join(uploaded) or "(no file)"}: Upload exactly one {kind} (received {len(uploaded)} files; a cancelled dialog sends none). Run this cell again.')
        name, payload = next(iter(uploaded.items()))
        source = Path('work') / Path(name).name
        source.parent.mkdir(parents=True, exist_ok=True)
        source.write_bytes(payload)
    if suffixes and not source.name.lower().endswith(tuple(suffixes)):
        raise ValueError(f'{source.name}: expected a {kind} ending in {" or ".join(suffixes)}.')
    return source


def load_byod_splits():
    """Acquire, extract, parse and split the BYOD zip; every refusal names the zip (or the upload step) and the rule."""
    byod_zip = byod_file(BYOD_PATH, 'zip of images plus records.jsonl', ('.zip',))
    file_name, payload = byod_zip.name, byod_zip.read_bytes()
    byod_dir = Path('work') / 'byod'
    shutil.rmtree(byod_dir, ignore_errors=True)
    byod_dir.mkdir(parents=True, exist_ok=True)
    BYOD_MAX_MEMBERS, BYOD_MAX_EXPANDED_BYTES = MAX_RECORDS + 2, 2 * 1024 ** 3  # one image per record plus the records file; 2 GiB extracted
    try:
        archive = zipfile.ZipFile(io.BytesIO(payload))
    except zipfile.BadZipFile:
        raise ValueError(f'{file_name}: not a readable zip archive; re-create it with a zip tool.') from None
    with archive:
        members = [m for m in archive.infolist() if not m.is_dir() and Path(m.filename).name and not Path(m.filename).name.startswith('.') and '__MACOSX' not in m.filename]
        expanded = sum(m.file_size for m in members)
        if len(members) > BYOD_MAX_MEMBERS or expanded > BYOD_MAX_EXPANDED_BYTES:
            raise ValueError(f'{file_name}: {len(members)} files and {expanded:,} bytes when extracted exceed the BYOD ceiling of {BYOD_MAX_MEMBERS} files / {BYOD_MAX_EXPANDED_BYTES:,} bytes (a dataset holds at most MAX_RECORDS = {MAX_RECORDS} records); pack fewer or smaller images.')
        for member in members:  # folders inside the zip are kept, so records may name images as images/x.jpg
            relative = Path(member.filename.replace(chr(92), '/'))
            if relative.is_absolute() or '..' in relative.parts:
                raise ValueError(f'{file_name}: member {member.filename!r} points outside the zip (absolute path or ..); repack it with relative paths.')
            target = byod_dir / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(member))
    records_files = sorted(p for p in byod_dir.rglob('*') if p.is_file() and p.name in ('records.jsonl', 'records.json'))
    if len(records_files) != 1:
        found = [p.relative_to(byod_dir).as_posix() for p in records_files]
        raise ValueError(f'{file_name}: the zip must hold exactly one records.jsonl (or records.json) — at its root or inside one folder — next to the images; found {found or "none"}. Add one JSON object per line.')
    records_file = records_files[0]
    try:
        records = load_byod_dataset(records_file)
        splits = split_dataset(records, seed=SPLIT_SEED, base_dir=records_file.parent)  # image paths are relative to the records file
    except (TypeError, ValueError, json.JSONDecodeError) as exc:
        raise ValueError(f'{file_name}: {exc}') from None
    small = {name: len(rows) for name, rows in splits.items() if len(rows) < MIN_RECORDS}
    if small:
        raise ValueError(f'{file_name}: split(s) {small} hold fewer than MIN_RECORDS = {MIN_RECORDS} records (rows per split { {name: len(rows) for name, rows in splits.items()} }, {len(records)} records in total). Every split is validated with the same floor; with the 0.15 / 0.20 validation / test fractions the smallest dataset that passes is 50 records at one record per image — add records.')
    print({'byod_zip': file_name, 'members': len(members), 'expanded_bytes': expanded, 'records_file': records_file.relative_to(byod_dir).as_posix(), 'records': len(records)})
    return file_name, records, splits


invalid_marker = Path('outputs/pix2struct_textcaps_run_invalidated.json')
fresh_records, fresh_pool_records, fresh_report = None, None, None
if USE_BYOD:
    try:
        file_name, records, splits = load_byod_splits()
        try:
            dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
        except (TypeError, ValueError) as exc:  # validate_dataset names the record and rule; the zip is named here
            raise ValueError(f'{file_name}: {exc}') from None
    except (FileNotFoundError, ValueError, TypeError) as exc:  # a refused input; other errors keep their traceback
        refusal = str(exc)
        for stale_name in ('splits', 'train_records', 'val_records', 'test_records', 'dataset_manifests', 'disjoint', 'categories', 'fresh_records', 'fresh_report', 'frozen_test', 'adapt_result', 'adapted_test', 'comparison'):
            globals().pop(stale_name, None)
        kept = sorted(p.name for p in Path('outputs').glob('pix2struct_textcaps_*') if p.name != invalid_marker.name)
        RUN_STATE = {'valid': False, 'reason': 'BYOD input refused'}
        invalid_marker.write_text(json.dumps({'run_valid': False, 'refusal': refusal, 'time': time.strftime('%Y-%m-%d %H:%M:%S'), 'earlier_outputs_kept_not_results_for_this_input': kept}, indent=2), encoding='utf-8')
        note = f' Earlier files in outputs/ were kept but are not results for this input (listed in {invalid_marker}).' if kept else ''
        raise RunInvalidated(f'Refused: {refusal}{note} Nothing was scored; fix the named file or rule and run this cell again.') from None
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod': len(records)}
else:
    annotations = fetch_annotations(cache_dir='weights/vizwiz-captions')
    image_paths = fetch_images(sorted(IMAGE_PINS), cache_dir='weights/vizwiz-captions')
    raw_rows = {'annotations': len(annotations), 'photographs': len(image_paths), 'captioned': sum(1 for r in annotations if r['id'] in IMAGE_PINS and r['captions'])}
    splits = build_sample_dataset(annotations, seed=SPLIT_SEED, image_paths=image_paths)
    data_source = f'{CORPUS_NAME} {CORPUS_RELEASE} ({CORPUS_LICENSE})'
    # The fresh test: row group 2 of the same shard. dHash hits against the training photographs are candidates only;
    # only near-copies confirmed by the pinned manual review are excluded (pre-registration Amendment 1).
    fresh_paths = fetch_images(sorted(FRESH_IMAGE_PINS), cache_dir='weights/vizwiz-captions', fresh=True)
    screen_train = build_sample_dataset(annotations, seed=SAMPLE_SEED, image_paths=image_paths)['train']  # the pre-registered training photographs
    fresh_records, fresh_pool_records, fresh_report = build_fresh_test(annotations, fresh_paths, {r['image_id']: image_paths[r['image_id']] for r in screen_train})
    fresh_pool_records = validate_dataset(fresh_pool_records)['records']
    fresh_records = [r for r in fresh_pool_records if r['image_id'] not in set(FRESH_TEST_EXCLUDED)]
    dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
splits = {name: manifest['records'] for name, manifest in dataset_manifests.items()}
disjoint = check_split_disjoint({**splits, **({'fresh_test_pool': fresh_pool_records} if fresh_pool_records else {})})
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
categories = {name: manifest['categories'] for name, manifest in dataset_manifests.items()}
write_dataset_jsonl(splits['train'], 'outputs/pix2struct_textcaps_train.jsonl')
RUN_STATE = {'valid': True, 'reason': None}
if invalid_marker.exists():
    invalid_marker.unlink()
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint, 'text_sha256': CORPUS_FILE['text_sha256'][:16] + '...', 'pinned_photographs': len(IMAGE_PINS)})
for name, manifest in dataset_manifests.items():
    label = 'exposed development test' if name == 'test' and not USE_BYOD else name
    print({label: {'n': manifest['n_records'], 'unique_images': manifest['unique_images'], 'categories': manifest['categories'], 'captions_per_image': manifest['captions_per_image'], 'caption_words': manifest['caption_words'], 'digest': manifest['digest'][:16] + '...'}})
if fresh_report is not None:
    print({'fresh_test': {k: fresh_report[k] for k in ('row_group', 'photographs', 'pool', 'candidates', 'kept', 'shared_ids_with_row_group_0')}, 'uncaptioned_not_scorable': len(fresh_report['uncaptioned']), 'confirmed_near_copies_excluded': fresh_report['excluded'], 'screen': f"dHash candidates at distance <= {fresh_report['screen']['max_distance']} against {fresh_report['screen']['references']} training photographs, confirmed by a recorded review", 'ids_sha256': fresh_report['ids_sha256'][:16] + '...'})
    print({'profiles (n, text share, mean references)': fresh_report['profiles']})
example = splits['train'][0]
print({'example': {'id': example['id'], 'image': Path(example['image']).name, 'size': example['image_size'], 'category': example['category'], 'captions': example['captions']}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in splits['train'][:8]],
    'missing image file': [{**splits['train'][0], 'image': 'work/does-not-exist.jpg'}, *splits['train'][1:8]],
    'empty caption list': [{**splits['train'][0], 'captions': []}, *splits['train'][1:8]],
    'too small': splits['train'][:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})